# 04. 모델 선택 (Model Selection)

**Why (왜 이 노트북이 필요한가)**: `03_features.ipynb`에서 888개 컬럼짜리 피처를 다 만들었지만, 아직 어떤 모델로 예측할지는 정하지 않았습니다. CLAUDE.md 로드맵 7단계 — 후보 모델(선형회귀/LightGBM/XGBoost/CatBoost)을 **공정한 조건**에서 비교해서 `05_tuning.ipynb`로 넘길 1~2개를 고릅니다.

**입력**: `data/processed/train_features_v1.parquet` (26304행 × 888열)
**출력**: 이 노트북은 parquet을 새로 만들지 않습니다. 결과는 비교 표(마크다운)로 남기고, 최종 정리는 `reports/04_model_selection.md`에 기록합니다.

## 이 노트북에서 반드시 지키는 것

1. **A안 + B안 항상 병행** (`timeseries-validation` 스킬, CLAUDE.md 5장 확정 사항): 모든 비교표에 A안(2022~2023 학습→2024 검증)과 B안(3-fold 확장 윈도우) 점수를 같이 적습니다.
2. **fold-safe 피처만 사용**: `{group}_ws_est`, `{group}_power_curve_est`는 2022~2024 **전체** 데이터로 학습된 값이라 그대로 쓰면 미래 정보가 회귀계수를 통해 새어 들어갑니다(예측기준시점 원칙 위반). `03_features.ipynb`가 이미 만들어둔 `_cv_2023_07/2024_01/2024_07` fold-safe 버전을 씁니다.
3. **새로 발견한 것 (이번 세션)**: `{group}_ws_est_sq/_cube/_corrected`, `{group}_regime_calm/ramp/rated`, `{group}_high_wind_caution`, `{group}_icing_risk`도 전부 fold-safe 하지 않은 `ws_est`에서 파생됐습니다. 이 노트북 2절에서 각 fold의 `ws_est_cv_*`로부터 **그 자리에서 다시 계산**해서 씁니다(민석님 확인 후 채택한 방식 — "폴드별 즉석 재계산").


## 1. 셋업

이 셀은 파이썬 실행 환경(venv인지 확인), 필요한 라이브러리, 경로 상수, 시드를 준비합니다.

이번 노트북부터 **LightGBM / XGBoost / CatBoost / scikit-learn**을 새로 씁니다 (venv에 설치 완료: `lightgbm 4.7.0`, `xgboost 3.3.0`, `catboost 1.2.10`, `scikit-learn 1.9.0`). `requirements.txt`는 민석님과 상의한 대로 튜닝 단계까지 패키지가 다 들어간 뒤 한 번에 정리합니다.

In [1]:
import sys
print(sys.executable)  # venv\Scripts\python.exe 인지 확인

d:\공모전\wind_forecast_new\venv\Scripts\python.exe


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from src.metric import metric, TARGET_COLS, CAPACITY_KWH

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

SEED = 42
np.random.seed(SEED)

PROCESSED_DIR = Path("../data/processed") if Path.cwd().name == "notebooks" else Path("data/processed")
GROUP_COLS = TARGET_COLS  # ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
print(CAPACITY_KWH)

{'kpx_group_1': 21600, 'kpx_group_2': 21600, 'kpx_group_3': 21000}


## 2. 데이터 로드

`train_features_v1.parquet`만 로드합니다. `test_features_v1.parquet`은 최종 재학습(9~10단계, `ensemble-final` 스킬)에서 씁니다 — 지금은 모델 비교 단계라 미래(2025) 예측은 하지 않습니다.

In [3]:
train = pd.read_parquet(PROCESSED_DIR / "train_features_v1.parquet")
print(train.shape)
display(train[["kst_dtm", *GROUP_COLS]].head(3))

(26304, 888)


,kst_dtm,kpx_group_1,kpx_group_2,kpx_group_3
0,2022-01-01 01:00:00,12004.421,9719.242,NaN
1,2022-01-01 02:00:00,12901.137,10297.768,NaN
2,2022-01-01 03:00:00,12091.200,10731.663,NaN


**확인할 것**: `(26304, 888)`. `kst_dtm`이 시간순으로 정렬돼 있는지, `kpx_group_1/2/3` 값이 정상 범위(0~설비용량 근처)인지 육안 확인.

In [4]:
print("kst_dtm 범위:", train["kst_dtm"].min(), "~", train["kst_dtm"].max())
print("결측 라벨 수:")
for g in GROUP_COLS:
    print(f"  {g}: {train[g].isna().sum()}건 (전체 {len(train)}행 중)")

kst_dtm 범위: 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00
결측 라벨 수:
  kpx_group_1: 516건 (전체 26304행 중)
  kpx_group_2: 516건 (전체 26304행 중)
  kpx_group_3: 9201건 (전체 26304행 중)


**확인할 것**: `kpx_group_3`은 2022년치가 전부 NaN이라 다른 그룹보다 결측이 많은 게 정상입니다(02_eda에서 이미 확인된 사실 — group_3은 2023년부터 라벨 존재).

## 3. 모델 입력 피처 정의 — 공통 피처 / 그룹 전용 피처 / 제외할 피처

**결론부터**: 888개 컬럼을 세 종류로 나눕니다.

1. **제외 컬럼** (`NON_FEATURE_COLS`): 정답(`kpx_group_1/2/3`), 시각(`kst_dtm`), 예보 도착 시각(`data_available_kst_dtm_*`), 검증용 임시 컬럼(`year`), 그리고 **SCADA 실측값**(`scada_ws_kpx_group_*`, `scada_kpx_group_*`) — SCADA는 test에 없는 학습 전용 정답 보조자료라 모델 입력으로 쓰면 안 됩니다(`03_features.ipynb` 요약에서 이미 못박은 규칙).
2. **공통 피처** (`COMMON_RAW_COLS`): 그룹 이름이 안 붙은 컬럼 — GFS 9개 격자, LDAPS 16격자 평균, `hour_sin/cos`, `month_sin/cos` 등. 세 그룹 모델 모두에 공통으로 씁니다.
3. **그룹 전용 피처** (`GROUP_SPECIFIC_COLS[g]`): `{group}_`로 시작하는 컬럼 — 그 그룹의 최근접 격자, 추정풍속, 파워커브 추정치, 결빙·돌풍 위험 등. 이 그룹의 모델에만 씁니다.

In [5]:
NON_FEATURE_COLS = set([
    "kst_dtm",
    "data_available_kst_dtm_gfs",
    "data_available_kst_dtm_ldaps",
    "year",
    *TARGET_COLS,
    *[f"scada_ws_{g}" for g in GROUP_COLS],
    *[f"scada_{g}" for g in GROUP_COLS],
])

COMMON_RAW_COLS = [
    c for c in train.columns
    if c not in NON_FEATURE_COLS and not any(c.startswith(f"{g}_") for g in GROUP_COLS)
]
GROUP_SPECIFIC_COLS = {
    g: [c for c in train.columns if c.startswith(f"{g}_") and c not in NON_FEATURE_COLS]
    for g in GROUP_COLS
}

print("제외 컬럼 수:", len(NON_FEATURE_COLS))
print("공통 피처 수:", len(COMMON_RAW_COLS))
for g in GROUP_COLS:
    print(f"{g} 전용 피처 수:", len(GROUP_SPECIFIC_COLS[g]))

제외 컬럼 수: 13
공통 피처 수: 804
kpx_group_1 전용 피처 수: 24
kpx_group_2 전용 피처 수: 24
kpx_group_3 전용 피처 수: 23


**확인할 것**: 제외 컬럼 수 + 공통 피처 수 + 그룹 전용 피처 수(3개 그룹 합) = 888이 돼야 합니다(빠뜨리거나 중복된 컬럼이 없는지 검산).

In [6]:
total = len(NON_FEATURE_COLS) + len(COMMON_RAW_COLS) + sum(len(v) for v in GROUP_SPECIFIC_COLS.values())
print(total, "== 888 이어야 함:", total == train.shape[1])

888 == 888 이어야 함: True


## 4. 검증 fold 정의 — A안 + B안

**무엇을**: A안(2022~2023 학습 → 2024 전체 검증)과 B안(확장 윈도우 3-fold)의 학습/검증 구간 경계를 정의합니다.

**왜 항상 같이 보는가**: `timeseries-validation` 스킬 1절 — A안 하나만 보면 2024년이 이례적인 해였을 경우 속을 수 있고, B안 fold만 보면 fold별 6개월 구간이 계절에 치우칠 수 있습니다. 두 방식을 같이 봐야 서로의 편향을 잡아줍니다.

| fold | 학습 구간 | 검증 구간 | 쓰는 fold-safe 컬럼 |
|---|---|---|---|
| A안(2024) | 2022-01 ~ 2023-12 | 2024-01 ~ 2024-12 (전체) | `*_cv_2024_01` |
| B안 fold1 | ~2023-06 | 2023-07 ~ 2023-12 | `*_cv_2023_07` |
| B안 fold2 | ~2023-12 | 2024-01 ~ 2024-06 | `*_cv_2024_01` |
| B안 fold3 | ~2024-06 | 2024-07 ~ 2024-12 | `*_cv_2024_07` |

**A안과 B안 fold2가 같은 cv 컬럼을 쓰는 이유**: 둘 다 학습 구간이 정확히 2022~2023년 전체로 같기 때문입니다. `ws_est_cv_2024_01`은 "2024-01-01 이전 데이터로만 학습한 버전"이라는 뜻이라, 검증 구간이 2024년 상반기(B안 fold2)든 2024년 전체(A안)든 상관없이 안전하게 재사용할 수 있습니다 — cv 컬럼의 안전성은 **학습에 쓰인 기간**으로만 결정되지, 검증 기간의 길이와는 무관하기 때문입니다.

In [7]:
CUTOFFS = {
    "2023_07": pd.Timestamp("2023-07-01"),
    "2024_01": pd.Timestamp("2024-01-01"),
    "2024_07": pd.Timestamp("2024-07-01"),
}
END_TRAIN = pd.Timestamp("2025-01-01")  # train 데이터의 끝(다음 시각의 시작점 기준, 배타적 상한)

FOLD_SPECS = {
    "A안(2024)":  dict(cv_suffix="2024_01", valid_start=CUTOFFS["2024_01"], valid_end=END_TRAIN),
    "B안 fold1": dict(cv_suffix="2023_07", valid_start=CUTOFFS["2023_07"], valid_end=CUTOFFS["2024_01"]),
    "B안 fold2": dict(cv_suffix="2024_01", valid_start=CUTOFFS["2024_01"], valid_end=CUTOFFS["2024_07"]),
    "B안 fold3": dict(cv_suffix="2024_07", valid_start=CUTOFFS["2024_07"], valid_end=END_TRAIN),
}

for name, spec in FOLD_SPECS.items():
    train_mask = train["kst_dtm"] < CUTOFFS[spec["cv_suffix"]]
    valid_mask = (train["kst_dtm"] >= spec["valid_start"]) & (train["kst_dtm"] < spec["valid_end"])
    print(f"{name}: 학습 후보 {train_mask.sum()}행 / 검증 후보 {valid_mask.sum()}행")
    for g in GROUP_COLS:
        n_tr = (train_mask & train[g].notna()).sum()
        n_va = (valid_mask & train[g].notna()).sum()
        print(f"   {g}: 학습 라벨 {n_tr}행 / 검증 라벨 {n_va}행")

A안(2024): 학습 후보 17519행 / 검증 후보 8784행
   kpx_group_1: 학습 라벨 17341행 / 검증 라벨 8446행
   kpx_group_2: 학습 라벨 17341행 / 검증 라벨 8446행
   kpx_group_3: 학습 라벨 8662행 / 검증 라벨 8440행
B안 fold1: 학습 후보 13103행 / 검증 후보 4416행
   kpx_group_1: 학습 라벨 12925행 / 검증 라벨 4416행
   kpx_group_2: 학습 라벨 12925행 / 검증 라벨 4416행
   kpx_group_3: 학습 라벨 4246행 / 검증 라벨 4416행
B안 fold2: 학습 후보 17519행 / 검증 후보 4368행
   kpx_group_1: 학습 라벨 17341행 / 검증 라벨 4030행
   kpx_group_2: 학습 라벨 17341행 / 검증 라벨 4030행
   kpx_group_3: 학습 라벨 8662행 / 검증 라벨 4030행
B안 fold3: 학습 후보 21887행 / 검증 후보 4416행
   kpx_group_1: 학습 라벨 21371행 / 검증 라벨 4416행
   kpx_group_2: 학습 라벨 21371행 / 검증 라벨 4416행
   kpx_group_3: 학습 라벨 12692행 / 검증 라벨 4410행


**확인할 것**: `kpx_group_3`은 2023년부터 라벨이 있으므로, B안 fold1(학습 구간이 ~2023-06)의 `group_3` 학습 라벨 수가 다른 그룹보다 훨씬 적게 나오는 게 정상입니다(2023년 상반기치만 있음). 이 부분은 `timeseries-validation` 스킬에서 이미 알려진 제약이라 결과 해석 시 감안해야 합니다.

## 5. fold-safe 파생 피처 재계산 (이번 세션에 새로 발견한 부분)

**무엇을**: `{group}_ws_est_sq/_cube/_corrected`, `{group}_regime_calm/ramp/rated`, `{group}_high_wind_caution`, `{group}_icing_risk`를 원래 `ws_est`(전체 데이터 학습판) 대신 **각 fold의 `ws_est_cv_{cutoff}`로부터 그 자리에서 재계산**합니다.

**왜**: `03_features.ipynb`를 다시 읽어보니 이 6종 피처가 전부 fold-safe하지 않은 `ws_est`를 통과시켜 만들어져 있었습니다(`ws_est` 본체와 `power_curve_est`만 fold-safe `_cv_*` 버전이 있었음). 그대로 A/B 검증에 쓰면 예측기준시점 원칙을 어기게 됩니다. 다행히 전부 **고정 임계값(3.0/12.0/15.0 m/s)이나 정해진 공식**으로 만든 파생값이라, 학습(fit)이 필요 없이 산술 연산만 다시 하면 fold-safe 버전을 만들 수 있습니다.

**공식은 `03_features.ipynb`와 완전히 동일하게 재사용**(임계값을 새로 정하지 않음 — 이미 근거를 남긴 값 그대로):
- `ws_est_corrected = ws × (공기밀도/표준밀도)^(1/3)` (4절), `sq`/`cube`는 이 보정값의 제곱/세제곱 (03_features 3-2절 순서 그대로: sq/cube는 corrected 기준)
- `regime_calm/ramp/rated`: `ws < 3.0` / `3.0 ≤ ws < 12.0` / `ws ≥ 12.0` (13-1절)
- `high_wind_caution`: `ws ≥ 15.0` (13-2절)
- `icing_risk`(group_1/2 전용): 기온 < 0℃ AND `3.0 ≤ ws ≤ 7.0` (10절)

`{group}_air_density`는 LDAPS 기온·기압만으로 계산돼 있어(라벨/SCADA 의존 없음) 그대로 재사용해도 안전합니다.

In [8]:
GROUP_NEAREST_LDAPS = {"kpx_group_1": 5, "kpx_group_2": 6, "kpx_group_3": 12}
ICING_RISK_GROUPS = ["kpx_group_1", "kpx_group_2"]

CUT_IN, RATED, HIGH_WIND_THRESHOLD = 3.0, 12.0, 15.0
ICING_WS_MIN, ICING_WS_MAX = 3.0, 7.0
R_DRY_AIR = 287.05
RHO_STANDARD = 1.225


def add_fold_safe_ws_features(df, group_col, ws_col):
    """ws_col(그 fold에서 안전한 풍속 추정치)로부터 corrected/sq/cube/regime_*/high_wind_caution/icing_risk를
    그 자리에서 다시 계산한다. 공식·임계값은 03_features.ipynb 4/10/13절과 완전히 동일하다."""
    ws = df[ws_col]
    rho = df[f"{group_col}_air_density"]
    corrected = ws * (rho / RHO_STANDARD) ** (1 / 3)

    out = {}
    out[f"{ws_col}__corrected"] = corrected
    out[f"{ws_col}__sq"] = corrected ** 2
    out[f"{ws_col}__cube"] = corrected ** 3
    out[f"{ws_col}__regime_calm"] = (ws < CUT_IN).astype(int)
    out[f"{ws_col}__regime_ramp"] = ((ws >= CUT_IN) & (ws < RATED)).astype(int)
    out[f"{ws_col}__regime_rated"] = (ws >= RATED).astype(int)
    out[f"{ws_col}__high_wind_caution"] = (ws >= HIGH_WIND_THRESHOLD).astype(int)

    if group_col in ICING_RISK_GROUPS:
        grid_id = GROUP_NEAREST_LDAPS[group_col]
        temp_col = f"ldaps_g{grid_id}_heightAboveGround_2_t"
        is_freezing = df[temp_col] < 273.15
        in_range = ws.between(ICING_WS_MIN, ICING_WS_MAX)
        out[f"{ws_col}__icing_risk"] = (is_freezing & in_range).astype(int)

    return pd.DataFrame(out, index=df.index)

In [9]:
# 확인: A안 fold(ws_est_cv_2024_01)로 만든 fold-safe 파생값이 원본 03_features 파생값과 "비슷한 크기"인지 산점 비교
sample_g = "kpx_group_1"
fs_sample = add_fold_safe_ws_features(train, sample_g, f"{sample_g}_ws_est_cv_2024_01")
print("재계산본 regime 분포:")
print(fs_sample[f"{sample_g}_ws_est_cv_2024_01__regime_calm"].value_counts(normalize=True).round(4))
print()
print("원본(03_features, 전체데이터판) regime 분포 (참고용, 실제 학습엔 fold-safe본만 사용):")
print(train[f"{sample_g}_regime_calm"].value_counts(normalize=True).round(4))

재계산본 regime 분포:
kpx_group_1_ws_est_cv_2024_01__regime_calm
0    0.9797
1    0.0203
Name: proportion, dtype: float64

원본(03_features, 전체데이터판) regime 분포 (참고용, 실제 학습엔 fold-safe본만 사용):
kpx_group_1_regime_calm
0    0.981
1    0.019
Name: proportion, dtype: float64


**확인할 것**: 두 분포(재계산본 vs 원본)가 완전히 같지는 않아도(학습 데이터 범위가 다르므로) 비슷한 자릿수로 나오면 정상입니다. 극단적으로 다르면(예: 한쪽은 90%, 한쪽은 1%) 공식을 잘못 옮겼다는 신호이니 코드를 다시 확인해야 합니다.

## 6. 그룹별 모델 입력 프레임 빌더 + 채점 함수

**무엇을**: 지금까지 정의한 것들을 묶어서, "그룹 g, fold X" 조합마다 모델에 넣을 피처 테이블을 한 번에 만들어주는 함수(`build_group_feature_frame`)와, 대회 공식 산식(`src/metric.py`)으로 점수를 매기는 함수를 만듭니다.

**leaky 컬럼 제외 규칙**: `{group}_ws_est`, `{group}_ws_est_sq/_cube/_corrected`, `{group}_regime_calm/ramp/rated`, `{group}_high_wind_caution`, `{group}_icing_risk`, 그리고 **이 fold가 아닌 다른 cutoff의 `_cv_*` 컬럼**은 전부 빼고, 이 fold에 맞는 `ws_est_cv_{cutoff}` / `power_curve_est_cv_{cutoff}`와 5절에서 재계산한 fold-safe 파생값으로 채웁니다.

In [37]:
def all_cv_cols_for_group(g):
    return [c for c in train.columns if c.startswith(f"{g}_ws_est_cv_") or c.startswith(f"{g}_power_curve_est_cv_")]


# ⚠️ 2026-08-01 수정: 아래 목록에 f"{g}_power_curve_est"(원본)가 빠져 있어서 9~12절이 전부 누수 상태로 실행됐다.
#    이름을 하나하나 열거하는 방식은 언젠가 반드시 하나를 빠뜨린다.
#    그래서 열거 목록은 남기되, 진짜 방어선은 아래 assert_no_leak()의 "규칙 기반" 점검으로 옮긴다.
LABEL_DERIVED_MARKERS = ("ws_est", "power_curve_est")   # 라벨/SCADA를 참고해 만든 피처의 이름 표식


def leaky_cols_for_group(g):
    """fold-safe가 아닌(전체 데이터로 적합한) 라벨 의존 컬럼 — 검증에서 반드시 제외."""
    base = [f"{g}_ws_est", f"{g}_ws_est_sq", f"{g}_ws_est_cube", f"{g}_ws_est_corrected",
            f"{g}_power_curve_est",                      # ← 이번에 빠져 있던 컬럼
            f"{g}_regime_calm", f"{g}_regime_ramp", f"{g}_regime_rated", f"{g}_high_wind_caution"]
    if g in ICING_RISK_GROUPS:
        base.append(f"{g}_icing_risk")
    return base


def assert_no_leak(columns, g, cv_suffix):
    """규칙 기반 누수 점검.
    이름에 라벨 의존 표식이 들어간 컬럼은 '__' 앞부분(뿌리)을 구해서,
    그 뿌리가 이번 fold에서 허용된 2개가 아니면 무조건 실패시킨다.
    이 방식이었다면 power_curve_est 누수도 처음부터 걸렸다."""
    allowed = {f"{g}_ws_est_cv_{cv_suffix}", f"{g}_power_curve_est_cv_{cv_suffix}"}
    leaky_named = set(leaky_cols_for_group(g))
    bad = [c for c in columns
           if (any(m in c for m in LABEL_DERIVED_MARKERS) and c.split("__")[0] not in allowed)
           or c in leaky_named]
    assert not bad, f"{g}/{cv_suffix} 누수 의심 컬럼 {len(set(bad))}개: {sorted(set(bad))[:8]}"


def build_group_feature_frame(df, g, cv_suffix):
    """cv_suffix가 주어지면(A/B 검증용) fold-safe 버전 사용.
    cv_suffix=None이면(최종 전체 재학습 전용) 원본 ws_est/power_curve_est 그대로 사용."""
    all_cv = set(all_cv_cols_for_group(g))
    if cv_suffix is None:
        base_cols = [c for c in GROUP_SPECIFIC_COLS[g] if c not in all_cv]
        return df[COMMON_RAW_COLS + base_cols].copy()

    ws_col = f"{g}_ws_est_cv_{cv_suffix}"
    pc_col = f"{g}_power_curve_est_cv_{cv_suffix}"
    exclude = set(leaky_cols_for_group(g)) | (all_cv - {ws_col, pc_col})
    base_cols = [c for c in GROUP_SPECIFIC_COLS[g] if c not in exclude]
    fs = add_fold_safe_ws_features(df, g, ws_col)
    out = pd.concat([df[COMMON_RAW_COLS + base_cols], fs], axis=1)
    assert_no_leak(out.columns, g, cv_suffix)
    return out


# 안전 점검
for g in GROUP_COLS:
    for suffix in ["2023_07", "2024_01", "2024_07"]:
        frame = build_group_feature_frame(train, g, suffix)
        assert frame.isna().sum().sum() == 0, f"{g} {suffix}: NaN 발견"
print("전체 fold x 그룹 조합에서 NaN 없음 + 규칙 기반 누수 점검 통과")
for g in GROUP_COLS:
    print(f"   {g} 피처 수:", build_group_feature_frame(train, g, "2024_01").shape[1])

전체 fold x 그룹 조합에서 NaN 없음 + 규칙 기반 누수 점검 통과
   kpx_group_1 피처 수: 822
   kpx_group_2 피처 수: 822
   kpx_group_3 피처 수: 821


In [11]:
def score_predictions(actual_df, pred_df):
    """대회 공식 metric()을 그대로 사용 (src/metric.py 수정 금지 원칙)."""
    return metric(actual_df[TARGET_COLS], pred_df[TARGET_COLS])

## 7. 베이스라인 사다리 (`model-selection` 스킬 1절)

복잡한 모델의 개선 효과는 단순한 기준과 비교해야 의미가 있습니다. 아래에서 위로 4단계를 순서대로 쌓습니다.

1. **시간대×월 평균** (기상 정보 전혀 안 씀): 이 점수가 "기상 정보의 가치" 자체를 재는 바닥입니다.
2. **물리 파워커브** (ML 없음): SCADA로 만든 파워커브에 fold-safe 추정풍속만 통과시킨 예측. 도메인 지식만으로 어디까지 가는지 봅니다.
3. **선형회귀** (피처 8개뿐: 추정풍속·제곱·세제곱·풍향 sin/cos·시간 sin/cos·월 sin/cos)
4. **LightGBM 기본값** (876개 전체 피처, 본 게임의 출발점)

각 단계 점수 차이가 "무엇이 성능을 만드는가"를 분해해서 보여줍니다.

### 7-1. 베이스라인 0 — 시간대×월 평균 (기상 미사용)

In [38]:
def baseline_hour_month_mean(g, train_mask, valid_idx):
    train_g = train.loc[train_mask & train[g].notna()]
    profile = train_g.groupby([train_g["kst_dtm"].dt.hour, train_g["kst_dtm"].dt.month])[g].mean()
    overall_mean = train_g[g].mean()
    hm_keys = list(zip(train.loc[valid_idx, "kst_dtm"].dt.hour, train.loc[valid_idx, "kst_dtm"].dt.month))
    pred = pd.Series([profile.get(k, overall_mean) for k in hm_keys], index=valid_idx)
    return pred.clip(lower=0, upper=CAPACITY_KWH[g])

### 7-2. 베이스라인 1 — 물리 파워커브 (ML 없음)

In [39]:
def baseline_power_curve(g, cv_suffix, valid_idx):
    pc_col = f"{g}_power_curve_est_cv_{cv_suffix}"
    return train.loc[valid_idx, pc_col].clip(lower=0, upper=CAPACITY_KWH[g])

### 7-3. 베이스라인 2 — 선형회귀 (소수 피처: 추정풍속·제곱·세제곱·풍향/시간/월 sin·cos)

In [40]:
def linreg_feature_frame(g, cv_suffix):
    ws_col = f"{g}_ws_est_cv_{cv_suffix}"
    fs = add_fold_safe_ws_features(train, g, ws_col)
    return pd.concat([
        train[[ws_col, f"{g}_wd_sin", f"{g}_wd_cos", "hour_sin", "hour_cos", "month_sin", "month_cos"]],
        fs[[f"{ws_col}__sq", f"{ws_col}__cube"]],
    ], axis=1)


def baseline_linreg(g, cv_suffix, train_mask, valid_idx):
    X_full = linreg_feature_frame(g, cv_suffix)
    fit_idx = train_mask & train[g].notna()
    model = LinearRegression()
    model.fit(X_full.loc[fit_idx], train.loc[fit_idx, g])
    pred = pd.Series(model.predict(X_full.loc[valid_idx]), index=valid_idx)
    return pred.clip(lower=0, upper=CAPACITY_KWH[g])

### 7-4. 베이스라인 3 — LightGBM 기본값

**early stopping 설계**: 검증 fold의 데이터로 early stopping을 하면(몇 라운드에서 멈출지 결정하는 데 검증 정보를 쓰는 셈이라) 또 다른 형태의 정보 누수가 됩니다. 그래서 **학습 구간 내부**에서 시간순으로 마지막 10%를 떼어 early stopping 전용 검증셋으로 씁니다(공식 검증 fold와는 무관).

In [41]:
def baseline_lgbm(g, cv_suffix, train_mask, valid_idx):
    X_full = build_group_feature_frame(train, g, cv_suffix)
    fit_idx = train_mask & train[g].notna()
    fit_times = train.loc[fit_idx, "kst_dtm"]
    cutoff_es = fit_times.quantile(0.9)  # 학습 구간 내부에서 시간순 마지막 10%를 early stopping용으로 분리
    es_mask = fit_idx & (train["kst_dtm"] > cutoff_es)
    tr_mask = fit_idx & (train["kst_dtm"] <= cutoff_es)

    model = lgb.LGBMRegressor(objective="l1", random_state=SEED, n_estimators=2000, verbosity=-1)
    model.fit(
        X_full.loc[tr_mask], train.loc[tr_mask, g],
        eval_set=[(X_full.loc[es_mask], train.loc[es_mask, g])],
        eval_metric="l1",
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    pred = pd.Series(model.predict(X_full.loc[valid_idx]), index=valid_idx)
    return pred.clip(lower=0, upper=CAPACITY_KWH[g]), model.best_iteration_

### 7-5. 4개 베이스라인을 모든 fold(A안+B안 3개)에서 실행하고 표로 비교

In [42]:
baseline_results = []
lgbm_best_iters = {}

for fold_name, spec in FOLD_SPECS.items():
    cv_suffix = spec["cv_suffix"]
    train_mask = train["kst_dtm"] < CUTOFFS[cv_suffix]
    valid_mask = (train["kst_dtm"] >= spec["valid_start"]) & (train["kst_dtm"] < spec["valid_end"])
    valid_idx = train.index[valid_mask]
    actual_df = train.loc[valid_idx, TARGET_COLS]

    preds_by_baseline = {"0_시간x월평균": {}, "1_물리파워커브": {}, "2_선형회귀": {}, "3_LightGBM기본값": {}}
    for g in GROUP_COLS:
        preds_by_baseline["0_시간x월평균"][g] = baseline_hour_month_mean(g, train_mask, valid_idx)
        preds_by_baseline["1_물리파워커브"][g] = baseline_power_curve(g, cv_suffix, valid_idx)
        preds_by_baseline["2_선형회귀"][g] = baseline_linreg(g, cv_suffix, train_mask, valid_idx)
        pred3, best_iter = baseline_lgbm(g, cv_suffix, train_mask, valid_idx)
        preds_by_baseline["3_LightGBM기본값"][g] = pred3
        lgbm_best_iters[(fold_name, g)] = best_iter

    for baseline_name, group_preds in preds_by_baseline.items():
        pred_df = pd.DataFrame(group_preds, index=valid_idx)
        score, one_minus_nmae, ficr = score_predictions(actual_df, pred_df)
        baseline_results.append({
            "fold": fold_name, "baseline": baseline_name,
            "score": score, "1-NMAE": one_minus_nmae, "FICR": ficr,
        })
    print(f"{fold_name} 완료")

baseline_df = pd.DataFrame(baseline_results)
print(lgbm_best_iters)

KeyboardInterrupt: 

**확인할 것**: 각 fold가 끝날 때마다 `완료` 메시지가 뜹니다. LightGBM은 4 fold × 3 그룹 = 12번 학습하므로 몇 분 걸릴 수 있습니다. `lgbm_best_iters`에서 조기 종료 라운드 수가 2000(=n_estimators 상한)에 딱 붙어있으면 상한을 늘려야 한다는 신호이니 확인해 주세요.

In [17]:
pivot_score = baseline_df.pivot(index="baseline", columns="fold", values="score").round(4)
b_cols = [c for c in pivot_score.columns if c.startswith("B안")]
pivot_score["B안 평균"] = pivot_score[b_cols].mean(axis=1)
pivot_score["B안 표준편차"] = pivot_score[b_cols].std(axis=1)
display(pivot_score)

fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
baseline,,,,,,
0_시간x월평균,0.4334,0.4188,0.4486,0.4183,0.428567,0.017351
1_물리파워커브,0.5825,0.5550,0.5653,0.6006,0.573633,0.023915
2_선형회귀,0.5797,0.5429,0.5614,0.6001,0.568133,0.029188
3_LightGBM기본값,0.5965,0.5697,0.5947,0.6154,0.593267,0.022884


**사전 검증 결과(AI가 미리 같은 로직으로 돌려본 참고용 수치 — 민석님이 실제로 실행하면 이 값과 거의 같아야 정상입니다)**:

| baseline | A안(2024) | B안 fold1 | B안 fold2 | B안 fold3 |
|---|---|---|---|---|
| 0_시간x월평균 | 0.4334 | 0.4188 | 0.4486 | 0.4183 |
| 1_물리파워커브 | 0.5825 | 0.5550 | 0.5653 | 0.6006 |
| 2_선형회귀 | 0.5797 | 0.5429 | 0.5614 | 0.6001 |
| 3_LightGBM기본값 | 0.5965 | 0.5697 | 0.5947 | 0.6154 |

**확인할 것 — 사다리가 꼭 0<1<2<3 순서로 오르는 게 아닙니다**:
- 0 < 1 < 3은 모든 fold에서 뚜렷합니다(기상정보의 가치, 그리고 트리 모델+전체 피처의 가치).
- **2(선형회귀)가 1(물리 파워커브)보다 모든 fold에서 근소하게 낮습니다.** 이건 버그가 아니라 물리적으로 설명되는 결과입니다: 파워커브는 컷인(3m/s)~정격(12m/s)에서 급격히 꺾이고 그 이후엔 평평해지는 S자형 곡선인데, 3차 다항식(추정풍속+제곱+세제곱)으로 전체 구간을 한 번에 피팅하면 이 꺾임(특히 정격 이후 평평해지는 부분)을 잘 못 따라갑니다. 반면 물리 파워커브는 SCADA 실측을 0.5m/s 구간별로 그대로 평균낸 것이라 이 비선형을 그대로 담고 있습니다.
  - **도메인 관점**: 파워커브의 비선형성은 다항식보다 구간별 실측 평균(또는 트리 모델의 분기)이 더 잘 잡는다는, 익히 알려진 사실과 정확히 일치합니다.
  - **DS 관점**: 이 결과는 "피처를 억지로 늘린다고 항상 좋아지는 게 아니다"를 보여주는 좋은 예시이자, 3(LightGBM)이 1·2를 모두 확실히 이기는 것으로 미루어 트리 기반 비선형 모델이 이 문제에 적합하다는 것도 같이 확인해줍니다(model-selection 스킬 2절의 "GBDT 계열이 우세하다"는 가정과 부합).
- 만약 실제 실행 결과가 위 표와 크게 다르면(특히 3이 1·2보다 낮게 나오면) 피처/누수 문제를 의심하고 알려주세요.

In [18]:
display(baseline_df.pivot(index="baseline", columns="fold", values="1-NMAE").round(4))
display(baseline_df.pivot(index="baseline", columns="fold", values="FICR").round(4))

fold,A안(2024),B안 fold1,B안 fold2,B안 fold3
baseline,,,,
0_시간x월평균,0.7487,0.7312,0.7602,0.7366
1_물리파워커브,0.8531,0.8284,0.8472,0.8595
2_선형회귀,0.8601,0.8346,0.8533,0.8675
3_LightGBM기본값,0.8659,0.8475,0.8646,0.8736


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3
baseline,,,,
0_시간x월평균,0.1182,0.1063,0.1370,0.1000
1_물리파워커브,0.3119,0.2817,0.2835,0.3416
2_선형회귀,0.2994,0.2512,0.2695,0.3327
3_LightGBM기본값,0.3271,0.2920,0.3248,0.3572


## 8. 요약 및 다음 단계

**이번 노트북(1~7절)에서 확정한 것**:
- A안+B안 fold 구조와 fold-safe 컬럼 매핑 (`FOLD_SPECS`)
- `ws_est` 파생 6종(sq/cube/corrected/regime_*/high_wind_caution/icing_risk)의 fold-safe 재계산 방식 (`add_fold_safe_ws_features`) — 이번 세션에 새로 발견한 누수 위험을 해결
- 그룹별 피처 프레임 빌더 (`build_group_feature_frame`)와 채점 함수 (`score_predictions`)
- 베이스라인 사다리 4단계(시간×월 평균 / 물리 파워커브 / 선형회귀 / LightGBM 기본값) 실행 결과

**다음 단계 (민석님 실행·확인 후 이어서 작성)**:
1. 위 표를 보고 사다리가 예상대로(0<1<2<3) 올라가는지 확인
2. 확인되면 9절 이후에 **후보 모델 비교**(LightGBM 기본값 vs XGBoost vs CatBoost, 그룹별 3모델 vs 통합 1모델 구조, 타깃 스케일(kWh vs 이용률) 실험)를 추가로 작성
3. `reports/04_model_selection.md`에 이 단계 결과를 Why/How/Result/So-what 구조로 기록

---

> ## ⚠️ 아래 9~12절의 출력 숫자는 전부 "누수 상태"에서 나온 것입니다
>
> 2026-08-01에 6절 `leaky_cols_for_group()`에서 **`{group}_power_curve_est`(fold-safe가 아닌 원본)가 제외 목록에서 빠져 있는 것**을 발견했습니다. 이 컬럼은 2022~2024 전체 라벨로 적합한 파워커브를 통과시킨 값이라, **검증 구간의 정답이 피처 안에 들어 있습니다.** feature importance상 group_2/3에서는 gain 1위(14.7% / 13.8%)를 차지할 만큼 크게 쓰였습니다.
>
> 6절은 이미 수정했지만, **아래 9~12절 셀들의 저장된 출력은 수정 전 숫자**입니다.
>
> - ✅ **상대 비교(어느 모델·설정이 더 나은가)는 대체로 유효**합니다 — 모든 변형이 똑같은 누수를 공유했기 때문입니다
> - ❌ **절대 점수는 낙관적으로 부풀려져 있습니다.** 보고서에 인용하면 안 됩니다
> - ❌ 파워커브 피처 자체에 대한 판단(예: "group_3에서 파워커브가 gain 22%")은 신뢰할 수 없습니다
>
> **수정 후 재실행은 13절에서 합니다.** 9~12절의 무거운 셀은 다시 돌릴 필요 없습니다.

---

## 9. 후보 모델 비교 — LightGBM / XGBoost / CatBoost (+ 단조 제약) / MLP

**Why**: 7절 베이스라인에서 LightGBM 기본값이 물리 파워커브·선형회귀를 확실히 이겼습니다. 이제 트리 계열 라이브러리 3종(LightGBM/XGBoost/CatBoost)을 공정하게 비교하고, 민석님이 제안하신 두 아이디어를 더합니다.

1. **단조 제약(monotonic constraint)**: "추정풍속이 세지면 예측 발전량도 커지거나 그대로여야 한다"는 물리 법칙(파워커브는 원래 단조증가)을 트리 모델 구조에 직접 강제합니다. `{group}_ws_est_cv_*`, `{group}_power_curve_est_cv_*` 두 컬럼에만 "증가 방향(+1)" 제약을 겁니다 — 나머지 피처(결빙 위험, 돌풍성, 그룹ID 등)는 단조성 근거가 없어 제약을 걸지 않습니다.
2. **MLP(신경망)**: `model-selection` 스킬 원칙("딥러닝은 GBDT 대비 명확한 개선을 보일 때만 도입")에 따라, 지금 GBDT와 나란히 비교해서 실제로 이기는지 확인합니다. 이겨야만 최종 앙상블(9~10단계) 후보로 남습니다.

**공정성 유지**: 7종 모두 같은 `build_group_feature_frame`(fold-safe 피처), 같은 fold, 같은 손실 기준(MAE 계열), 같은 early stopping 방식(학습구간 내부 시간순 마지막 10%)을 씁니다.

### 9-1. 단조 제약 스펙 만들기

**무엇을**: 피처 컬럼 순서에 맞춰 "이 컬럼은 +1(증가 방향 제약), 나머지는 0(제약 없음)"인 리스트를 만듭니다. 라이브러리마다 이 리스트를 넘기는 형식이 달라서(LightGBM/CatBoost는 리스트 그대로, XGBoost는 문자열), 변환은 각 모델 함수 안에서 처리합니다.

In [19]:
def build_monotone_spec(columns, g, cv_suffix):
    """ws_est_cv/power_curve_est_cv 두 컬럼만 +1(비감소) 제약. 나머지는 0(제약 없음)."""
    up_cols = {f"{g}_ws_est_cv_{cv_suffix}", f"{g}_power_curve_est_cv_{cv_suffix}"}
    return [1 if c in up_cols else 0 for c in columns]

### 9-2. GBDT 3종 (LightGBM/XGBoost/CatBoost) 학습 함수 — 단조 제약 켜기/끄기 지원

**작은 함정 하나 발견**: LightGBM은 `monotone_constraints`를 `regression_l1`(MAE)이나 `quantile` 목적함수와 같이 못 씁니다(제약 알고리즘이 2차 도함수를 쓰는데, L1 계열은 2차 도함수가 0이라 라이브러리 자체에서 에러를 냅니다 — 직접 돌려보고 확인함). 그래서 **LightGBM에 단조 제약을 걸 때만 손실을 `huber`로 바꿉니다**(`model-selection` 스킬의 "비교 실험" 손실 목록에 있는 선택지 — MAE와 비슷하게 이상치에 강건하면서도 2차 도함수가 있어 제약과 호환됨). XGBoost·CatBoost는 이런 제약이 없어 MAE 그대로 씁니다. early stopping 기준(`eval_metric`)은 네 경우 모두 MAE로 통일해서, 학습 손실이 달라도 "언제 멈출지" 판단 기준은 공정하게 맞췄습니다.

In [20]:
def fit_predict_gbdt(model_type, g, cv_suffix, train_mask, valid_idx, monotone):
    X_full = build_group_feature_frame(train, g, cv_suffix)
    fit_idx = train_mask & train[g].notna()
    fit_times = train.loc[fit_idx, "kst_dtm"]
    cutoff_es = fit_times.quantile(0.9)
    es_mask = fit_idx & (train["kst_dtm"] > cutoff_es)
    tr_mask = fit_idx & (train["kst_dtm"] <= cutoff_es)
    Xtr, ytr = X_full.loc[tr_mask], train.loc[tr_mask, g]
    Xes, yes = X_full.loc[es_mask], train.loc[es_mask, g]
    mono_spec = build_monotone_spec(X_full.columns, g, cv_suffix) if monotone else None

    if model_type == "lightgbm":
        objective = "huber" if monotone else "l1"  # 단조 제약 + L1 조합은 LightGBM이 지원 안 함
        params = dict(objective=objective, random_state=SEED, n_estimators=2000, verbosity=-1)
        if mono_spec is not None:
            params["monotone_constraints"] = mono_spec
        model = lgb.LGBMRegressor(**params)
        model.fit(Xtr, ytr, eval_set=[(Xes, yes)], eval_metric="l1",
                   callbacks=[lgb.early_stopping(50, verbose=False)])
    elif model_type == "xgboost":
        params = dict(objective="reg:absoluteerror", random_state=SEED, n_estimators=2000,
                      early_stopping_rounds=50, eval_metric="mae")
        if mono_spec is not None:
            params["monotone_constraints"] = "(" + ",".join(map(str, mono_spec)) + ")"
        model = xgb.XGBRegressor(**params)
        model.fit(Xtr, ytr, eval_set=[(Xes, yes)], verbose=False)
    elif model_type == "catboost":
        params = dict(loss_function="MAE", random_state=SEED, n_estimators=2000,
                      early_stopping_rounds=50, verbose=False)
        if mono_spec is not None:
            params["monotone_constraints"] = mono_spec
        model = CatBoostRegressor(**params)
        model.fit(Xtr, ytr, eval_set=(Xes, yes))
    else:
        raise ValueError(model_type)

    pred = pd.Series(model.predict(X_full.loc[valid_idx]), index=valid_idx)
    return pred.clip(lower=0, upper=CAPACITY_KWH[g])

### 9-3. MLP (PyTorch)

**왜 스케일링을 새로 하는가**: 트리 모델(LightGBM 등)은 피처 크기(단위)가 달라도 분기 기준만 찾으면 되니 상관없지만, 신경망은 입력 크기가 들쭉날쭉하면(기온 ~270K, 기압 ~100000Pa, 더미 0/1 등) 학습이 잘 안 됩니다. 그래서 학습 구간의 평균·표준편차로 표준화(z-score)합니다.

**왜 타깃(발전량)을 설비용량으로 나누는가**: 발전량(kWh) 그대로 쓰면 그룹마다 스케일이 달라(21600 vs 21000) 손실 함수 값이 커져서 학습이 불안정해집니다. 0~1 사이 이용률로 바꾸면 신경망 학습이 안정적입니다(예측할 때 다시 설비용량을 곱해 kWh로 되돌립니다). **주의**: 이건 MLP 학습 안정성을 위한 임시 스케일 변환일 뿐, "타깃 스케일(kWh vs 이용률)을 어떤 모델 구조로 최종 채택할지"는 이후 별도 실험(모델 구조 결정)에서 다룹니다.

**구조**: 입력층 → 128 → 64 → 출력 1 (ReLU), Adam 옵티마이저, L1 손실(MAE와 동일 — 다른 후보와 손실 기준 통일), early stopping은 GBDT와 같은 방식(학습구간 내부 시간순 마지막 10%).

In [22]:
import torch
import torch.nn as nn


class SimpleMLP(nn.Module):
    def __init__(self, n_features, hidden=(128, 64)):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def fit_predict_mlp(g, cv_suffix, train_mask, valid_idx, max_epochs=200, patience=20):
    capacity = CAPACITY_KWH[g]
    X_full = build_group_feature_frame(train, g, cv_suffix)
    fit_idx = train_mask & train[g].notna()
    fit_times = train.loc[fit_idx, "kst_dtm"]
    cutoff_es = fit_times.quantile(0.9)
    es_mask = fit_idx & (train["kst_dtm"] > cutoff_es)
    tr_mask = fit_idx & (train["kst_dtm"] <= cutoff_es)

    mean = X_full.loc[tr_mask].mean()
    std = X_full.loc[tr_mask].std().replace(0, 1)

    def to_tensor(df):
        return torch.tensor(((df - mean) / std).to_numpy(), dtype=torch.float32)

    Xtr_t = to_tensor(X_full.loc[tr_mask])
    ytr_t = torch.tensor(train.loc[tr_mask, g].to_numpy() / capacity, dtype=torch.float32)
    Xes_t = to_tensor(X_full.loc[es_mask])
    yes_t = torch.tensor(train.loc[es_mask, g].to_numpy() / capacity, dtype=torch.float32)
    Xva_t = to_tensor(X_full.loc[valid_idx])

    torch.manual_seed(SEED)
    model = SimpleMLP(Xtr_t.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.L1Loss()

    best_es, best_state, patience_left = float("inf"), None, patience
    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(Xtr_t), ytr_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            es_loss = loss_fn(model(Xes_t), yes_t).item()
        if es_loss < best_es - 1e-6:
            best_es, best_state, patience_left = es_loss, {k: v.clone() for k, v in model.state_dict().items()}, patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred_va = model(Xva_t).numpy() * capacity
    return pd.Series(pred_va, index=valid_idx).clip(lower=0, upper=capacity), epoch

### 9-4. 7개 후보(LightGBM/XGBoost/CatBoost × 단조제약 유무 + MLP)를 모든 fold에서 실행

In [23]:
candidate_results = []
mlp_epochs_used = {}

for fold_name, spec in FOLD_SPECS.items():
    cv_suffix = spec["cv_suffix"]
    train_mask = train["kst_dtm"] < CUTOFFS[cv_suffix]
    valid_mask = (train["kst_dtm"] >= spec["valid_start"]) & (train["kst_dtm"] < spec["valid_end"])
    valid_idx = train.index[valid_mask]
    actual_df = train.loc[valid_idx, TARGET_COLS]

    variant_names = ["lightgbm", "lightgbm_mono", "xgboost", "xgboost_mono", "catboost", "catboost_mono", "mlp"]
    preds_by_variant = {v: {} for v in variant_names}
    for g in GROUP_COLS:
        preds_by_variant["lightgbm"][g] = fit_predict_gbdt("lightgbm", g, cv_suffix, train_mask, valid_idx, False)
        preds_by_variant["lightgbm_mono"][g] = fit_predict_gbdt("lightgbm", g, cv_suffix, train_mask, valid_idx, True)
        preds_by_variant["xgboost"][g] = fit_predict_gbdt("xgboost", g, cv_suffix, train_mask, valid_idx, False)
        preds_by_variant["xgboost_mono"][g] = fit_predict_gbdt("xgboost", g, cv_suffix, train_mask, valid_idx, True)
        preds_by_variant["catboost"][g] = fit_predict_gbdt("catboost", g, cv_suffix, train_mask, valid_idx, False)
        preds_by_variant["catboost_mono"][g] = fit_predict_gbdt("catboost", g, cv_suffix, train_mask, valid_idx, True)
        pred_mlp, n_epoch = fit_predict_mlp(g, cv_suffix, train_mask, valid_idx)
        preds_by_variant["mlp"][g] = pred_mlp
        mlp_epochs_used[(fold_name, g)] = n_epoch

    for variant_name, group_preds in preds_by_variant.items():
        pred_df = pd.DataFrame(group_preds, index=valid_idx)
        score, one_minus_nmae, ficr = score_predictions(actual_df, pred_df)
        candidate_results.append({"fold": fold_name, "model": variant_name, "score": score, "1-NMAE": one_minus_nmae, "FICR": ficr})
    print(f"{fold_name} 완료")

candidate_df = pd.DataFrame(candidate_results)
print(mlp_epochs_used)

d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

A안(2024) 완료


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

B안 fold1 완료


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

B안 fold2 완료


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

B안 fold3 완료
{('A안(2024)', 'kpx_group_1'): 97, ('A안(2024)', 'kpx_group_2'): 104, ('A안(2024)', 'kpx_group_3'): 64, ('B안 fold1', 'kpx_group_1'): 100, ('B안 fold1', 'kpx_group_2'): 38, ('B안 fold1', 'kpx_group_3'): 36, ('B안 fold2', 'kpx_group_1'): 97, ('B안 fold2', 'kpx_group_2'): 104, ('B안 fold2', 'kpx_group_3'): 64, ('B안 fold3', 'kpx_group_1'): 86, ('B안 fold3', 'kpx_group_2'): 74, ('B안 fold3', 'kpx_group_3'): 47}


**확인할 것**: fold마다 7개 후보 × 3그룹 = 21번 학습이 돕니다(GBDT 18번 + MLP 3번). 전체 4 fold라 시간이 꽤 걸릴 수 있습니다(수십 분 단위 각오). `mlp_epochs_used`가 200(=max_epochs 상한)에 자꾸 붙어있으면 상한을 늘려야 한다는 신호입니다.

In [24]:
pivot_candidate = candidate_df.pivot(index="model", columns="fold", values="score").round(4)
b_cols = [c for c in pivot_candidate.columns if c.startswith("B안")]
pivot_candidate["B안 평균"] = pivot_candidate[b_cols].mean(axis=1)
pivot_candidate["B안 표준편차"] = pivot_candidate[b_cols].std(axis=1)
display(pivot_candidate)

fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
catboost,0.5983,0.5688,0.5903,0.6149,0.591333,0.023067
catboost_mono,0.3666,0.3488,0.3820,0.3344,0.355067,0.024411
lightgbm,0.5965,0.5697,0.5947,0.6154,0.593267,0.022884
lightgbm_mono,0.4177,0.4274,0.4334,0.4096,0.423467,0.012378
mlp,0.6019,0.5652,0.6005,0.6019,0.589200,0.020796
xgboost,0.5864,0.5648,0.5778,0.6073,0.583300,0.021777
xgboost_mono,0.5870,0.5576,0.5850,0.6094,0.584000,0.025914


**확인할 것 — 세 가지 질문에 답하기 위한 표입니다**:
1. **어떤 GBDT 라이브러리가 제일 좋은가** (`lightgbm` vs `xgboost` vs `catboost`, 단조 제약 없는 버전끼리 비교)
2. **단조 제약이 도움이 되는가** (`xxx` vs `xxx_mono` 짝을 비교 — A안과 B안 3-fold 모두에서 개선돼야 "효과 있음"으로 채택. 한쪽만 개선되면 `timeseries-validation` 스킬 원칙대로 "효과 불확실")
3. **MLP가 GBDT를 이기는가** (`mlp`가 최고 GBDT 변형보다 낮으면, `model-selection` 스킬 원칙에 따라 지금 단계에서는 후보에서 제외 — 나중에 앙상블 다양성 확보용으로만 재고려)

### 9-5. 단조 제약 진단 — 범인은 "제약"인가 "손실함수"인가

9-4절 표에서 `lightgbm_mono`(0.42)와 `catboost_mono`(0.36)가 폭락했습니다. 하지만 이걸 그대로 "단조 제약은 나쁘다"로 읽으면 안 됩니다. **비교 조건이 서로 달랐기 때문입니다.**

- `xgboost` vs `xgboost_mono`는 **공정한 비교**였습니다. 손실함수(`reg:absoluteerror`)도, early stopping도 똑같고 제약 유무만 달랐습니다. 결과는 0.5864 → 0.5870 (거의 변화 없음).
- `lightgbm_mono`는 **불공정한 비교**였습니다. LightGBM이 `l1`(MAE) + 단조 제약 조합을 지원하지 않아서 손실을 `huber`로 바꿨는데, 이때 **`alpha` 기본값 0.9를 그대로 뒀습니다.**

여기서 `alpha`(후버 손실의 임계값)가 문제입니다. 후버 손실은 "오차가 alpha보다 작으면 제곱(부드럽게), 크면 선형(강건하게)" 처리하는 손실인데, **이 alpha는 타깃과 같은 단위**입니다. 우리 타깃은 발전량 kWh(0~21,600)인데 alpha가 0.9라는 건 "0.9 kWh보다 큰 오차는 전부 선형 구간"이라는 뜻입니다. 선형 구간에서 후버의 기울기(그래디언트)는 **최대 alpha=0.9로 잘립니다.** 학습률은 그대로인데 한 걸음 크기만 1/1000 수준으로 줄어드는 셈이라, 트리 2000개를 다 써도 모델이 거의 학습되지 않습니다.

> 비유: 목적지는 20km 앞인데 "한 걸음은 최대 90cm"라고 정해놓고 2000걸음만 걷게 한 것. 방향은 맞지만 도착을 못 합니다.

**그래서 대조군 4개를 돌려 이 가설을 숫자로 확인합니다.**

| 변형 | 손실함수 | alpha | 단조 제약 | 이 변형의 역할 |
|---|---|---|---|---|
| `lgb_l1` | l1(MAE) | – | ✗ | 9절 `lightgbm`과 동일 조건 (재현 확인 + 기준선) |
| `lgb_huber_a0.9` | huber | 0.9 (기본값) | **✗** | **핵심 대조군.** 제약을 뺐는데도 0.42대면 → 범인은 손실함수 |
| `lgb_huber_auto` | huber | 설비용량의 2% | ✗ | alpha를 타깃 스케일에 맞춘 정상 후버 |
| `lgb_huber_auto_mono` | huber | 설비용량의 2% | ✓ | **공정한 단조 제약 비교** (바로 윗줄과 손실 조건이 동일) |

alpha를 "설비용량의 2%"로 잡은 근거: FICR 문턱이 설비용량의 6%이므로, 그보다 안쪽(2% 이내)의 작은 오차만 부드럽게(제곱) 다루고 그 밖은 MAE처럼 강건하게 다루자는 의도입니다. group_1 기준 432 kWh입니다.

**읽는 법**
1. `lgb_huber_a0.9` ≈ 0.42 (9절 `lightgbm_mono`와 비슷) → 범인은 손실함수 확정, 단조 제약은 무죄
2. `lgb_huber_auto_mono` vs `lgb_huber_auto` → 이게 LightGBM에서의 **진짜** 단조 제약 효과

또 이 셀에서는 앞으로 계속 쓸 **예측 캐시(`PRED_CACHE`)** 인프라도 같이 만듭니다. 한 번 학습한 예측값을 저장해두면, 나중에 앙상블 가중치를 바꿔가며 실험할 때 **다시 학습할 필요 없이** 저장된 예측을 섞기만 하면 되기 때문입니다(앙상블 실험이 몇 초 만에 끝납니다).

In [25]:
import warnings
warnings.filterwarnings("ignore", message=".*'eval_set' is deprecated.*")

# ── (1) fold 정보 미리 계산해두기 (매번 다시 만들지 않도록) ──────────────
FOLD_INFO = {}
for fold_name, spec in FOLD_SPECS.items():
    cv_suffix = spec["cv_suffix"]
    train_mask = train["kst_dtm"] < CUTOFFS[cv_suffix]
    valid_mask = (train["kst_dtm"] >= spec["valid_start"]) & (train["kst_dtm"] < spec["valid_end"])
    valid_idx = train.index[valid_mask]
    FOLD_INFO[fold_name] = dict(
        cv_suffix=cv_suffix, train_mask=train_mask, valid_idx=valid_idx,
        actual_df=train.loc[valid_idx, TARGET_COLS],
    )

# ── (2) 예측 캐시: (fold, 변형이름, 그룹) -> 예측 Series ─────────────────
PRED_CACHE = {}


def run_variant(variant, fit_fn, verbose=True):
    """fit_fn(g, cv_suffix, train_mask, valid_idx) -> pd.Series 를
    모든 fold x 그룹에 돌려서 PRED_CACHE에 저장하고, fold별 점수표를 돌려준다."""
    rows = []
    for fold_name, info in FOLD_INFO.items():
        preds = {}
        for g in GROUP_COLS:
            p = fit_fn(g, info["cv_suffix"], info["train_mask"], info["valid_idx"])
            PRED_CACHE[(fold_name, variant, g)] = p
            preds[g] = p
        s, n, f = score_predictions(info["actual_df"], pd.DataFrame(preds, index=info["valid_idx"]))
        rows.append({"fold": fold_name, "model": variant, "score": s, "1-NMAE": n, "FICR": f})
        if verbose:
            print(f"  [{variant}] {fold_name}: score={s:.4f}  (1-NMAE={n:.4f}, FICR={f:.4f})")
    return pd.DataFrame(rows)


B_FOLDS = ["B안 fold1", "B안 fold2", "B안 fold3"]


def summarize(result_dfs):
    """여러 run_variant 결과를 모아 A안/B안 3fold/B안 평균·표준편차 표로."""
    df = pd.concat(result_dfs, ignore_index=True)
    p = df.pivot(index="model", columns="fold", values="score")
    p["B안 평균"] = p[B_FOLDS].mean(axis=1)
    p["B안 표준편차"] = p[B_FOLDS].std(axis=1)
    return p.round(4).sort_values("B안 평균", ascending=False)


# ── (3) 유연한 LightGBM 러너 (손실함수/alpha/단조제약/샘플가중 전부 지원) ──
def make_sample_weight(y, g, mode):
    """산식을 학습에 반영하기 위한 표본 가중치.
    mode=None      : 가중 없음(기존)
    mode='eval_only': 채점 대상(실제>=설비용량10%)에 1.0, 나머지엔 0.1
    mode='actual'  : 가중 = 실제발전량/설비용량 (하한 0.1) — FICR의 actual 가중을 모사
    """
    if mode is None:
        return None
    ratio = np.asarray(y, dtype=float) / CAPACITY_KWH[g]
    if mode == "eval_only":
        return np.where(ratio >= 0.10, 1.0, 0.1)
    if mode == "actual":
        return np.maximum(ratio, 0.1)
    raise ValueError(mode)


def fit_predict_lgbm(g, cv_suffix, train_mask, valid_idx, *, objective="l1", alpha=None,
                     monotone=False, weight_mode=None, n_estimators=2000):
    """LightGBM 한 번 학습 후 검증 구간 예측. (예측Series, 모델객체) 반환."""
    X_full = build_group_feature_frame(train, g, cv_suffix)
    fit_idx = train_mask & train[g].notna()
    cutoff_es = train.loc[fit_idx, "kst_dtm"].quantile(0.9)   # 학습구간 뒤 10%를 early stopping용으로
    tr_mask = fit_idx & (train["kst_dtm"] <= cutoff_es)
    es_mask = fit_idx & (train["kst_dtm"] > cutoff_es)
    Xtr, ytr = X_full.loc[tr_mask], train.loc[tr_mask, g]
    Xes, yes = X_full.loc[es_mask], train.loc[es_mask, g]

    params = dict(objective=objective, random_state=SEED, n_estimators=n_estimators,
                  verbosity=-1, importance_type="gain")
    if alpha is not None:
        params["alpha"] = alpha
    if monotone:
        params["monotone_constraints"] = build_monotone_spec(X_full.columns, g, cv_suffix)

    w_tr = make_sample_weight(ytr, g, weight_mode)
    w_es = make_sample_weight(yes, g, weight_mode)

    model = lgb.LGBMRegressor(**params)
    model.fit(Xtr, ytr, sample_weight=w_tr,
              eval_set=[(Xes, yes)],
              eval_sample_weight=[w_es] if w_es is not None else None,
              eval_metric="l1", callbacks=[lgb.early_stopping(50, verbose=False)])

    pred = pd.Series(model.predict(X_full.loc[valid_idx]), index=valid_idx)
    return pred.clip(lower=0, upper=CAPACITY_KWH[g]), model


def lgbm_fit_fn(**kwargs):
    """run_variant에 넘길 수 있는 형태로 감싸기. alpha='auto'면 설비용량의 2%로 자동 설정."""
    def fit_fn(g, cv_suffix, train_mask, valid_idx):
        kw = dict(kwargs)
        if kw.get("alpha") == "auto":
            kw["alpha"] = 0.02 * CAPACITY_KWH[g]
        return fit_predict_lgbm(g, cv_suffix, train_mask, valid_idx, **kw)[0]
    return fit_fn


print("인프라 준비 완료. fold 수:", len(FOLD_INFO))

인프라 준비 완료. fold 수: 4


In [26]:
# 9-5 진단 실행: LightGBM 4개 변형 x 4 fold x 3 그룹 = 48번 학습 (LightGBM만이라 비교적 빠름)
diag_specs = {
    "lgb_l1":               dict(objective="l1"),
    "lgb_huber_a0.9":       dict(objective="huber", alpha=0.9),
    "lgb_huber_auto":       dict(objective="huber", alpha="auto"),
    "lgb_huber_auto_mono":  dict(objective="huber", alpha="auto", monotone=True),
}

diag_results = []
for name, kw in diag_specs.items():
    print(f"=== {name} ===")
    diag_results.append(run_variant(name, lgbm_fit_fn(**kw)))

diag_table = summarize(diag_results)
display(diag_table)

=== lgb_l1 ===
  [lgb_l1] A안(2024): score=0.5965  (1-NMAE=0.8659, FICR=0.3271)
  [lgb_l1] B안 fold1: score=0.5697  (1-NMAE=0.8475, FICR=0.2920)
  [lgb_l1] B안 fold2: score=0.5947  (1-NMAE=0.8646, FICR=0.3248)
  [lgb_l1] B안 fold3: score=0.6154  (1-NMAE=0.8736, FICR=0.3572)
=== lgb_huber_a0.9 ===
  [lgb_huber_a0.9] A안(2024): score=0.4177  (1-NMAE=0.7317, FICR=0.1036)
  [lgb_huber_a0.9] B안 fold1: score=0.4275  (1-NMAE=0.7460, FICR=0.1089)
  [lgb_huber_a0.9] B안 fold2: score=0.4334  (1-NMAE=0.7461, FICR=0.1207)
  [lgb_huber_a0.9] B안 fold3: score=0.4094  (1-NMAE=0.7273, FICR=0.0915)
=== lgb_huber_auto ===
  [lgb_huber_auto] A안(2024): score=0.5954  (1-NMAE=0.8668, FICR=0.3239)
  [lgb_huber_auto] B안 fold1: score=0.5557  (1-NMAE=0.8410, FICR=0.2705)
  [lgb_huber_auto] B안 fold2: score=0.5951  (1-NMAE=0.8653, FICR=0.3250)
  [lgb_huber_auto] B안 fold3: score=0.6060  (1-NMAE=0.8726, FICR=0.3393)
=== lgb_huber_auto_mono ===
  [lgb_huber_auto_mono] A안(2024): score=0.5897  (1-NMAE=0.8656, FICR=0.3139)
  

fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
lgb_l1,0.5965,0.5697,0.5947,0.6154,0.5933,0.0229
lgb_huber_auto_mono,0.5897,0.5642,0.5886,0.6071,0.5866,0.0215
lgb_huber_auto,0.5954,0.5557,0.5951,0.6060,0.5856,0.0264
lgb_huber_a0.9,0.4177,0.4275,0.4334,0.4094,0.4234,0.0125


**확인할 것**

1. `lgb_l1`이 9-4절 표의 `lightgbm`(A안 0.5965 / B평균 0.5933)과 **거의 같아야** 합니다. 다르면 캐시 인프라를 옮기는 과정에서 뭔가 어긋난 것이니 알려주세요.
2. `lgb_huber_a0.9`가 0.42 근처면 → **가설 확정**: 9-4절의 `lightgbm_mono` 폭락은 단조 제약이 아니라 후버 손실의 alpha 설정 때문. 단조 제약은 누명을 쓴 것입니다.
3. `lgb_huber_auto_mono` vs `lgb_huber_auto` 차이가 LightGBM에서의 **진짜** 단조 제약 효과입니다. A안과 B안 3-fold **모두**에서 올라가야만 "효과 있음"으로 채택합니다(`timeseries-validation` 원칙). 한쪽만 오르면 "효과 불확실"로 두고 넘어갑니다.
4. 덤으로 `lgb_huber_auto`가 `lgb_l1`을 이기는지도 보세요. 이기면 **손실함수 자체가 개선 레버**라는 뜻이라 10절에서 더 파볼 가치가 있습니다.

## 10. 산식-인지 학습 (metric-aware) — 평가 산식에 맞춰 학습을 바꾼다

지금까지는 "발전량을 잘 맞추자"는 일반적인 회귀 문제로 풀었습니다. 하지만 우리 점수는 일반적인 회귀 오차가 아닙니다. `src/metric.py`를 한 줄씩 뜯어보면 **최적화해야 할 대상이 꽤 다르다**는 게 드러납니다.

### 산식 해부 — 네 가지 사실

```python
valid = actual >= capacity * 0.10                    # ① 실제 발전량 10% 미만은 채점 제외
error_rate = np.abs(forecast - actual) / capacity    # ② 실제값이 아니라 "설비용량"으로 나눔
group_nmae.append(np.mean(error_rate))               #    → 1-NMAE는 이 오차율의 단순 평균
unit_price = 4.0 if error_rate <= 0.06 else (3.0 if error_rate <= 0.08 else 0.0)   # ③ 계단
group_ficr.append(np.sum(actual * unit_price) / np.sum(actual * 4.0))              # ④ actual 가중
```

**① 채점 대상이 아닌 행이 존재한다.** 실제 발전량이 설비용량의 10%(group_1 기준 2,160 kWh) 미만인 시간대는 아무리 정확하게 맞혀도 **점수에 1도 기여하지 않습니다.** 그런데 지금 우리 모델은 그 행들까지 똑같은 비중으로 학습하고 있습니다. 모델의 "집중력"을 채점되지 않는 구간에 나눠 쓰고 있는 셈입니다.

**② 오차율의 분모가 실제값이 아니라 설비용량이다.** 이게 가장 흔히 오해하는 부분입니다. "발전량 1,000 kWh인데 6% 안에 들려면 60 kWh 안에 맞춰야 한다"가 **아닙니다.** 분모가 늘 21,600으로 고정이라, **어느 시간대든 ±1,296 kWh라는 똑같은 절대 오차 예산**을 받습니다. 발전량이 클 때가 오히려 상대적으로 관대합니다.
> 비유: 시험을 상대평가가 아니라 "문항마다 정답에서 ±5점 안이면 만점" 방식으로 채점하는 것. 배점이 큰 문항이든 작은 문항이든 허용 오차는 똑같습니다.

**③ FICR은 평균이 아니라 계단이다.** 오차율 5.9%와 0.1%는 **똑같이 4원**입니다. 반대로 6.1%는 3원으로 떨어집니다. 즉 이미 6% 안에 들어온 예측을 더 정밀하게 만드는 건 FICR에 아무 이득이 없고, **6.1%짜리를 5.9%로 밀어넣는 것만이 이득**입니다. 이건 "평균 오차를 줄여라"와 미묘하게 다른 목표입니다.

**④ FICR은 실제 발전량으로 가중된다.** 분자·분모 모두 `actual`을 곱합니다. 발전량이 20,000 kWh인 시간대 하나를 놓치는 손해가, 3,000 kWh인 시간대 하나를 놓치는 손해보다 **6.7배 큽니다.**

### 여기서 나오는 세 가지 개선 레버

| 레버 | 근거 | 어디서 실험 |
|---|---|---|
| **A. 표본 가중** — 채점 대상 행과 발전량 큰 행에 학습 비중을 몰아준다 | ①·④ | 10-3 |
| **B. 예측 위치 이동(분위수)** — 계단 문턱을 최대한 많이 통과하도록 예측을 살짝 밀어준다 | ②·③ | 10-4 |
| **C. 앙상블** — 서로 다른 모델의 오차를 상쇄시켜 문턱 통과율을 올린다 | ③ | 11절 |

레버 B를 조금 더 풀어 설명하면: 만약 모델이 발전량이 큰 시간대를 **체계적으로 과소예측**한다면(회귀 모델은 평균으로 끌어당기는 성질이 있어 흔한 현상입니다), 예측을 전체적으로 조금 위로 올리는 것만으로도 문턱을 통과하는 표본이 늘어날 수 있습니다. 이걸 하는 정석적인 방법이 **분위수 회귀(quantile regression)** 입니다. τ=0.5면 중앙값(=MAE 최적), τ=0.6이면 "실제값이 예측보다 클 확률 60%" 지점을 맞추므로 예측이 위로 이동합니다.

먼저 **지금 우리 모델이 산식 관점에서 어디서 점수를 잃고 있는지**부터 숫자로 보겠습니다.

### 10-1. 비교 대상 모델들의 예측을 캐시에 채우기

9-4절에서 이미 학습은 했지만 **예측값을 저장해두지 않아서**(점수만 계산하고 버림) 앙상블 실험에 쓸 수가 없습니다. 그래서 앙상블에 쓸 후보 3종(XGBoost / CatBoost / MLP)을 다시 돌려 캐시에 넣습니다. LightGBM(`lgb_l1`)은 9-5절에서 이미 캐시에 들어갔으므로 다시 안 돌립니다.

단조 제약 변형들은 9-5절 결론에 따라 앙상블 후보에서 제외합니다(제약 자체가 무해하더라도 개선이 없으므로 굳이 넣을 이유가 없습니다).

⚠️ **시간 안내**: CatBoost가 병목입니다(HANDOFF 주의사항 13번). 너무 오래 걸리면 `INCLUDE_CATBOOST = False`로 바꾸고 다시 돌리세요 — CatBoost 없이도 LightGBM+XGBoost+MLP 3종으로 앙상블 실험은 충분히 가능합니다.

In [27]:
INCLUDE_CATBOOST = True   # 너무 느리면 False로 바꾸고 재실행

cache_results = []
print("=== xgboost ===")
cache_results.append(run_variant("xgboost", lambda g, c, t, v: fit_predict_gbdt("xgboost", g, c, t, v, False)))
print("=== mlp ===")
cache_results.append(run_variant("mlp", lambda g, c, t, v: fit_predict_mlp(g, c, t, v)[0]))
if INCLUDE_CATBOOST:
    print("=== catboost (느림) ===")
    cache_results.append(run_variant("catboost", lambda g, c, t, v: fit_predict_gbdt("catboost", g, c, t, v, False)))

display(summarize(diag_results + cache_results))
print("\n캐시에 저장된 예측 개수:", len(PRED_CACHE), "(변형수 x 4fold x 3그룹)")

=== xgboost ===
  [xgboost] A안(2024): score=0.5864  (1-NMAE=0.8610, FICR=0.3118)
  [xgboost] B안 fold1: score=0.5648  (1-NMAE=0.8449, FICR=0.2847)
  [xgboost] B안 fold2: score=0.5778  (1-NMAE=0.8587, FICR=0.2970)
  [xgboost] B안 fold3: score=0.6073  (1-NMAE=0.8701, FICR=0.3445)
=== mlp ===
  [mlp] A안(2024): score=0.6019  (1-NMAE=0.8639, FICR=0.3400)
  [mlp] B안 fold1: score=0.5652  (1-NMAE=0.8469, FICR=0.2834)
  [mlp] B안 fold2: score=0.6005  (1-NMAE=0.8607, FICR=0.3402)
  [mlp] B안 fold3: score=0.6019  (1-NMAE=0.8689, FICR=0.3349)
=== catboost (느림) ===
  [catboost] A안(2024): score=0.5983  (1-NMAE=0.8672, FICR=0.3294)
  [catboost] B안 fold1: score=0.5688  (1-NMAE=0.8460, FICR=0.2916)
  [catboost] B안 fold2: score=0.5903  (1-NMAE=0.8634, FICR=0.3172)
  [catboost] B안 fold3: score=0.6149  (1-NMAE=0.8748, FICR=0.3549)


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
lgb_l1,0.5965,0.5697,0.5947,0.6154,0.5933,0.0229
catboost,0.5983,0.5688,0.5903,0.6149,0.5913,0.0231
mlp,0.6019,0.5652,0.6005,0.6019,0.5892,0.0208
lgb_huber_auto_mono,0.5897,0.5642,0.5886,0.6071,0.5866,0.0215
lgb_huber_auto,0.5954,0.5557,0.5951,0.6060,0.5856,0.0264
xgboost,0.5864,0.5648,0.5778,0.6073,0.5833,0.0218
lgb_huber_a0.9,0.4177,0.4275,0.4334,0.4094,0.4234,0.0125



캐시에 저장된 예측 개수: 84 (변형수 x 4fold x 3그룹)


**확인할 것**: `xgboost` / `catboost` / `mlp` 점수가 9-4절 표와 **똑같아야** 합니다(같은 함수·같은 seed). 다르면 인프라 이전 과정에 버그가 있는 것이니 멈추고 알려주세요.

### 10-2. 지금 우리 모델은 어디서 점수를 잃고 있나 — 오차 해부

`lgb_l1`(현재 최고 단일 모델급)의 예측을 산식 그대로 뜯어봅니다. 표에 나오는 항목의 뜻은 이렇습니다.

- **평균 오차율**: `1-NMAE`를 만드는 값. 낮을수록 좋음
- **편향**: `(예측-실제)/설비용량`의 평균. **양수면 과대예측, 음수면 과소예측**. 이 값이 0에서 멀면 레버 B(분위수 이동)가 통할 여지가 큼
- **≤6% 비중 / 6\~8% 비중 / >8% 비중**: 실제발전량으로 가중한 비율. 셋을 더하면 1
- **FICR 재현값**: `≤6% 비중 + 0.75 × (6~8% 비중)`. 산식의 FICR과 정확히 같은 값이 나와야 정상입니다(4원/4원=1.0, 3원/4원=0.75이므로)
- **8\~10% 비중**: 지금은 0원을 받고 있지만 **문턱 바로 바깥에 아깝게 걸쳐 있는** 표본. 여기가 두꺼우면 개선 여지가 큼
- **6\~8%를 전부 통과시키면**: 6~8% 구간 표본을 전부 6% 안으로 밀어넣었을 때 오를 FICR 상한치(= `0.25 × 6~8% 비중`). "레버 B·C로 최대 얼마나 벌 수 있나"의 천장

In [28]:
def eval_frame(variant):
    """채점 대상(실제 >= 설비용량 10%) 행만 모아 fold/그룹/풍속/예측/실제를 하나의 긴 표로."""
    parts = []
    for fold_name, info in FOLD_INFO.items():
        for g in GROUP_COLS:
            cap = CAPACITY_KWH[g]
            ws_col = f"{g}_ws_est_cv_{info['cv_suffix']}"
            d = pd.DataFrame({
                "fold": fold_name,
                "group": g,
                "actual": info["actual_df"][g].to_numpy(dtype=float),
                "pred": PRED_CACHE[(fold_name, variant, g)].to_numpy(dtype=float),
                "ws": train.loc[info["valid_idx"], ws_col].to_numpy(dtype=float),
            })
            d = d[d["actual"] >= cap * 0.10].copy()
            d["cap"] = cap
            d["er"] = (d["pred"] - d["actual"]).abs() / cap
            d["signed"] = (d["pred"] - d["actual"]) / cap
            parts.append(d)
    return pd.concat(parts, ignore_index=True)


def wmean(mask, weight):
    return float(np.average(np.asarray(mask, dtype=float), weights=weight))


def error_anatomy(ef, by="group"):
    rows = []
    for key, d in ef.groupby(by, sort=False):
        w = d["actual"].to_numpy()
        p6 = wmean(d["er"] <= 0.06, w)
        p68 = wmean((d["er"] > 0.06) & (d["er"] <= 0.08), w)
        p8 = wmean(d["er"] > 0.08, w)
        p810 = wmean((d["er"] > 0.08) & (d["er"] <= 0.10), w)
        rows.append({
            by: key, "표본수": len(d),
            "평균오차율": d["er"].mean(),
            "편향": d["signed"].mean(),
            "≤6% 비중": p6, "6~8% 비중": p68, ">8% 비중": p8,
            "FICR 재현값": p6 + 0.75 * p68,
            "8~10% 비중": p810,
            "6~8%를 전부 통과시키면": 0.25 * p68,
        })
    return pd.DataFrame(rows).set_index(by).round(4)


ef_lgb = eval_frame("lgb_l1")
print("=== 그룹별 오차 해부 (lgb_l1, 4 fold 전체 합산) ===")
display(error_anatomy(ef_lgb, "group"))
print("\n=== fold별 오차 해부 ===")
display(error_anatomy(ef_lgb, "fold"))

=== 그룹별 오차 해부 (lgb_l1, 4 fold 전체 합산) ===


,표본수,평균오차율,편향,≤6% 비중,6~8% 비중,>8% 비중,FICR 재현값,8~10% 비중,6~8%를 전부 통과시키면
group,,,,,,,,,
kpx_group_1,12608,0.1328,-0.0640,0.2497,0.0804,0.6699,0.3100,0.0802,0.0201
kpx_group_2,12553,0.1279,-0.0191,0.3401,0.0922,0.5677,0.4093,0.0818,0.0230
kpx_group_3,11331,0.1495,-0.0623,0.2061,0.0672,0.7267,0.2565,0.0636,0.0168



=== fold별 오차 해부 ===


,표본수,평균오차율,편향,≤6% 비중,6~8% 비중,>8% 비중,FICR 재현값,8~10% 비중,6~8%를 전부 통과시키면
fold,,,,,,,,,
A안(2024),14432,0.1337,-0.0506,0.2700,0.0817,0.6483,0.3313,0.0762,0.0204
B안 fold1,7628,0.1519,-0.0494,0.2440,0.0710,0.6850,0.2973,0.0671,0.0178
B안 fold2,7433,0.1350,-0.0440,0.2687,0.0803,0.6510,0.3289,0.0786,0.0201
B안 fold3,6999,0.1260,-0.0453,0.2942,0.0890,0.6168,0.3609,0.0816,0.0222


**읽는 법**
- **편향**이 뚜렷하게 음수(예: -0.02 이하)면 → 모델이 채점 구간에서 과소예측 중. 레버 B(τ를 0.5보다 크게)가 통할 가능성이 높습니다
- **6\~8% 비중**과 **8\~10% 비중**이 두꺼우면 → 문턱 근처에 표본이 몰려 있다는 뜻이라 작은 개선으로 FICR이 크게 움직입니다
- 세 그룹의 편향 방향이 서로 다르면 → **그룹별로 다른 τ**가 필요하다는 신호 (`CLAUDE.md`가 예고한 "그룹별 τ")

### 10-3. HANDOFF 미해결 질문 2건 정리 — regime별 잔차와 feature importance

성능 실험으로 넘어가기 전에, HANDOFF에 "04에서 확인하기로 했다"고 적어둔 두 가지를 여기서 매듭짓습니다.

- **미해결 질문 6번**: 12절 파워커브 피처가 고풍속 구간에서 위험하다(`np.maximum.accumulate`가 우연히 튄 값을 이후 전 구간에 전파). 실제로 고풍속에서 오차가 큰지 확인
- **미해결 질문 4번**: `{group}_high_wind_caution`(15m/s 이상, group_1은 표본 0.36%뿐)이 정말 쓸모 있는 피처인지 확인

In [29]:
# 10-3a. regime(풍속 구간)별 잔차 — 미해결 질문 6번
CUT_IN_, RATED_, HIGH_ = 3.0, 12.0, 15.0
bins = [-np.inf, CUT_IN_, RATED_, HIGH_, np.inf]
labels = ["calm(<3)", "ramp(3~12)", "rated(12~15)", "high(>=15)"]
ef_lgb["regime"] = pd.cut(ef_lgb["ws"], bins=bins, labels=labels, right=False)

rows = []
for g in GROUP_COLS:
    for r in labels:
        d = ef_lgb[(ef_lgb["group"] == g) & (ef_lgb["regime"] == r)]
        if len(d) == 0:
            rows.append({"group": g, "regime": r, "표본수": 0})
            continue
        w = d["actual"].to_numpy()
        rows.append({
            "group": g, "regime": r, "표본수": len(d),
            "비중%": round(100 * len(d) / (ef_lgb["group"] == g).sum(), 2),
            "평균오차율": round(d["er"].mean(), 4),
            "편향": round(d["signed"].mean(), 4),
            "≤6% 통과율": round(wmean(d["er"] <= 0.06, w), 4),
        })
print("=== 풍속 구간별 잔차 (채점 대상 행만, lgb_l1) ===")
display(pd.DataFrame(rows))

=== 풍속 구간별 잔차 (채점 대상 행만, lgb_l1) ===


,group,regime,표본수,비중%,평균오차율,편향,≤6% 통과율
0,kpx_group_1,calm(<3),9,0.07,0.1365,-0.1365,0.0000
1,kpx_group_1,ramp(3~12),11344,89.97,0.1347,-0.0657,0.2401
2,kpx_group_1,rated(12~15),1206,9.57,0.1131,-0.0526,0.3020
3,kpx_group_1,high(>=15),49,0.39,0.1778,0.0657,0.2355
4,kpx_group_2,calm(<3),54,0.43,0.1313,-0.1313,0.0837
5,kpx_group_2,ramp(3~12),10203,81.28,0.1339,-0.0345,0.2758
6,kpx_group_2,rated(12~15),1897,15.11,0.0995,0.0521,0.4882
7,kpx_group_2,high(>=15),399,3.18,0.1097,0.0519,0.5864
8,kpx_group_3,calm(<3),125,1.10,0.1452,-0.1452,0.0171
9,kpx_group_3,ramp(3~12),10246,90.42,0.1446,-0.0532,0.2180


**확인할 것**: `rated`/`high` 구간의 **평균오차율이 `ramp`보다 뚜렷하게 크고 편향이 한쪽으로 쏠려 있으면** → 12-4절에 기록한 파워커브 고풍속 리스크가 실재한다는 뜻이므로, 05_tuning에서 파워커브 평활화·상한 클리핑을 검토합니다. 반대로 `ramp` 구간 오차가 제일 크면(파워커브가 가장 가파른 구간이라 이쪽일 가능성도 높습니다) 리스크의 초점이 달라집니다. 어느 쪽이든 **숫자로 확인하고 넘어가는 것**이 핵심입니다.

In [30]:
# 10-3b. feature importance — 미해결 질문 4번 (A안 fold 기준, 그룹당 1번씩 = 3번 학습)
info = FOLD_INFO["A안(2024)"]
imp_cols, models_a = {}, {}
for g in GROUP_COLS:
    _, m = fit_predict_lgbm(g, info["cv_suffix"], info["train_mask"], info["valid_idx"])
    models_a[g] = m
    s = pd.Series(m.feature_importances_, index=m.feature_name_, dtype=float)
    imp_cols[g] = s / s.sum()          # 전체 gain 대비 비율로 정규화
imp = pd.DataFrame(imp_cols)

print("=== 그룹별 상위 15개 피처 (gain 비중) ===")
for g in GROUP_COLS:
    top = imp[g].sort_values(ascending=False).head(15)
    print(f"\n--- {g} (상위 15개가 전체 gain의 {top.sum()*100:.1f}%) ---")
    for name, v in top.items():
        print(f"   {v*100:6.2f}%  {name}")

# 우리가 만든 도메인 피처들이 실제로 몇 등인지
def watch_map(g, suffix):
    ws = f"{g}_ws_est_cv_{suffix}"
    return {
        "추정풍속 ws_est": ws,
        "파워커브 power_curve_est": f"{g}_power_curve_est_cv_{suffix}",
        "밀도보정 풍속": f"{ws}__corrected",
        "풍속^3": f"{ws}__cube",
        "고풍속주의(15m/s)": f"{ws}__high_wind_caution",
        "regime_calm": f"{ws}__regime_calm",
        "regime_ramp": f"{ws}__regime_ramp",
        "regime_rated": f"{ws}__regime_rated",
        "결빙위험": f"{ws}__icing_risk",
        "돌풍 gust_proxy": f"{g}_gust_proxy",
        "GFS-LDAPS 차이": f"{g}_gfs_ldaps_diff",
    }

rows = []
n_feat = len(imp)
for g in GROUP_COLS:
    rank = imp[g].rank(ascending=False, method="min")
    for label, col in watch_map(g, info["cv_suffix"]).items():
        if col not in imp.index:
            continue
        rows.append({"group": g, "피처": label,
                     "gain비중%": round(imp.loc[col, g] * 100, 3),
                     "순위": int(rank[col]), "전체피처수": n_feat})
print(f"\n=== 우리가 만든 도메인 피처들의 실제 순위 (전체 {n_feat}개 중) ===")
display(pd.DataFrame(rows).pivot(index="피처", columns="group", values=["gain비중%", "순위"]))

=== 그룹별 상위 15개 피처 (gain 비중) ===

--- kpx_group_1 (상위 15개가 전체 gain의 64.6%) ---
    33.98%  kpx_group_1_ws_est_cv_2024_01__corrected
    12.66%  kpx_group_1_ws_est_cv_2024_01
     3.94%  kpx_group_1_power_curve_est
     3.04%  gfs_g5_heightAboveGround_100_100u
     1.59%  gfs_g1_isobaricInhPa_850_u
     1.56%  gfs_g5_ws_850hPa
     1.22%  gfs_g2_heightAboveGround_10_10u
     1.14%  kpx_group_1_power_curve_est_cv_2024_01
     1.04%  ldaps_ws10_avg16
     0.99%  gfs_g2_planetaryBoundaryLayer_0_u
     0.87%  ldaps_g13_heightAboveGround_50_50MUmin
     0.80%  gfs_g4_isobaricInhPa_850_u
     0.66%  gfs_g2_isobaricInhPa_850_u
     0.61%  gfs_g2_heightAboveGround_100_100u
     0.55%  gfs_g2_heightAboveGround_80_u

--- kpx_group_2 (상위 15개가 전체 gain의 43.5%) ---
    14.71%  kpx_group_2_power_curve_est
    10.60%  kpx_group_2_ws_est_cv_2024_01__corrected
     7.89%  kpx_group_2_ws_est_cv_2024_01
     2.28%  gfs_g1_isobaricInhPa_850_u
     1.37%  gfs_g5_ws_850hPa
     1.20%  ldaps_g13_heightAboveGrou

gain비중%                                  순위                        
group                kpx_group_1 kpx_group_2 kpx_group_3 kpx_group_1 kpx_group_2 kpx_group_3
피처                                                                                          
GFS-LDAPS 차이               0.082       0.213       0.131       134.0        42.0       143.0
regime_calm                0.000       0.000       0.007       584.0       743.0       683.0
regime_ramp                0.000       0.000       0.007       584.0       743.0       686.0
regime_rated               0.000       0.000       0.000       584.0       743.0       700.0
결빙위험                       0.000       0.007         NaN       584.0       701.0         NaN
고풍속주의(15m/s)               0.000       0.000       0.016       584.0       743.0       609.0
돌풍 gust_proxy              0.037       0.162       0.231       314.0        92.0        40.0
밀도보정 풍속                   33.981      10.600       4.950         1.0         2.0         5.0
추정풍속 ws_est               12.661       7.892       3.414         2.0         3.0         6.0
파워커브 power_curve_est       1.139       0.665       8.389         8.0         9.0         2.0
풍속^3                       0.000       0.000       0.000       584.0       743.0       700.0

**확인할 것 — 미해결 질문 4번의 답**: `고풍속주의(15m/s)`의 순위가 전체 800여 개 중 하위권이고 gain 비중이 0.0x% 수준이면 → **모델이 거의 쓰지 않는 피처**이므로, 남겨도 해롭진 않지만 "유효했다"고 주장할 근거는 없습니다. 반대로 상위 100위 안에 들면 표본이 적어도 의미 있게 쓰이고 있다는 뜻입니다.

동시에 `파워커브 power_curve_est`가 예상대로 압도적 1~2위인지도 확인하세요. 그렇다면 03_features 12절의 판단(파워커브 변환이 최우선 피처)이 모델 관점에서도 검증된 것입니다.

한 가지 더 눈여겨볼 것: **상위 15개가 전체 gain의 몇 %인가**입니다. 이 비율이 매우 높으면(예: 80% 이상) 나머지 800여 개 피처가 사실상 잡음이라는 뜻이라, 다음 단계에서 **피처 축소**가 큰 레버가 될 수 있습니다(학습 속도도 크게 빨라집니다).

### 10-4. 레버 A — 표본 가중을 산식에 맞추기

산식 해부 ①·④번에서 나온 아이디어를 그대로 코드로 옮깁니다. 세 가지를 비교합니다.

| 변형 | 학습 가중치 | 의도 |
|---|---|---|
| `lgb_l1` | 전부 1 (기존) | 기준선 |
| `lgb_w_eval` | 채점 대상(실제≥용량10%) 1.0, 나머지 0.1 | 산식 ①: 채점 안 되는 구간에 힘 빼지 말자 |
| `lgb_w_actual` | 실제발전량/설비용량 (하한 0.1) | 산식 ④: FICR의 actual 가중을 그대로 모사 |

**왜 0으로 안 하고 0.1을 주는가?** 채점 대상이 아닌 행을 학습에서 완전히 빼버리면, 모델이 저풍속 구간을 아예 배우지 못해 엉뚱한 값(예: 바람이 약한데 큰 발전량)을 내뱉을 수 있습니다. 문제는 **우리가 예측 시점에 어떤 행이 채점 대상인지 모른다**는 것입니다 — 실제 발전량을 봐야 알 수 있으니까요. 예보가 틀려서 "저풍속인 줄 알았는데 실제로는 발전량이 컸던" 시간대는 채점 대상인데, 그런 행을 크게 틀리면 손해가 큽니다. 그래서 0이 아니라 **0.1로 약하게 남겨두는** 절충안을 씁니다.

⚠️ 24번 학습(2변형 × 4fold × 3그룹).

In [31]:
weight_results = []
for name, mode in [("lgb_w_eval", "eval_only"), ("lgb_w_actual", "actual")]:
    print(f"=== {name} ===")
    weight_results.append(run_variant(name, lgbm_fit_fn(objective="l1", weight_mode=mode)))

display(summarize(diag_results + cache_results + weight_results))

=== lgb_w_eval ===
  [lgb_w_eval] A안(2024): score=0.6079  (1-NMAE=0.8730, FICR=0.3428)
  [lgb_w_eval] B안 fold1: score=0.5789  (1-NMAE=0.8541, FICR=0.3036)
  [lgb_w_eval] B안 fold2: score=0.6062  (1-NMAE=0.8702, FICR=0.3422)
  [lgb_w_eval] B안 fold3: score=0.6272  (1-NMAE=0.8812, FICR=0.3733)
=== lgb_w_actual ===
  [lgb_w_actual] A안(2024): score=0.6175  (1-NMAE=0.8675, FICR=0.3674)
  [lgb_w_actual] B안 fold1: score=0.5861  (1-NMAE=0.8501, FICR=0.3222)
  [lgb_w_actual] B안 fold2: score=0.6058  (1-NMAE=0.8601, FICR=0.3516)
  [lgb_w_actual] B안 fold3: score=0.6347  (1-NMAE=0.8757, FICR=0.3937)


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
lgb_w_actual,0.6175,0.5861,0.6058,0.6347,0.6089,0.0244
lgb_w_eval,0.6079,0.5789,0.6062,0.6272,0.6041,0.0243
lgb_l1,0.5965,0.5697,0.5947,0.6154,0.5933,0.0229
catboost,0.5983,0.5688,0.5903,0.6149,0.5913,0.0231
mlp,0.6019,0.5652,0.6005,0.6019,0.5892,0.0208
lgb_huber_auto_mono,0.5897,0.5642,0.5886,0.6071,0.5866,0.0215
lgb_huber_auto,0.5954,0.5557,0.5951,0.6060,0.5856,0.0264
xgboost,0.5864,0.5648,0.5778,0.6073,0.5833,0.0218
lgb_huber_a0.9,0.4177,0.4275,0.4334,0.4094,0.4234,0.0125


**확인할 것**: `lgb_w_eval` / `lgb_w_actual`이 `lgb_l1`보다 **A안과 B안 3-fold 전부에서** 높아야 채택합니다. 특히 **FICR 항목이 오르는지**를 따로 보세요 — 이 레버는 원래 FICR(actual 가중 계단)을 겨냥한 것이라, 1-NMAE는 그대로거나 살짝 떨어지고 FICR만 오르는 형태로 나타날 수 있습니다. 그래도 총점이 오르면 성공입니다.

### 10-5. 레버 B — 분위수(τ) 스윕: 예측을 문턱 안으로 밀어넣기

**분위수 회귀(quantile regression)란**: 보통 회귀는 "평균" 또는 "중앙값"을 맞춥니다. 분위수 회귀는 "몇 번째 백분위수를 맞출지"를 τ로 지정합니다. τ=0.5면 중앙값(MAE 최적과 동일), **τ=0.6이면 "실제값이 이 예측보다 클 확률이 60%"인 지점**을 맞추므로 예측이 전체적으로 아래로 내려가고, τ=0.4면 위로 올라갑니다.

> 정확히는: 분위수 손실은 과소예측(실제>예측)에 τ, 과대예측에 (1-τ)의 벌점을 줍니다. τ가 크면 과소예측 벌점이 커지므로 **예측이 위로 올라갑니다.** (앞 문장을 정정합니다 — τ↑ = 예측↑입니다.)

**왜 이게 점수를 올릴 수 있나**: FICR은 계단 함수라, 예측 분포 전체를 몇 백 kWh만 옮겨도 문턱을 넘나드는 표본이 수백 개 바뀔 수 있습니다. 10-2절에서 편향이 음수(과소예측)로 나왔다면 τ를 0.5보다 크게, 양수면 작게 잡는 게 유리합니다. **이건 "그냥 해보자"가 아니라 10-2절 편향 수치라는 근거에서 나온 실험입니다.**

τ ∈ {0.40, 0.45, 0.50, 0.55, 0.60}을 훑습니다. τ=0.50은 `lgb_l1`과 이론상 같은 최적점이라 **내부 검산**도 됩니다(구현이 달라 미세하게는 다를 수 있습니다).

⚠️ 60번 학습(5τ × 4fold × 3그룹). 가장 무거운 셀입니다.

In [32]:
TAUS = [0.40, 0.45, 0.50, 0.55, 0.60]
tau_results = []
for tau in TAUS:
    name = f"lgb_q{int(tau*100)}"
    print(f"=== {name} (tau={tau}) ===")
    tau_results.append(run_variant(name, lgbm_fit_fn(objective="quantile", alpha=tau)))

tau_table = summarize(tau_results)
display(tau_table)

# 그룹별로 최적 tau가 다른지 확인 (CLAUDE.md가 예고한 "그룹별 tau")
print("\n=== 그룹별 오차 해부: tau별 편향과 FICR 재현값 ===")
rows = []
for tau in TAUS:
    name = f"lgb_q{int(tau*100)}"
    an = error_anatomy(eval_frame(name), "group")
    for g in GROUP_COLS:
        rows.append({"tau": tau, "group": g,
                     "편향": an.loc[g, "편향"], "FICR 재현값": an.loc[g, "FICR 재현값"],
                     "평균오차율": an.loc[g, "평균오차율"]})
tau_by_group = pd.DataFrame(rows).pivot(index="tau", columns="group",
                                        values=["편향", "FICR 재현값", "평균오차율"])
display(tau_by_group.round(4))

=== lgb_q40 (tau=0.4) ===
  [lgb_q40] A안(2024): score=0.5671  (1-NMAE=0.8545, FICR=0.2797)
  [lgb_q40] B안 fold1: score=0.5456  (1-NMAE=0.8354, FICR=0.2558)
  [lgb_q40] B안 fold2: score=0.5603  (1-NMAE=0.8522, FICR=0.2684)
  [lgb_q40] B안 fold3: score=0.5828  (1-NMAE=0.8607, FICR=0.3049)
=== lgb_q45 (tau=0.45) ===
  [lgb_q45] A안(2024): score=0.5851  (1-NMAE=0.8615, FICR=0.3086)
  [lgb_q45] B안 fold1: score=0.5614  (1-NMAE=0.8424, FICR=0.2804)
  [lgb_q45] B안 fold2: score=0.5828  (1-NMAE=0.8607, FICR=0.3049)
  [lgb_q45] B안 fold3: score=0.5961  (1-NMAE=0.8678, FICR=0.3243)
=== lgb_q50 (tau=0.5) ===
  [lgb_q50] A안(2024): score=0.5921  (1-NMAE=0.8651, FICR=0.3191)
  [lgb_q50] B안 fold1: score=0.5677  (1-NMAE=0.8473, FICR=0.2880)
  [lgb_q50] B안 fold2: score=0.5892  (1-NMAE=0.8634, FICR=0.3150)
  [lgb_q50] B안 fold3: score=0.6151  (1-NMAE=0.8743, FICR=0.3559)
=== lgb_q55 (tau=0.55) ===
  [lgb_q55] A안(2024): score=0.6092  (1-NMAE=0.8698, FICR=0.3486)
  [lgb_q55] B안 fold1: score=0.5720  (1-NMAE=0.849

fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
lgb_q60,0.6151,0.5813,0.6151,0.6289,0.6084,0.0245
lgb_q55,0.6092,0.5720,0.6079,0.6265,0.6022,0.0277
lgb_q50,0.5921,0.5677,0.5892,0.6151,0.5907,0.0237
lgb_q45,0.5851,0.5614,0.5828,0.5961,0.5801,0.0175
lgb_q40,0.5671,0.5456,0.5603,0.5828,0.5629,0.0187



=== 그룹별 오차 해부: tau별 편향과 FICR 재현값 ===


편향                            FICR 재현값                               평균오차율                        
group kpx_group_1 kpx_group_2 kpx_group_3 kpx_group_1 kpx_group_2 kpx_group_3 kpx_group_1 kpx_group_2 kpx_group_3
tau                                                                                                              
0.40      -0.1043     -0.0565     -0.0995      0.2441      0.3644      0.2240      0.1493      0.1359      0.1610
0.45      -0.0826     -0.0426     -0.0806      0.2840      0.3795      0.2518      0.1386      0.1327      0.1530
0.50      -0.0643     -0.0228     -0.0675      0.3078      0.3996      0.2498      0.1328      0.1290      0.1498
0.55      -0.0477     -0.0025     -0.0455      0.3354      0.4250      0.2681      0.1284      0.1264      0.1459
0.60      -0.0314      0.0138     -0.0296      0.3523      0.4308      0.2783      0.1264      0.1264      0.1441

**확인할 것 (세 가지)**

1. **`lgb_q50`이 `lgb_l1`과 비슷한가** — 이론상 같은 최적점이므로 크게 다르면 뭔가 이상한 것입니다
2. **총점이 가장 높은 τ가 0.5가 아닌가** — 그렇다면 레버 B가 통한다는 뜻. 단, **A안과 B안 3-fold에서 일관되게 같은 방향**이어야 합니다. fold마다 최적 τ가 제각각이면 그건 검증 데이터에 대한 과적합이지 진짜 개선이 아닙니다
3. **그룹별 표에서 세 그룹의 최적 τ가 서로 다른가** — 다르면 그룹별 τ를 05_tuning에서 정식으로 다룹니다. 여기서 그룹별로 최적값을 골라 총점을 만드는 건 **하지 않습니다**(검증 데이터를 보고 3개 파라미터를 고르는 셈이라 낙관 편향이 생깁니다). 지금은 "경향이 있는지"만 확인합니다

## 11. 앙상블 — 서로 다른 모델의 오차를 상쇄시키기

**앙상블이 왜 통하는가**: 모델 A와 B가 각각 평균 오차가 같더라도, **틀리는 방향이 서로 다르면** 둘을 평균 냈을 때 오차가 줄어듭니다. 특히 우리 산식은 계단 함수라, "둘 다 6~8%에 걸쳐 있지만 하나는 위로, 하나는 아래로 틀린" 경우 평균이 6% 안으로 들어오면서 **점수가 3원에서 4원으로 점프**합니다.

> 비유: 시계 두 개가 각각 5분 빠르고 5분 느리면, 두 시계의 평균은 정확합니다. 반대로 둘 다 5분 빠르면 평균도 5분 빠릅니다 — **오차의 방향이 달라야** 이득이 생깁니다.

그래서 앙상블 전에 **잔차 상관(두 모델이 같은 방향으로 틀리는 정도)** 부터 확인합니다. 상관이 0.99면 사실상 같은 모델이라 섞어도 소용없고, 0.8 이하면 다양성이 있어 이득이 기대됩니다.

이 절의 모든 실험은 **캐시된 예측을 섞기만 하므로 재학습이 전혀 없습니다** — 몇 초면 끝납니다.

In [33]:
# 11-1. 후보 모델들 사이의 잔차 상관 — 다양성이 있는가
ENSEMBLE_CANDIDATES = [v for v in ["lgb_l1", "xgboost", "catboost", "mlp"]
                       if (list(FOLD_INFO)[0], v, GROUP_COLS[0]) in PRED_CACHE]
print("앙상블 후보:", ENSEMBLE_CANDIDATES)

resid = {}
for v in ENSEMBLE_CANDIDATES:
    ef = eval_frame(v)
    resid[v] = ef["signed"].to_numpy()
resid_df = pd.DataFrame(resid)
print("\n=== 잔차(예측-실제, 용량정규화) 상관행렬 — 낮을수록 앙상블 이득 기대 ===")
display(resid_df.corr().round(3))
print("\n=== 각 모델의 잔차 표준편차 ===")
display(resid_df.std().round(4))

앙상블 후보: ['lgb_l1', 'xgboost', 'catboost', 'mlp']

=== 잔차(예측-실제, 용량정규화) 상관행렬 — 낮을수록 앙상블 이득 기대 ===


,lgb_l1,xgboost,catboost,mlp
lgb_l1,1.000,0.937,0.951,0.869
xgboost,0.937,1.000,0.926,0.839
catboost,0.951,0.926,1.000,0.894
mlp,0.869,0.839,0.894,1.000



=== 각 모델의 잔차 표준편차 ===


lgb_l1      0.1706
xgboost     0.1748
catboost    0.1699
mlp         0.1784
dtype: float64

In [34]:
# 11-2. 블렌드 점수 계산 함수 (재학습 없음 — 캐시된 예측을 가중평균)
def score_blend(weights, label=None, verbose=False):
    """weights: {변형이름: 가중치} (합이 1이 되도록 자동 정규화)"""
    total_w = sum(weights.values())
    w = {k: v / total_w for k, v in weights.items()}
    rows = []
    for fold_name, info in FOLD_INFO.items():
        preds = {}
        for g in GROUP_COLS:
            acc = None
            for v, wt in w.items():
                s = PRED_CACHE[(fold_name, v, g)] * wt
                acc = s if acc is None else acc + s
            preds[g] = acc.clip(lower=0, upper=CAPACITY_KWH[g])
        sc, nm, fi = score_predictions(info["actual_df"], pd.DataFrame(preds, index=info["valid_idx"]))
        rows.append({"fold": fold_name, "model": label or "+".join(w), "score": sc, "1-NMAE": nm, "FICR": fi})
    return pd.DataFrame(rows)


# (a) 두 모델 쌍별로 가중치를 훑어본다
pair_rows = []
for i, a in enumerate(ENSEMBLE_CANDIDATES):
    for b in ENSEMBLE_CANDIDATES[i + 1:]:
        for wa in np.arange(0.0, 1.01, 0.1):
            df = score_blend({a: wa, b: 1 - wa}, label=f"{a}:{b}")
            piv = df.set_index("fold")["score"]
            pair_rows.append({"pair": f"{a} + {b}", f"{a} 비중": round(wa, 1),
                              "A안": piv["A안(2024)"],
                              "B안 평균": piv[B_FOLDS].mean(),
                              "B안 최솟값": piv[B_FOLDS].min()})
pair_df = pd.DataFrame(pair_rows)
for pair, d in pair_df.groupby("pair", sort=False):
    print(f"\n=== {pair} ===")
    display(d.drop(columns="pair").set_index(d.columns[1]).round(4))


=== lgb_l1 + xgboost ===


,A안,B안 평균,B안 최솟값,xgboost 비중,catboost 비중
lgb_l1 비중,,,,,
0.0,0.5864,0.5833,0.5648,NaN,NaN
0.1,0.5883,0.5848,0.5655,NaN,NaN
0.2,0.5893,0.5860,0.5658,NaN,NaN
0.3,0.5916,0.5875,0.5662,NaN,NaN
0.4,0.5929,0.5889,0.5667,NaN,NaN
0.5,0.5935,0.5902,0.5685,NaN,NaN
0.6,0.5946,0.5908,0.5685,NaN,NaN
0.7,0.5956,0.5915,0.5688,NaN,NaN
0.8,0.5963,0.5927,0.5701,NaN,NaN



=== lgb_l1 + catboost ===


,A안,B안 평균,B안 최솟값,xgboost 비중,catboost 비중
lgb_l1 비중,,,,,
0.0,0.5983,0.5913,0.5688,NaN,NaN
0.1,0.5994,0.5929,0.5704,NaN,NaN
0.2,0.5998,0.5931,0.5711,NaN,NaN
0.3,0.6002,0.5940,0.5715,NaN,NaN
0.4,0.5998,0.5935,0.5711,NaN,NaN
0.5,0.5995,0.5934,0.5709,NaN,NaN
0.6,0.5993,0.5939,0.5717,NaN,NaN
0.7,0.5984,0.5938,0.5714,NaN,NaN
0.8,0.5981,0.5935,0.5701,NaN,NaN



=== lgb_l1 + mlp ===


...

,A안,B안 평균,B안 최솟값,xgboost 비중,catboost 비중
lgb_l1 비중,,,,,
NaN,0.5983,0.5913,0.5688,0.0,NaN
NaN,0.5982,0.5916,0.5689,0.1,NaN
NaN,0.5972,0.5912,0.5691,0.2,NaN
NaN,0.5964,0.5910,0.5696,0.3,NaN
NaN,0.5949,0.5909,0.5703,0.4,NaN
NaN,0.5943,0.5898,0.5691,0.5,NaN
NaN,0.5927,0.5891,0.5688,0.6,NaN
NaN,0.5911,0.5873,0.5680,0.7,NaN
NaN,0.5895,0.5860,0.5662,0.8,NaN



=== xgboost + mlp ===


,A안,B안 평균,B안 최솟값,xgboost 비중,catboost 비중
lgb_l1 비중,,,,,
NaN,0.6019,0.5892,0.5652,0.0,NaN
NaN,0.6052,0.5925,0.5669,0.1,NaN
NaN,0.6054,0.5941,0.5687,0.2,NaN
NaN,0.6055,0.5946,0.5691,0.3,NaN
NaN,0.6046,0.5951,0.5704,0.4,NaN
NaN,0.6023,0.5942,0.5696,0.5,NaN
NaN,0.5993,0.5930,0.5695,0.6,NaN
NaN,0.5961,0.5916,0.5705,0.7,NaN
NaN,0.5926,0.5888,0.5689,0.8,NaN



=== catboost + mlp ===


,A안,B안 평균,B안 최솟값,xgboost 비중,catboost 비중
lgb_l1 비중,,,,,
NaN,0.6019,0.5892,0.5652,NaN,0.0
NaN,0.6047,0.5918,0.5668,NaN,0.1
NaN,0.6052,0.5933,0.5684,NaN,0.2
NaN,0.6063,0.5944,0.5702,NaN,0.3
NaN,0.6060,0.5952,0.5703,NaN,0.4
NaN,0.6056,0.5956,0.5713,NaN,0.5
NaN,0.6047,0.5952,0.5708,NaN,0.6
NaN,0.6036,0.5953,0.5712,NaN,0.7
NaN,0.6028,0.5951,0.5715,NaN,0.8


In [35]:
# (b) 균등 가중 앙상블들 + 단일 모델 최고와 비교
blend_specs = {}
if len(ENSEMBLE_CANDIDATES) >= 2:
    blend_specs["균등(전체)"] = {v: 1 for v in ENSEMBLE_CANDIDATES}
if all(v in ENSEMBLE_CANDIDATES for v in ["lgb_l1", "mlp"]):
    blend_specs["GBDT3+MLP(GBDT 0.7)"] = {v: 0.7 / 3 for v in ENSEMBLE_CANDIDATES if v != "mlp"} | {"mlp": 0.3}
if all(v in ENSEMBLE_CANDIDATES for v in ["lgb_l1", "xgboost", "catboost"]):
    blend_specs["GBDT 3종 균등"] = {"lgb_l1": 1, "xgboost": 1, "catboost": 1}

blend_results = [score_blend(w, label=name) for name, w in blend_specs.items()]
single_results = [pd.concat(diag_results + cache_results, ignore_index=True)]

final_table = summarize(single_results + blend_results)
display(final_table)

fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
GBDT3+MLP(GBDT 0.7),0.6032,0.5730,0.6028,0.6127,0.5962,0.0206
균등(전체),0.6021,0.5736,0.6005,0.6135,0.5959,0.0203
lgb_l1,0.5965,0.5697,0.5947,0.6154,0.5933,0.0229
GBDT 3종 균등,0.5975,0.5693,0.5923,0.6148,0.5921,0.0228
catboost,0.5983,0.5688,0.5903,0.6149,0.5913,0.0231
mlp,0.6019,0.5652,0.6005,0.6019,0.5892,0.0208
lgb_huber_auto_mono,0.5897,0.5642,0.5886,0.6071,0.5866,0.0215
lgb_huber_auto,0.5954,0.5557,0.5951,0.6060,0.5856,0.0264
xgboost,0.5864,0.5648,0.5778,0.6073,0.5833,0.0218


**확인할 것**

1. **앙상블이 최고 단일 모델을 A안·B안 3-fold 모두에서 이기는가** — 하나라도 지면 그 조합은 "효과 불확실"입니다
2. **B안 최솟값**(가장 나쁜 fold 점수)도 같이 보세요. 평균만 오르고 최악 fold가 나빠지면 2025년 실전에서 위험합니다
3. **가중치는 0.5, 0.7 같은 둥근 값을 고르세요.** 스윕 표에서 최고점이 0.63에서 나왔다고 0.63을 쓰면 검증 데이터에 과적합됩니다. 곡선이 완만한 구간의 가운데를 고르는 게 안전합니다

**주의(중요)**: 이 절에서 가중치를 검증 점수를 보고 고르는 순간, 그 검증 점수는 더 이상 "순수한 미래 성능 추정치"가 아닙니다(약간 낙관적으로 부풀려집니다). 그래서 가중치는 **fold 전체에서 안정적인 둥근 값**으로 고르고, 최종 확정은 05_tuning에서 별도로 다룹니다.

## 12. 요약 — 04_model_selection에서 확정한 것

> ⚠️ 아래 숫자는 전부 **13절(누수 수정 후) 기준**입니다. 9~11절의 저장된 출력은 누수 상태이므로 인용하지 마세요.

### 최종 성적

| | A안(2024) | B fold1 | B fold2 | B fold3 | B평균 |
|---|---|---|---|---|---|
| `fix_base` (누수 없는 출발점) | 0.5915 | 0.5665 | 0.5898 | 0.6102 | 0.5888 |
| `dyn_w_q60` (최고 단일) | 0.6252 | 0.5926 | 0.6086 | 0.6361 | 0.6124 |
| **`dyn_w_q60` 0.7 + `dyn_mlp` 0.3** | **0.6288** | – | – | – | **0.6170** |
| **개선폭** | **+0.0373** | | | | **+0.0282** |

### 확정 사항 8가지

1. **모델: LightGBM.** CatBoost와 성능 동률(차이 0.002 < fold 표준편차 0.023)이나 훨씬 빠름. XGBoost는 일관되게 0.01 낮음
2. **단조 제약: 채택 안 함.** 9절의 폭락(0.42/0.37)은 제약이 아니라 **후버 손실 `alpha` 기본값 0.9를 kWh 타깃에 그대로 쓴 것**이 원인 — 9-5절에서 제약 없는 `lgb_huber_a0.9`가 0.4177로 소수점 넷째 자리까지 재현되어 확정. 공정 비교(XGBoost, LightGBM 둘 다)에서 개선 없음
3. **표본 가중: 채택** (`weight = actual/capacity`, 하한 0.1). 단독 +0.018. 산식이 ① 발전량 10% 미만을 채점 제외하고 ② FICR을 actual로 가중하는 것을 학습에 반영
4. **분위수 τ = 0.60: 채택.** 단독 +0.017. 0.65는 거의 동률, 0.70은 하락 — **꼭짓점 확인됨**
5. **가중 × τ 결합: 채택** (+0.021). 단 두 레버가 겹쳐 단순 합(+0.035)에 못 미침
6. **예보 lag/lead: 조건부 유지.** 4 fold 중 3개 개선(+0.0027), fold1만 -0.0016 → 규칙상 "효과 불확실". 다만 **잔차 σ를 줄인 유일한 변형**(0.1708→0.1679)이라 유지
7. **MLP 블렌드 0.3: 채택** (+0.005). 단 이득의 상당 부분이 편향 상쇄(LightGBM +0.054 과대 / MLP -0.042 과소)에서 왔을 수 있어, MLP에도 같은 가중·τ를 적용하면 사라질 수 있음
8. **구간 더미 피처(`regime_*`, `high_wind_caution`, `풍속^3`)는 무용.** gain 0.000%, 순위 584~743위. 트리 모델이 연속 변수를 알아서 구간으로 자르기 때문

### ⭐ 가장 중요한 결론 — 위치 최적화는 끝났고, 남은 건 폭(σ)이다

| 변형 | 잔차 σ | 편향 | 밴드폭 ÷ σ |
|---|---|---|---|
| `fix_base` | 0.1708 | -0.0503 | 0.351 |
| `fix_dyn` | 0.1679 | -0.0502 | 0.357 |
| `dyn_w_q60` | 0.1728 | **+0.0539** | 0.347 |

- 이번에 얻은 **+0.028은 전부 "예측을 FICR 밴드의 좋은 위치로 옮긴 것"** 에서 나왔고, **오차의 폭은 하나도 못 줄였습니다**
- 편향이 -0.050 → **+0.054로 반대편까지 넘어갔습니다.** τ=0.70이 오히려 하락한 것이 그 증거 — **더 밀 여지가 없습니다**
- FICR 밴드(±6% 용량)는 σ의 **0.35배**뿐입니다. σ를 절반으로 줄이면 통과율이 27%→52%로 뛰어 총점 +0.1 규모가 됩니다. **남은 상방은 전부 여기 있습니다**

### 그룹별 편향 — τ를 그룹별로 나눠야 함 (`dyn_w_q60` 기준)

| 그룹 | 편향 | 해석 |
|---|---|---|
| kpx_group_1 | +0.036 | 적정 |
| kpx_group_2 | **+0.076** | **과하게 밀림 → τ를 낮춰야** |
| kpx_group_3 | +0.050 | 약간 과함 |

group_2는 원래 편향이 가장 작았던(-0.021) 그룹이라 같은 τ를 적용하니 과대로 넘어갔습니다.

---

## 다음 단계 (05_tuning으로 넘길 것)

**우선순위 1 — σ를 줄이는 길이 남아있는지부터 확인 (오라클 실험)**
`scada_ws_{group}`(SCADA 실측 풍속)을 풍속 피처 자리에 넣고 같은 모델을 돌려 **"바람을 완벽히 알았다면 몇 점인가"** 를 잰다. 이 값이 현재(0.62)와 크게 차이 나면 풍속 추정 개선에 투자할 가치가 있다는 뜻이고, 비슷하면 예보 오차가 아니라 **발전량 자체의 불확실성**이 한계이므로 다른 길을 찾아야 한다. **12번 학습이면 끝나는데 앞으로의 방향 전체를 결정한다.**
(주의: 이건 진단 전용이다 — SCADA는 test에 없으므로 실제 제출에는 절대 쓸 수 없다)

**우선순위 2 — 그룹별 τ** (근거: 위 그룹별 편향 표)

**우선순위 3 — MLP 제대로 학습** (현재 full-batch 100 에폭의 미학습 상태) + 같은 가중·τ 적용 후 블렌드 재확인

**우선순위 4 — 오라클 결과에 따라 분기**
- 오라클이 높으면 → 공간 CNN(격자를 이미지로) / 시퀀스 모델로 풍속 추정 개선
- 오라클도 낮으면 → 분포 예측 + 기대 FICR 최대화(B1)로 남은 여지를 짜내는 쪽

세부 아이디어 목록과 근거는 `HANDOFF.md`의 "성능 개선 아이디어 목록" 참조.

---

# 13. 누수 수정 + 예보 lag/lead 피처 + 검증된 레버 결합

이 절은 세 가지를 한 번에 처리합니다.

1. **누수 수정** — 6절을 이미 고쳤으므로, 그 위에서 다시 돌려 **누수가 실제로 얼마나 점수를 부풀렸는지** 측정합니다
2. **예보 lag/lead 피처 추가** — 지금까지 없던 시간 방향 정보를 넣습니다 (아래 13-1에서 이유 설명)
3. **검증된 레버 결합** — 표본 가중(+0.016)과 분위수 τ(+0.015)를 **처음으로 같이** 써봅니다

### ▶ 실행 순서 (중요 — 이대로만 하세요)

커널을 새로 시작한 뒤:

| 순서 | 실행할 것 | 소요 |
|---|---|---|
| 1 | **1~6절 코드 셀 전부** (셋업·로드·피처 분류·fold 정의·fold-safe 재계산·프레임 빌더) | 1분 이내 |
| 2 | **9-3절 코드 셀 1개** (`import torch` + `SimpleMLP`/`fit_predict_mlp` **정의만** 있는 셀) | 즉시 |
| 3 | **9-5절 첫 코드 셀 1개** (`FOLD_INFO`/`PRED_CACHE`/`run_variant`/`summarize` 인프라 **정의만** 있는 셀) | 즉시 |
| 4 | **13절을 13-1부터 순서대로** | 아래 각 셀에 표시 |

**7·8·9-4·10·11절의 무거운 셀은 건너뜁니다.** 이미 실행했고, 어차피 누수 상태라 다시 볼 필요가 없습니다.

> 6절을 실행할 때 `전체 fold x 그룹 조합에서 NaN 없음 + 규칙 기반 누수 점검 통과`가 뜨고, 그 아래 그룹별 피처 수가 출력되면 정상입니다. 여기서 `AssertionError`가 나면 멈추고 알려주세요.

## 13-1. 예보 lag/lead 피처 — 지금까지 없던 "시간 방향" 정보

### 왜 이게 필요한가

`train_features_v1.parquet`의 888개 컬럼을 전부 확인했는데, **lag/lead/rolling 계열 피처가 하나도 없습니다.** 지금 모델은 **t시각의 예보만 보고 t시각의 발전량을 예측**하고 있습니다.

이게 왜 손해냐면, 수치예보에서 가장 흔한 오차가 **위상 오차(phase error)** 이기 때문입니다.

> 예: 실제로는 15시에 전선이 통과하면서 풍속이 5→13 m/s로 급증했는데, 예보는 이 급증을 **정확히 맞혔지만 시각을 18시로 잡았다**. 그러면 15~17시는 "예보 5, 실제 13"으로 크게 틀리고, 18~20시는 "예보 13, 실제 5"로 또 크게 틀립니다. **예보의 모양은 맞았는데 시각만 3시간 어긋나서 6시간이 전부 망가지는 것**입니다.

t시각 예보 하나만 보면 모델은 이걸 알아챌 방법이 없습니다. 하지만 **t-3 ~ t+3의 예보를 같이 주면** "지금 예보는 낮지만 2시간 뒤에 급증 예보가 있다 = 지금은 램프 구간이고 위상이 어긋났을 수 있다"는 판단이 가능해집니다.

### 이게 데이터 누수가 아닌 이유 (`leakage-guard` 점검)

"미래 시각(t+3)의 값을 쓴다"고 하면 반사적으로 누수를 의심하게 되는데, **여기서는 아닙니다.**

- 누수의 정의는 "**그 시점에 실제로 활용 가능하지 않았던** 정보를 쓰는 것"입니다 (CLAUDE.md 5장 예측기준시점 원칙)
- 수치예보는 발표 시점에 **미래 시각까지 통째로** 나옵니다. 오늘 아침에 받은 GFS 예보에는 이미 내일 오후까지의 예보값이 들어 있습니다
- 즉 **t+3 시각의 예보값은 t시점에 이미 손에 있습니다.** 실전에서도 똑같이 쓸 수 있습니다
- ❌ 금지되는 것은 **실측**(SCADA·발전량 라벨)의 lag/lead입니다. 그건 절대 안 됩니다
- 여기서 파생시키는 대상은 `ws_est_cv_*`, `power_curve_est_cv_*` 두 개인데, 둘 다 **예보로부터 계산되는 fold-safe 컬럼**입니다. 시간축으로 밀어도 fold-safe성이 유지됩니다(같은 컬럼을 옆으로 옮긴 것뿐이므로)

### 무엇을 만드는가 (기준 컬럼 1개당 13개)

| 종류 | 컬럼 | 의미 |
|---|---|---|
| lag 3개 | `__lag1/2/3` | 1·2·3시간 **전** 예보 |
| lead 3개 | `__lead1/2/3` | 1·2·3시간 **후** 예보 |
| 변화율 2개 | `__ramp_prev`, `__ramp_next` | 직전/직후 대비 변화량 (램프 감지) |
| 이동평균 2개 | `__roll3_mean`, `__roll7_mean` | ±1시간, ±3시간 평균 (예보를 부드럽게 = 위상 오차에 둔감해짐) |
| 변동성 3개 | `__roll7_std/max/min` | ±3시간 범위의 흔들림 (= 이 시각 예보의 불확실성) |

기준 컬럼 2개(`ws_est_cv_*`, `power_curve_est_cv_*`) × 13 = **그룹당 26개 추가**입니다.

`__roll7_std`는 갈래 A5(예보 불확실성 피처) 역할도 겸합니다 — "이 시각은 예보가 흔들리고 있으니 못 믿겠다"를 모델이 알 수 있게 해줍니다.

### 왜 `03_features.ipynb`가 아니라 여기서 만드는가

parquet을 다시 만들지 않기로 했습니다. 이유는 세 가지입니다.
1. parquet 재생성 없이 **즉시** 실험할 수 있습니다
2. 기준 컬럼이 **fold마다 다릅니다**(`_cv_2023_07`/`_cv_2024_01`/`_cv_2024_07`). parquet에 미리 만들면 컬럼이 3배로 폭증합니다
3. `build_group_feature_frame`이 이미 `df`를 인자로 받으므로, **test에도 같은 코드가 그대로 적용**됩니다

효과가 확인되면 그때 `03_features.ipynb`로 옮겨 정식화할지 결정하겠습니다.

In [43]:
# 이전 절(누수 상태)의 예측이 섞이지 않도록 캐시를 비우고 시작
PRED_CACHE.clear()
print("캐시 초기화 완료")

DYN_SHIFTS = [1, 2, 3]


def add_forecast_dynamics(df, cols):
    """예보 계열 컬럼에 시간 방향 파생(lag/lead/변화율/이동통계)을 붙인다.

    누수 아님: 수치예보는 발표 시점에 미래 시각까지 통째로 제공되므로 lead도 합법.
    (실측 SCADA/라벨의 lag/lead였다면 절대 금지)
    """
    dt = df["kst_dtm"]
    assert dt.is_monotonic_increasing, "kst_dtm이 시간순이 아님 — shift가 엉뚱한 값을 가져옴"
    gaps = dt.diff().dropna().unique()
    assert len(gaps) == 1 and gaps[0] == pd.Timedelta("1h"), f"1시간 간격 연속이 아님: {gaps[:5]}"

    out = {}
    for c in cols:
        s = df[c]
        for k in DYN_SHIFTS:
            out[f"{c}__lag{k}"] = s.shift(k)      # k시간 전 예보
            out[f"{c}__lead{k}"] = s.shift(-k)    # k시간 후 예보
        out[f"{c}__ramp_prev"] = s - s.shift(1)   # 직전 대비 변화량
        out[f"{c}__ramp_next"] = s.shift(-1) - s  # 직후 변화량
        roll3 = s.rolling(3, center=True, min_periods=1)
        roll7 = s.rolling(7, center=True, min_periods=1)
        out[f"{c}__roll3_mean"] = roll3.mean()
        out[f"{c}__roll7_mean"] = roll7.mean()
        out[f"{c}__roll7_std"] = roll7.std()      # 예보 불확실성 대리지표
        out[f"{c}__roll7_max"] = roll7.max()
        out[f"{c}__roll7_min"] = roll7.min()

    res = pd.DataFrame(out, index=df.index)
    res = res.bfill().ffill()   # 양 끝 몇 행에만 생기는 결측 채우기
    assert res.isna().sum().sum() == 0, "동적 피처에 결측 잔존"
    return res


def build_group_feature_frame_v2(df, g, cv_suffix, dynamics=True):
    """6절의 (수정된) 빌더 + 예보 시간 방향 피처."""
    base = build_group_feature_frame(df, g, cv_suffix)   # 누수 점검은 여기서 이미 수행됨
    if not dynamics:
        return base
    ws_col = f"{g}_ws_est_cv_{cv_suffix}"
    pc_col = f"{g}_power_curve_est_cv_{cv_suffix}"
    out = pd.concat([base, add_forecast_dynamics(df, [ws_col, pc_col])], axis=1)
    assert_no_leak(out.columns, g, cv_suffix)   # 파생까지 포함해 다시 점검
    return out


# 확인: 피처 수가 26개 늘었는지, 만들어진 이름이 의도대로인지
for g in GROUP_COLS:
    n0 = build_group_feature_frame_v2(train, g, "2024_01", dynamics=False).shape[1]
    f1 = build_group_feature_frame_v2(train, g, "2024_01", dynamics=True)
    print(f"{g}: {n0} -> {f1.shape[1]} (+{f1.shape[1]-n0})")
print("\n새로 만들어진 컬럼 예시:")
for c in [c for c in f1.columns if "__lag" in c or "__roll" in c or "__ramp" in c][:8]:
    print("   ", c)

캐시 초기화 완료
kpx_group_1: 822 -> 848 (+26)
kpx_group_2: 822 -> 848 (+26)
kpx_group_3: 821 -> 847 (+26)

새로 만들어진 컬럼 예시:
    kpx_group_3_ws_est_cv_2024_01__lag1
    kpx_group_3_ws_est_cv_2024_01__lag2
    kpx_group_3_ws_est_cv_2024_01__lag3
    kpx_group_3_ws_est_cv_2024_01__ramp_prev
    kpx_group_3_ws_est_cv_2024_01__ramp_next
    kpx_group_3_ws_est_cv_2024_01__roll3_mean
    kpx_group_3_ws_est_cv_2024_01__roll7_mean
    kpx_group_3_ws_est_cv_2024_01__roll7_std


**확인할 것**: 그룹마다 피처 수가 **정확히 +26** 이어야 합니다. 그리고 `AssertionError: ... 1시간 간격 연속이 아님`이 뜨면 시간축에 구멍이 있다는 뜻이니 멈추고 알려주세요(`02_eda` 시간 무결성 점검에서 문제없다고 확인했으므로 뜨지 않아야 정상입니다).

## 13-2. 학습 함수 (누수 수정판 + 동적 피처 지원)

9-5절의 `fit_predict_lgbm`과 9-3절의 `fit_predict_mlp`을 **수정된 프레임 빌더를 쓰도록** 다시 정의합니다. 함수를 새로 쓰는 대신 이름만 바꿔 복사한 이유는, 어떤 셀이 어떤 빌더를 쓰는지 헷갈리지 않게 하기 위해서입니다(HANDOFF에 기록된 "모호하면 다음 세션에서 엉뚱하게 처리된다"는 교훈).

단조 제약은 9-5절에서 "효과 불확실"로 판정 났으므로 아예 뺐습니다.

In [44]:
def fit_predict_lgbm2(g, cv_suffix, train_mask, valid_idx, *, objective="l1", alpha=None,
                      weight_mode=None, dynamics=True, n_estimators=2000):
    """LightGBM 한 번 학습 후 검증 구간 예측. (예측Series, 모델객체) 반환."""
    X_full = build_group_feature_frame_v2(train, g, cv_suffix, dynamics=dynamics)
    fit_idx = train_mask & train[g].notna()
    cutoff_es = train.loc[fit_idx, "kst_dtm"].quantile(0.9)   # 학습구간 뒤 10%를 early stopping용으로
    tr_mask = fit_idx & (train["kst_dtm"] <= cutoff_es)
    es_mask = fit_idx & (train["kst_dtm"] > cutoff_es)
    Xtr, ytr = X_full.loc[tr_mask], train.loc[tr_mask, g]
    Xes, yes = X_full.loc[es_mask], train.loc[es_mask, g]

    params = dict(objective=objective, random_state=SEED, n_estimators=n_estimators,
                  verbosity=-1, importance_type="gain")
    if alpha is not None:
        params["alpha"] = alpha

    w_tr = make_sample_weight(ytr, g, weight_mode)
    w_es = make_sample_weight(yes, g, weight_mode)

    model = lgb.LGBMRegressor(**params)
    model.fit(Xtr, ytr, sample_weight=w_tr,
              eval_set=[(Xes, yes)],
              eval_sample_weight=[w_es] if w_es is not None else None,
              eval_metric="l1", callbacks=[lgb.early_stopping(50, verbose=False)])

    pred = pd.Series(model.predict(X_full.loc[valid_idx]), index=valid_idx)
    return pred.clip(lower=0, upper=CAPACITY_KWH[g]), model


def lgbm2_fit_fn(**kwargs):
    def fit_fn(g, cv_suffix, train_mask, valid_idx):
        return fit_predict_lgbm2(g, cv_suffix, train_mask, valid_idx, **kwargs)[0]
    return fit_fn


def fit_predict_mlp2(g, cv_suffix, train_mask, valid_idx, dynamics=True, max_epochs=200, patience=20):
    """9-3절 MLP와 동일하되 수정된 프레임 빌더 사용."""
    capacity = CAPACITY_KWH[g]
    X_full = build_group_feature_frame_v2(train, g, cv_suffix, dynamics=dynamics)
    fit_idx = train_mask & train[g].notna()
    cutoff_es = train.loc[fit_idx, "kst_dtm"].quantile(0.9)
    tr_mask = fit_idx & (train["kst_dtm"] <= cutoff_es)
    es_mask = fit_idx & (train["kst_dtm"] > cutoff_es)

    mean = X_full.loc[tr_mask].mean()
    std = X_full.loc[tr_mask].std().replace(0, 1)

    def to_tensor(d):
        return torch.tensor(((d - mean) / std).to_numpy(), dtype=torch.float32)

    Xtr_t = to_tensor(X_full.loc[tr_mask])
    ytr_t = torch.tensor(train.loc[tr_mask, g].to_numpy() / capacity, dtype=torch.float32)
    Xes_t = to_tensor(X_full.loc[es_mask])
    yes_t = torch.tensor(train.loc[es_mask, g].to_numpy() / capacity, dtype=torch.float32)
    Xva_t = to_tensor(X_full.loc[valid_idx])

    torch.manual_seed(SEED)
    model = SimpleMLP(Xtr_t.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.L1Loss()

    best_es, best_state, patience_left = float("inf"), None, patience
    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        loss_fn(model(Xtr_t), ytr_t).backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            es_loss = loss_fn(model(Xes_t), yes_t).item()
        if es_loss < best_es - 1e-6:
            best_es, best_state, patience_left = es_loss, {k: v.clone() for k, v in model.state_dict().items()}, patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred_va = model(Xva_t).numpy() * capacity
    return pd.Series(pred_va, index=valid_idx).clip(lower=0, upper=capacity)


print("학습 함수 준비 완료")

학습 함수 준비 완료


## 13-3. 1단계 — 누수의 크기와 lag/lead의 효과를 **분리해서** 측정

두 변화를 한꺼번에 넣으면 점수가 움직여도 어느 쪽 때문인지 알 수 없습니다. 그래서 하나씩 켭니다.

| 변형 | 누수 수정 | lag/lead | 이 변형으로 알 수 있는 것 |
|---|---|---|---|
| `fix_base` | ✓ | ✗ | **누수가 점수를 얼마나 부풀렸나** (기존 `lgb_l1` 0.5965/0.5933과 비교) |
| `fix_dyn` | ✓ | ✓ | **lag/lead가 실제로 도움이 되나** (`fix_base`와 비교) |

⚠️ 24번 학습 (2변형 × 4fold × 3그룹).

In [45]:
stage1 = []
print("=== fix_base (누수 수정만) ===")
stage1.append(run_variant("fix_base", lgbm2_fit_fn(objective="l1", dynamics=False)))
print("=== fix_dyn (누수 수정 + lag/lead) ===")
stage1.append(run_variant("fix_dyn", lgbm2_fit_fn(objective="l1", dynamics=True)))

display(summarize(stage1))

print("\n=== 참고: 누수 상태였던 기존 숫자 ===")
print("  lgb_l1(누수):  A안 0.5965 / B평균 0.5933")

=== fix_base (누수 수정만) ===
  [fix_base] A안(2024): score=0.5915  (1-NMAE=0.8649, FICR=0.3180)
  [fix_base] B안 fold1: score=0.5665  (1-NMAE=0.8466, FICR=0.2865)
  [fix_base] B안 fold2: score=0.5898  (1-NMAE=0.8633, FICR=0.3163)
  [fix_base] B안 fold3: score=0.6102  (1-NMAE=0.8736, FICR=0.3468)
=== fix_dyn (누수 수정 + lag/lead) ===
  [fix_dyn] A안(2024): score=0.5991  (1-NMAE=0.8685, FICR=0.3296)
  [fix_dyn] B안 fold1: score=0.5649  (1-NMAE=0.8473, FICR=0.2826)
  [fix_dyn] B안 fold2: score=0.5981  (1-NMAE=0.8668, FICR=0.3293)
  [fix_dyn] B안 fold3: score=0.6116  (1-NMAE=0.8741, FICR=0.3492)


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
fix_dyn,0.5991,0.5649,0.5981,0.6116,0.5915,0.0240
fix_base,0.5915,0.5665,0.5898,0.6102,0.5888,0.0218



=== 참고: 누수 상태였던 기존 숫자 ===
  lgb_l1(누수):  A안 0.5965 / B평균 0.5933


**확인할 것 — 두 가지 질문**

1. **누수는 얼마나 컸나**: `fix_base` A안이 0.5965보다 얼마나 낮은가? 0.01 이내면 누수 영향이 작았던 것이고, 0.03 이상 떨어지면 컸던 것입니다. **어느 쪽이든 이제부터는 `fix_base`가 진짜 기준선**입니다
2. **lag/lead는 통하는가**: `fix_dyn`이 `fix_base`를 **A안·B안 3-fold 전부에서** 이겨야 채택합니다. 한두 fold만 오르면 "효과 불확실"입니다

만약 `fix_dyn`이 크게 이긴다면, 그건 **위상 오차가 실제로 큰 오차원이었다**는 증거이고, 갈래 A2/A3(공간 CNN·시퀀스 모델)로 확장할 강력한 근거가 됩니다. 반대로 효과가 없다면 시간 방향은 이미 짜낼 게 없다는 뜻이므로 다른 레버로 옮겨야 합니다.

## 13-4. 2단계 — 검증된 레버 결합 (표본 가중 × 분위수 τ)

10절에서 각각 확인된 두 레버를 **처음으로 같이** 씁니다.

| 변형 | 표본 가중 | τ | 목적 |
|---|---|---|---|
| `dyn_w` | actual | – | 가중만 (13-3의 `fix_dyn` 대비 순수 가중 효과) |
| `dyn_q60` | – | 0.60 | τ만 |
| `dyn_w_q60` | actual | 0.60 | **결합** |
| `dyn_w_q65` | actual | 0.65 | τ 최적점 탐색 |
| `dyn_w_q70` | actual | 0.70 | τ 최적점 탐색 |

**왜 τ를 0.65·0.70까지 보는가**: 10-5절에서 τ가 0.40→0.60까지 **단조 증가만 하고 꼭짓점이 안 나왔습니다.** 최고점이 범위 끝에 있으면 그건 "0.60이 최적"이 아니라 "아직 최적을 못 봤다"는 뜻입니다.

**주의할 점**: 표본 가중과 τ는 **둘 다 예측을 위로 올리는 효과**가 있습니다(가중은 발전량 큰 시간대에 집중시키고, τ>0.5는 과소예측에 더 큰 벌점을 줍니다). 그래서 결합 이득이 단순 합(+0.016 +0.015 = +0.031)보다 **작을 가능성이 높습니다.** 오히려 같이 쓰면 과대예측으로 넘어가 손해일 수도 있습니다 — 그래서 실제로 돌려봐야 합니다.

⚠️ 60번 학습 (5변형 × 4fold × 3그룹). 가장 무거운 셀입니다.

In [46]:
stage2_specs = {
    "dyn_w":     dict(objective="l1",       weight_mode="actual"),
    "dyn_q60":   dict(objective="quantile", alpha=0.60),
    "dyn_w_q60": dict(objective="quantile", alpha=0.60, weight_mode="actual"),
    "dyn_w_q65": dict(objective="quantile", alpha=0.65, weight_mode="actual"),
    "dyn_w_q70": dict(objective="quantile", alpha=0.70, weight_mode="actual"),
}

stage2 = []
for name, kw in stage2_specs.items():
    print(f"=== {name} ===")
    stage2.append(run_variant(name, lgbm2_fit_fn(dynamics=True, **kw)))

display(summarize(stage1 + stage2))

=== dyn_w ===
  [dyn_w] A안(2024): score=0.6171  (1-NMAE=0.8675, FICR=0.3667)
  [dyn_w] B안 fold1: score=0.5869  (1-NMAE=0.8506, FICR=0.3233)
  [dyn_w] B안 fold2: score=0.6089  (1-NMAE=0.8614, FICR=0.3563)
  [dyn_w] B안 fold3: score=0.6337  (1-NMAE=0.8763, FICR=0.3910)
=== dyn_q60 ===
  [dyn_q60] A안(2024): score=0.6174  (1-NMAE=0.8737, FICR=0.3611)
  [dyn_q60] B안 fold1: score=0.5789  (1-NMAE=0.8515, FICR=0.3062)
  [dyn_q60] B안 fold2: score=0.6137  (1-NMAE=0.8708, FICR=0.3566)
  [dyn_q60] B안 fold3: score=0.6326  (1-NMAE=0.8790, FICR=0.3862)
=== dyn_w_q60 ===
  [dyn_w_q60] A안(2024): score=0.6252  (1-NMAE=0.8620, FICR=0.3884)
  [dyn_w_q60] B안 fold1: score=0.5926  (1-NMAE=0.8483, FICR=0.3368)
  [dyn_w_q60] B안 fold2: score=0.6086  (1-NMAE=0.8530, FICR=0.3642)
  [dyn_w_q60] B안 fold3: score=0.6361  (1-NMAE=0.8702, FICR=0.4020)
=== dyn_w_q65 ===
  [dyn_w_q65] A안(2024): score=0.6214  (1-NMAE=0.8550, FICR=0.3878)
  [dyn_w_q65] B안 fold1: score=0.5927  (1-NMAE=0.8440, FICR=0.3414)
  [dyn_w_q65] B안 fol

fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
dyn_w_q60,0.6252,0.5926,0.6086,0.6361,0.6124,0.0220
dyn_w_q65,0.6214,0.5927,0.6057,0.6382,0.6122,0.0234
dyn_w,0.6171,0.5869,0.6089,0.6337,0.6098,0.0234
dyn_q60,0.6174,0.5789,0.6137,0.6326,0.6084,0.0273
dyn_w_q70,0.6138,0.5931,0.5927,0.6331,0.6063,0.0232
fix_dyn,0.5991,0.5649,0.5981,0.6116,0.5915,0.0240
fix_base,0.5915,0.5665,0.5898,0.6102,0.5888,0.0218


**확인할 것**

1. `dyn_w_q60/65/70` 중 **최고점이 또 범위 끝(0.70)에 있는가** — 그렇다면 0.75·0.80까지 더 봐야 합니다
2. **결합이 각각보다 나은가** — `dyn_w_q60`이 `dyn_w`와 `dyn_q60`을 둘 다 이기는지. 못 이기면 두 레버가 같은 일을 하고 있다는 뜻이므로 **더 단순한 쪽 하나만** 씁니다(오컴의 면도날 — 같은 성능이면 단순한 게 실전에서 안전합니다)
3. 늘 그렇듯 **A안과 B안 3-fold 전부에서** 이겨야 채택입니다

## 13-5. 3단계 — MLP 추가 + 블렌드

MLP는 GBDT와 **틀리는 방향이 가장 다른**(잔차 상관 0.84~0.89, GBDT끼리는 0.93~0.95) 모델이라 앙상블 가치가 있었습니다. 누수 수정 + lag/lead 조건에서 다시 확인하고, 13-4의 최고 LightGBM과 섞어봅니다.

블렌드 가중치는 **재학습 없이** 캐시된 예측을 섞어서 구합니다(몇 초).

⚠️ 12번 학습(MLP만) + 블렌드는 즉시.

In [47]:
print("=== dyn_mlp ===")
stage3 = [run_variant("dyn_mlp", lambda g, c, t, v: fit_predict_mlp2(g, c, t, v, dynamics=True))]

# 13-4에서 가장 좋았던 LightGBM 변형을 자동 선택 (B안 평균 기준)
all_lgb = summarize(stage1 + stage2)
best_lgb = all_lgb.index[0]
print(f"\n최고 LightGBM 변형: {best_lgb} (B안 평균 {all_lgb.loc[best_lgb, 'B안 평균']:.4f})")

# 블렌드 가중치 스윕
rows = []
for w in np.arange(0.0, 1.01, 0.1):
    df = score_blend({best_lgb: w, "dyn_mlp": 1 - w}, label="blend")
    piv = df.set_index("fold")["score"]
    rows.append({f"{best_lgb} 비중": round(w, 1), "A안": piv["A안(2024)"],
                 "B안 평균": piv[B_FOLDS].mean(), "B안 최솟값": piv[B_FOLDS].min()})
print(f"\n=== {best_lgb} + dyn_mlp 블렌드 ===")
display(pd.DataFrame(rows).set_index(f"{best_lgb} 비중").round(4))

=== dyn_mlp ===
  [dyn_mlp] A안(2024): score=0.5997  (1-NMAE=0.8650, FICR=0.3345)
  [dyn_mlp] B안 fold1: score=0.5567  (1-NMAE=0.8412, FICR=0.2723)
  [dyn_mlp] B안 fold2: score=0.5992  (1-NMAE=0.8621, FICR=0.3363)
  [dyn_mlp] B안 fold3: score=0.6077  (1-NMAE=0.8702, FICR=0.3451)

최고 LightGBM 변형: dyn_w_q60 (B안 평균 0.6124)

=== dyn_w_q60 + dyn_mlp 블렌드 ===


,A안,B안 평균,B안 최솟값
dyn_w_q60 비중,,,
0.0,0.5997,0.5879,0.5567
0.1,0.6097,0.5960,0.5650
0.2,0.6169,0.6026,0.5711
0.3,0.6227,0.6080,0.5767
0.4,0.6261,0.6116,0.5821
0.5,0.6283,0.6152,0.5878
0.6,0.6285,0.6165,0.5910
0.7,0.6288,0.6170,0.5934
0.8,0.6275,0.6150,0.5943


**확인할 것**: 블렌드 곡선이 **완만한 봉우리**를 그리는지 보고, 봉우리 근처의 **둥근 값**(0.6, 0.7 같은)을 고르세요. `B안 최솟값`(가장 나쁜 fold)도 같이 올라가야 안전합니다 — 평균만 오르고 최악 fold가 나빠지면 2025년 실전에서 위험합니다.

## 13-6. 최종 진단 — 어디까지 왔고, 다음에 뭘 노려야 하나

최고 조합의 오차를 산식 관점으로 다시 해부합니다. 10-2절과 같은 표를 만들되, 이번엔 **누수 없는 진짜 숫자**입니다.

특히 **잔차 표준편차 σ**를 봅니다. 10절에서 확인한 핵심 통찰이 "FICR이 낮은 건 예측 위치가 아니라 **오차의 폭**(σ ≈ 0.17×용량, 밴드는 ±0.35σ) 때문"이었으므로, σ가 줄었는지가 이번 개선의 본질적 성패를 말해줍니다.

In [48]:
# 10-2절 분석 도구 재정의 (이 절만 실행해도 되도록 self-contained)
def eval_frame(variant):
    parts = []
    for fold_name, info in FOLD_INFO.items():
        for g in GROUP_COLS:
            cap = CAPACITY_KWH[g]
            ws_col = f"{g}_ws_est_cv_{info['cv_suffix']}"
            d = pd.DataFrame({
                "fold": fold_name, "group": g,
                "actual": info["actual_df"][g].to_numpy(dtype=float),
                "pred": PRED_CACHE[(fold_name, variant, g)].to_numpy(dtype=float),
                "ws": train.loc[info["valid_idx"], ws_col].to_numpy(dtype=float),
            })
            d = d[d["actual"] >= cap * 0.10].copy()
            d["er"] = (d["pred"] - d["actual"]).abs() / cap
            d["signed"] = (d["pred"] - d["actual"]) / cap
            parts.append(d)
    return pd.concat(parts, ignore_index=True)


def wmean(mask, weight):
    return float(np.average(np.asarray(mask, dtype=float), weights=weight))


def error_anatomy(ef, by="group"):
    rows = []
    for key in ef[by].unique():
        d = ef[ef[by] == key]
        w = d["actual"].to_numpy()
        p6 = wmean(d["er"] <= 0.06, w)
        p68 = wmean((d["er"] > 0.06) & (d["er"] <= 0.08), w)
        rows.append({by: key, "표본수": len(d),
                     "평균오차율": d["er"].mean(), "편향": d["signed"].mean(),
                     "잔차σ": d["signed"].std(),
                     "≤6% 비중": p6, "6~8% 비중": p68, ">8% 비중": wmean(d["er"] > 0.08, w),
                     "FICR 재현값": p6 + 0.75 * p68})
    return pd.DataFrame(rows).set_index(by).round(4)


# 누수 상태 기준선(lgb_l1) vs 이번 최고 조합 비교
print(f"=== 그룹별 오차 해부: fix_base (누수 없는 기준선) ===")
display(error_anatomy(eval_frame("fix_base"), "group"))
print(f"\n=== 그룹별 오차 해부: {best_lgb} (이번 최고) ===")
ef_best = eval_frame(best_lgb)
display(error_anatomy(ef_best, "group"))

# σ가 실제로 줄었는지 한눈에
print("\n=== 잔차 표준편차 σ (설비용량 대비) — 작을수록 FICR 상방이 커짐 ===")
sig = []
for v in ["fix_base", "fix_dyn", best_lgb, "dyn_mlp"]:
    e = eval_frame(v)
    sig.append({"변형": v, "σ": round(e["signed"].std(), 4),
                "편향": round(e["signed"].mean(), 4),
                "밴드폭(±0.06)이 σ의 몇 배": round(0.06 / e["signed"].std(), 3)})
display(pd.DataFrame(sig))

# 풍속 구간별 잔차 (미해결 질문 6번 재확인 — 누수 없는 조건에서)
ef_best["regime"] = pd.cut(ef_best["ws"], [-np.inf, 3.0, 12.0, 15.0, np.inf],
                           labels=["calm(<3)", "ramp(3~12)", "rated(12~15)", "high(>=15)"], right=False)
rows = []
for g in GROUP_COLS:
    for r in ["calm(<3)", "ramp(3~12)", "rated(12~15)", "high(>=15)"]:
        d = ef_best[(ef_best["group"] == g) & (ef_best["regime"] == r)]
        if len(d) == 0:
            continue
        rows.append({"group": g, "regime": r, "표본수": len(d),
                     "평균오차율": round(d["er"].mean(), 4), "편향": round(d["signed"].mean(), 4),
                     "≤6% 통과율": round(wmean(d["er"] <= 0.06, d["actual"].to_numpy()), 4)})
print(f"\n=== 풍속 구간별 잔차 ({best_lgb}) — 그룹별 보정이 필요한지 확인 ===")
display(pd.DataFrame(rows))

=== 그룹별 오차 해부: fix_base (누수 없는 기준선) ===


,표본수,평균오차율,편향,잔차σ,≤6% 비중,6~8% 비중,>8% 비중,FICR 재현값
group,,,,,,,,
kpx_group_1,12608,0.1347,-0.0666,0.1597,0.2379,0.0764,0.6857,0.2952
kpx_group_2,12553,0.1285,-0.0209,0.1700,0.3336,0.0904,0.5760,0.4014
kpx_group_3,11331,0.1495,-0.0647,0.1792,0.2052,0.0647,0.7301,0.2537



=== 그룹별 오차 해부: dyn_w_q60 (이번 최고) ===


,표본수,평균오차율,편향,잔차σ,≤6% 비중,6~8% 비중,>8% 비중,FICR 재현값
group,,,,,,,,
kpx_group_1,12608,0.1328,0.0360,0.1658,0.3036,0.1012,0.5952,0.3795
kpx_group_2,12553,0.1393,0.0756,0.1678,0.3644,0.1052,0.5305,0.4432
kpx_group_3,11331,0.1512,0.0496,0.1830,0.2462,0.0775,0.6763,0.3044



=== 잔차 표준편차 σ (설비용량 대비) — 작을수록 FICR 상방이 커짐 ===


,변형,σ,편향,밴드폭(±0.06)이 σ의 몇 배
0,fix_base,0.1708,-0.0503,0.351
1,fix_dyn,0.1679,-0.0502,0.357
2,dyn_w_q60,0.1728,0.0539,0.347
3,dyn_mlp,0.1768,-0.0419,0.339



=== 풍속 구간별 잔차 (dyn_w_q60) — 그룹별 보정이 필요한지 확인 ===


,group,regime,표본수,평균오차율,편향,≤6% 통과율
0,kpx_group_1,calm(<3),9,0.1080,-0.1080,0.0940
1,kpx_group_1,ramp(3~12),11344,0.1360,0.0423,0.2891
2,kpx_group_1,rated(12~15),1206,0.1017,-0.0246,0.3797
3,kpx_group_1,high(>=15),49,0.1761,0.1027,0.3473
4,kpx_group_2,calm(<3),54,0.1188,-0.1132,0.1639
5,kpx_group_2,ramp(3~12),10203,0.1470,0.0780,0.3070
6,kpx_group_2,rated(12~15),1897,0.1051,0.0710,0.4884
7,kpx_group_2,high(>=15),399,0.1081,0.0631,0.6198
8,kpx_group_3,calm(<3),125,0.1288,-0.1234,0.0987
9,kpx_group_3,ramp(3~12),10246,0.1516,0.0606,0.2521


**확인할 것 — 이번 작업의 성패를 가르는 숫자**

1. **σ가 줄었는가** (`fix_base` → `best_lgb`). 0.17 → 0.16이면 미미하고, 0.15 이하로 내려가면 lag/lead가 실제로 효과가 있었다는 뜻입니다. **"밴드폭이 σ의 몇 배" 값이 0.35에서 얼마나 올라갔는지**가 FICR 상방을 직접 결정합니다
2. **편향이 0에 가까워졌는가** — τ와 가중이 의도대로 작동했다면 -0.05에서 0 근처로 와야 합니다
3. **풍속 구간별 표에서 group_3의 rated/high 과소예측(-0.15/-0.13)이 완화됐는가** — 여전히 크면 05_tuning에서 **그룹별 보정**을 별도로 다뤄야 합니다

### 이 절이 끝나면
- `reports/04_model_selection.md`를 작성합니다 (**13절 숫자 기준. 9~12절의 누수 상태 숫자는 인용 금지**)
- HANDOFF.md의 "성능 개선 아이디어 목록"에서 다음 우선순위를 고릅니다: B2(그룹별 τ) → C1(MLP 제대로 학습) → B1(분포 기반 결정) → A2/A3(공간 CNN·시퀀스 모델)

---

# 14. 오라클 실험 — "바람을 완벽히 알았다면 몇 점인가"

## 왜 이걸 먼저 하는가

13절에서 확인한 것: **이번 개선(+0.028)은 전부 "예측을 FICR 밴드의 좋은 위치로 옮긴 것"이고, 오차의 폭(σ)은 하나도 못 줄였다.** 편향도 -0.050에서 +0.054로 반대편까지 넘어가 더 밀 여지가 없다. **남은 상방은 전부 σ에 있다.**

그런데 σ를 줄이는 길(공간 CNN, 시퀀스 모델, 풍속 추정 개선)은 전부 비용이 큽니다. **투자하기 전에 그 길에 실제로 상방이 있는지부터 재는 것**이 이 절의 목적입니다.

방법은 간단합니다. **`scada_ws_{group}`(터빈 나셀에서 실측한 진짜 풍속)을 풍속 피처 자리에 넣고 같은 모델을 돌립니다.** 예보 오차가 0인 세계에서 몇 점이 나오는지 재는 것입니다.

> ## 🚨 이건 진단 전용입니다
> `scada_ws_*`는 **test(2025년)에 존재하지 않습니다.** 실제 예측에 쓰면 그 순간 예측 자체가 불가능해집니다. 이 절의 결과는 **"방향 결정용 상한선"** 으로만 씁니다. 여기서 만든 어떤 컬럼도 제출용 모델에 들어가면 안 됩니다.

## 오차를 세 조각으로 나눠 본다

우리 예측 오차 σ(현재 0.173)는 개념적으로 세 부분의 합입니다.

| 조각 | 정체 | 줄일 수 있나 |
|---|---|---|
| ① **예보 오차** | 예보 풍속 ≠ 실제 풍속 | **가능** — 더 좋은 풍속 추정·공간 CNN·시퀀스 모델 |
| ② **모델 오차** | 바람을 알아도 모델이 발전량 매핑을 덜 배움 | **가능** — 튜닝·앙상블·더 좋은 구조 |
| ③ **예측 불가 성분** | 같은 풍속인데도 발전량이 다름 (터빈 가동상태, 정비, 계통 제약, 난류, 후류 등) | **불가능** — 이게 진짜 바닥 |

이 절은 **①이 얼마나 큰지**를 재고, 동시에 **③의 바닥이 어디인지**도 함께 잽니다.

## 해석 규칙을 미리 정해둡니다 (사전 등록)

결과를 본 뒤에 유리한 쪽으로 해석하지 않기 위해, **판단 기준을 먼저 적어둡니다.**

| 오라클 점수 | 해석 | 다음 행동 |
|---|---|---|
| **0.70 이상** | 예보 오차가 주범. 풍속만 잘 맞히면 크게 오른다 | 공간 CNN(격자를 이미지로) · 시퀀스 모델 · 풍속 추정 개선에 집중 투자 |
| **0.65 ~ 0.70** | 예보 오차가 상당하지만 절대적이진 않음 | 풍속 개선과 산식 최적화를 병행 |
| **0.65 미만** | 바람을 완벽히 알아도 별로 안 오름 | **풍속 개선 포기.** 분포 예측 + 기대 FICR 최대화, 앙상블, 관측 불가 요인 모델링으로 방향 전환 |

(현재 최고: `dyn_w_q60` A안 0.6252 / B평균 0.6124)

## 14-1. 먼저 바닥(③ 예측 불가 성분)을 잰다

**"실측 풍속이 같은데도 발전량이 얼마나 다른가"** 를 직접 셉니다. 실측 풍속을 0.5 m/s 구간으로 자르고, **같은 구간 안에서** 발전량이 얼마나 흩어지는지(표준편차)를 봅니다.

이 값이 바로 **"바람을 완벽히 알아도 남는 오차"** 입니다. 오라클 모델의 σ는 이 값보다 작아질 수 없습니다.

> 비유: 키가 정확히 같은 사람들끼리 모아놔도 몸무게는 제각각입니다. 그 제각각인 정도가 "키만으로는 절대 알 수 없는 부분"입니다.

채점 대상(실제 ≥ 설비용량 10%) 행만 대상으로 계산합니다.

In [49]:
print("=== 실측 풍속 0.5m/s 구간 내부의 발전량 흩어짐 = 예측 불가 성분의 바닥 ===\n")
floor_rows = []
for g in GROUP_COLS:
    cap = CAPACITY_KWH[g]
    d = train[[f"scada_ws_{g}", g]].dropna()
    d = d[d[g] >= cap * 0.10]                      # 채점 대상 행만
    bins = pd.cut(d[f"scada_ws_{g}"], np.arange(0, 26, 0.5))
    grp = d.groupby(bins, observed=True)[g]
    # 구간별 표준편차를 표본수로 가중평균 (표본 많은 구간이 더 대표적)
    stds, ns = grp.std(), grp.size()
    ok = stds.notna() & (ns >= 10)
    floor = float(np.average(stds[ok], weights=ns[ok])) / cap
    floor_rows.append({"group": g, "표본수": len(d), "예측불가 σ(용량대비)": round(floor, 4),
                       "밴드폭 ÷ 이 σ": round(0.06 / floor, 3)})
display(pd.DataFrame(floor_rows))

print("\n=== 현재 실제 잔차 σ와 비교 ===")
print("  dyn_w_q60 전체 σ = 0.1728  (13-6절)")
print("  → 이 바닥보다 얼마나 위에 있는지가 '아직 줄일 수 있는 여지'입니다.")

=== 실측 풍속 0.5m/s 구간 내부의 발전량 흩어짐 = 예측 불가 성분의 바닥 ===



,group,표본수,예측불가 σ(용량대비),밴드폭 ÷ 이 σ
0,kpx_group_1,15903,0.0748,0.802
1,kpx_group_2,15856,0.0612,0.981
2,kpx_group_3,9346,0.0817,0.734



=== 현재 실제 잔차 σ와 비교 ===
  dyn_w_q60 전체 σ = 0.1728  (13-6절)
  → 이 바닥보다 얼마나 위에 있는지가 '아직 줄일 수 있는 여지'입니다.


**확인할 것**: `밴드폭 ÷ 이 σ` 값이 FICR의 **이론적 최대치**를 정합니다. 이 값이 1.0이면 정규분포 기준 통과율 68%, 0.6이면 45%입니다. 현재 우리 값은 0.347입니다.

group_3의 바닥이 가장 클 것으로 예상되는데(UNISON 터빈), 그렇다면 **group_3의 점수가 낮은 것은 모델 탓이 아니라 그 그룹이 원래 더 어렵기 때문**이라는 뜻입니다.

## 14-2. 오라클 피처 프레임 만들기

`dyn_w_q60`과 **모든 조건을 똑같이** 두고 풍속 계열만 바꿉니다.

| | `dyn_w_q60` (실제) | `oracle_ws` (오라클) |
|---|---|---|
| 풍속 | `ws_est_cv_*` (예보 6종의 Ridge 결합) | **`scada_ws_*` (나셀 실측)** |
| 파워커브 | `power_curve_est_cv_*` (예보 풍속 통과) | **실측 풍속을 통과시킨 파워커브** |
| 파생 8종 (밀도보정/제곱/세제곱/구간더미 등) | 예보 풍속 기준 | **실측 풍속 기준** |
| lag/lead 26종 | 예보 풍속 기준 | **실측 풍속 기준** |
| 나머지 800여 개 격자 피처 | 그대로 | 그대로 |
| 학습 설정 | quantile τ=0.60, actual 가중 | **동일** |

**파워커브도 실측 기준으로 다시 적합합니다.** 파워커브가 gain 상위 피처라 이것만 예보 기준으로 두면 오라클이 아니게 됩니다. 적합은 **학습 구간에서만**(fold-safe) 합니다 — 라벨을 미래에서 끌어오면 오라클이 아니라 그냥 부정행위가 됩니다.

**결측 처리**: `scada_ws_kpx_group_3`은 라벨이 있는데 실측이 없는 행이 **4개** 있습니다(2023-05-23, 2024-07-30 등). 이 4행만 그 fold의 예보 추정풍속으로 채웁니다.

In [50]:
def fit_power_curve_oracle(ws, power, bin_width=0.5, min_count=10):
    """03_features 12-1절과 같은 로직: 0.5m/s 구간평균 + 단조증가 강제."""
    edges = np.arange(0, np.nanmax(ws) + 2 * bin_width, bin_width)
    idx = np.digitize(ws, edges) - 1
    vals = np.full(len(edges), np.nan)
    for b in range(len(edges)):
        m = idx == b
        if m.sum() >= min_count:
            vals[b] = power[m].mean()
    vals = pd.Series(vals).interpolate(limit_direction="both").fillna(0.0).to_numpy()
    return edges, np.maximum.accumulate(vals)   # 단조증가 강제


def apply_power_curve_oracle(ws, edges, vals):
    return vals[np.clip(np.digitize(ws, edges) - 1, 0, len(vals) - 1)]


def oracle_wind(df, g, cv_suffix):
    """나셀 실측 풍속. 결측(group_3 4행)만 그 fold의 예보 추정풍속으로 채움."""
    return df[f"scada_ws_{g}"].fillna(df[f"{g}_ws_est_cv_{cv_suffix}"])


def build_oracle_frame(df, g, cv_suffix, train_mask, dynamics=True):
    """🚨 진단 전용. scada_ws_*는 test에 없으므로 절대 제출 모델에 쓰지 말 것.
    예보 기반 풍속/파워커브 계열을 전부 빼고 실측 기반으로 갈아끼운 프레임."""
    all_cv = set(all_cv_cols_for_group(g))
    exclude = set(leaky_cols_for_group(g)) | all_cv          # 예보 풍속·파워커브 전부 제거
    base_cols = [c for c in GROUP_SPECIFIC_COLS[g] if c not in exclude]

    o_ws, o_pc = f"{g}_ws_oracle", f"{g}_power_curve_oracle"
    tmp = pd.DataFrame({"kst_dtm": df["kst_dtm"], f"{g}_air_density": df[f"{g}_air_density"]},
                       index=df.index)
    tmp[o_ws] = oracle_wind(df, g, cv_suffix)
    if g in ICING_RISK_GROUPS:                                # 결빙 피처 계산에 기온이 필요
        tcol = f"ldaps_g{GROUP_NEAREST_LDAPS[g]}_heightAboveGround_2_t"
        tmp[tcol] = df[tcol]

    # 파워커브는 학습 구간에서만 적합 (fold-safe — 검증 구간 라벨은 절대 안 봄)
    fit_idx = train_mask & df[g].notna()
    edges, vals = fit_power_curve_oracle(tmp.loc[fit_idx, o_ws].to_numpy(dtype=float),
                                         df.loc[fit_idx, g].to_numpy(dtype=float))
    tmp[o_pc] = apply_power_curve_oracle(tmp[o_ws].to_numpy(dtype=float), edges, vals)

    parts = [df[COMMON_RAW_COLS + base_cols], tmp[[o_ws, o_pc]],
             add_fold_safe_ws_features(tmp, g, o_ws)]
    if dynamics:
        parts.append(add_forecast_dynamics(tmp, [o_ws, o_pc]))
    out = pd.concat(parts, axis=1)
    assert out.isna().sum().sum() == 0, "오라클 프레임에 결측"
    return out


# 확인: 예보판과 오라클판의 피처 수가 같아야 공정한 비교다
info = FOLD_INFO["A안(2024)"]
for g in GROUP_COLS:
    n_real = build_group_feature_frame_v2(train, g, info["cv_suffix"], dynamics=True).shape[1]
    n_orac = build_oracle_frame(train, g, info["cv_suffix"], info["train_mask"], dynamics=True).shape[1]
    print(f"{g}: 실제 {n_real}개 / 오라클 {n_orac}개", "✓ 동일" if n_real == n_orac else "⚠️ 다름")

kpx_group_1: 실제 848개 / 오라클 848개 ✓ 동일
kpx_group_2: 실제 848개 / 오라클 848개 ✓ 동일
kpx_group_3: 실제 847개 / 오라클 847개 ✓ 동일


**확인할 것**: 실제판과 오라클판의 피처 수가 **같아야** 합니다. 다르면 무엇이 빠졌거나 더해진 것이니 멈추고 알려주세요 — 피처 수가 다르면 "풍속의 차이"가 아니라 "피처 개수의 차이"를 재는 셈이 됩니다.

## 14-3. 실험 실행

네 가지를 비교합니다. 앞의 두 개는 **머신러닝 없이** 파워커브만 쓰는 것이라 학습이 0번입니다.

| 변형 | 풍속 | 방법 | 의미 |
|---|---|---|---|
| `pc_forecast` | 예보 | 파워커브만 (ML 없음) | 7-2절 베이스라인의 fold-safe판 |
| `pc_oracle` | **실측** | 파워커브만 (ML 없음) | **"바람을 알고 물리만 쓰면"** |
| `dyn_w_q60` | 예보 | LightGBM (13절 최고) | 현재 우리 위치 |
| **`oracle_ws`** | **실측** | LightGBM (동일 설정) | **상한선** |

`pc_forecast` → `pc_oracle`의 차이가 **순수한 "풍속 정보의 가치"** 이고, `dyn_w_q60` → `oracle_ws`의 차이가 **"우리 모델에서 예보 오차가 차지하는 몫"** 입니다.

⚠️ 12번 학습(`oracle_ws`만). 나머지는 즉시.

In [51]:
def _fit_lgbm_on(X_full, g, train_mask, valid_idx, objective, alpha, weight_mode, n_estimators=2000):
    """이미 만들어진 피처 프레임에 LightGBM을 학습시키는 공통 부분."""
    fit_idx = train_mask & train[g].notna()
    cutoff_es = train.loc[fit_idx, "kst_dtm"].quantile(0.9)
    tr_mask = fit_idx & (train["kst_dtm"] <= cutoff_es)
    es_mask = fit_idx & (train["kst_dtm"] > cutoff_es)
    Xtr, ytr = X_full.loc[tr_mask], train.loc[tr_mask, g]
    Xes, yes = X_full.loc[es_mask], train.loc[es_mask, g]

    params = dict(objective=objective, random_state=SEED, n_estimators=n_estimators,
                  verbosity=-1, importance_type="gain")
    if alpha is not None:
        params["alpha"] = alpha
    w_tr, w_es = make_sample_weight(ytr, g, weight_mode), make_sample_weight(yes, g, weight_mode)

    model = lgb.LGBMRegressor(**params)
    model.fit(Xtr, ytr, sample_weight=w_tr, eval_set=[(Xes, yes)],
              eval_sample_weight=[w_es] if w_es is not None else None,
              eval_metric="l1", callbacks=[lgb.early_stopping(50, verbose=False)])
    pred = pd.Series(model.predict(X_full.loc[valid_idx]), index=valid_idx)
    return pred.clip(lower=0, upper=CAPACITY_KWH[g])


# ── (1) ML 없는 파워커브 2종 (학습 0번) ────────────────────────────
def pc_forecast_fn(g, cv_suffix, train_mask, valid_idx):
    """fold-safe 예보 파워커브 값을 그대로 예측으로 사용."""
    p = train.loc[valid_idx, f"{g}_power_curve_est_cv_{cv_suffix}"]
    return p.clip(lower=0, upper=CAPACITY_KWH[g])


def pc_oracle_fn(g, cv_suffix, train_mask, valid_idx):
    """실측 풍속 -> (학습구간에서 적합한) 파워커브 -> 예측."""
    ws = oracle_wind(train, g, cv_suffix)
    fit_idx = train_mask & train[g].notna()
    edges, vals = fit_power_curve_oracle(ws[fit_idx].to_numpy(dtype=float),
                                         train.loc[fit_idx, g].to_numpy(dtype=float))
    p = pd.Series(apply_power_curve_oracle(ws.loc[valid_idx].to_numpy(dtype=float), edges, vals),
                  index=valid_idx)
    return p.clip(lower=0, upper=CAPACITY_KWH[g])


# ── (2) 오라클 LightGBM (dyn_w_q60과 동일 설정) ────────────────────
def oracle_ws_fn(g, cv_suffix, train_mask, valid_idx):
    X = build_oracle_frame(train, g, cv_suffix, train_mask, dynamics=True)
    return _fit_lgbm_on(X, g, train_mask, valid_idx, "quantile", 0.60, "actual")


stage4 = []
print("=== pc_forecast (예보 파워커브, ML 없음) ===")
stage4.append(run_variant("pc_forecast", pc_forecast_fn))
print("=== pc_oracle (실측 파워커브, ML 없음) ===")
stage4.append(run_variant("pc_oracle", pc_oracle_fn))
print("=== oracle_ws (실측 풍속 + LightGBM, 12번 학습) ===")
stage4.append(run_variant("oracle_ws", oracle_ws_fn))

display(summarize(stage1 + stage2 + stage3 + stage4))

=== pc_forecast (예보 파워커브, ML 없음) ===
  [pc_forecast] A안(2024): score=0.5825  (1-NMAE=0.8531, FICR=0.3119)
  [pc_forecast] B안 fold1: score=0.5550  (1-NMAE=0.8284, FICR=0.2817)
  [pc_forecast] B안 fold2: score=0.5653  (1-NMAE=0.8472, FICR=0.2835)
  [pc_forecast] B안 fold3: score=0.6006  (1-NMAE=0.8595, FICR=0.3416)
=== pc_oracle (실측 파워커브, ML 없음) ===
  [pc_oracle] A안(2024): score=0.8268  (1-NMAE=0.9506, FICR=0.7031)
  [pc_oracle] B안 fold1: score=0.8198  (1-NMAE=0.9429, FICR=0.6967)
  [pc_oracle] B안 fold2: score=0.8211  (1-NMAE=0.9492, FICR=0.6930)
  [pc_oracle] B안 fold3: score=0.8353  (1-NMAE=0.9523, FICR=0.7183)
=== oracle_ws (실측 풍속 + LightGBM, 12번 학습) ===
  [oracle_ws] A안(2024): score=0.8587  (1-NMAE=0.9536, FICR=0.7638)
  [oracle_ws] B안 fold1: score=0.8543  (1-NMAE=0.9482, FICR=0.7605)
  [oracle_ws] B안 fold2: score=0.8797  (1-NMAE=0.9549, FICR=0.8045)
  [oracle_ws] B안 fold3: score=0.8512  (1-NMAE=0.9538, FICR=0.7487)


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
oracle_ws,0.8587,0.8543,0.8797,0.8512,0.8618,0.0156
pc_oracle,0.8268,0.8198,0.8211,0.8353,0.8254,0.0086
dyn_w_q60,0.6252,0.5926,0.6086,0.6361,0.6124,0.0220
dyn_w_q65,0.6214,0.5927,0.6057,0.6382,0.6122,0.0234
dyn_w,0.6171,0.5869,0.6089,0.6337,0.6098,0.0234
dyn_q60,0.6174,0.5789,0.6137,0.6326,0.6084,0.0273
dyn_w_q70,0.6138,0.5931,0.5927,0.6331,0.6063,0.0232
fix_dyn,0.5991,0.5649,0.5981,0.6116,0.5915,0.0240
fix_base,0.5915,0.5665,0.5898,0.6102,0.5888,0.0218


## 14-4. 오차 분해 — 예보 오차가 차지하는 몫

σ를 나란히 놓고 봅니다. 실제 모델과 오라클의 σ 차이가 **예보 오차가 만들어낸 몫**입니다.

오차가 서로 독립이라고 가정하면 대략 이렇게 쪼갤 수 있습니다.

`σ(실제)² ≈ σ(오라클)² + σ(예보오차 기여)²`

즉 **예보오차 기여 ≈ √(σ실제² − σ오라클²)** 입니다. (완전히 독립은 아니므로 어림값입니다 — 그래도 규모를 보는 데는 충분합니다.)

In [52]:
print("=== σ 비교 ===")
rows = []
for v in ["pc_forecast", "pc_oracle", "fix_base", "dyn_w_q60", "oracle_ws"]:
    e = eval_frame(v)
    rows.append({"변형": v, "σ": round(e["signed"].std(), 4), "편향": round(e["signed"].mean(), 4),
                 "밴드폭 ÷ σ": round(0.06 / e["signed"].std(), 3)})
sig_df = pd.DataFrame(rows)
display(sig_df)

s_real = float(sig_df.loc[sig_df["변형"] == "dyn_w_q60", "σ"].iloc[0])
s_orac = float(sig_df.loc[sig_df["변형"] == "oracle_ws", "σ"].iloc[0])
print(f"\n실제 σ = {s_real:.4f}, 오라클 σ = {s_orac:.4f}")
if s_orac < s_real:
    print(f"→ 예보 오차 기여 ≈ {np.sqrt(max(s_real**2 - s_orac**2, 0)):.4f}"
          f"  (전체 σ의 {np.sqrt(max(s_real**2-s_orac**2,0))/s_real*100:.0f}%)")
else:
    print("→ 오라클 σ가 더 크거나 같음: 풍속 정보가 병목이 아니라는 뜻")

print("\n=== 그룹별 오차 해부: oracle_ws ===")
display(error_anatomy(eval_frame("oracle_ws"), "group"))

print("\n=== 점수 요약 (A안 / B안 평균) ===")
tbl = summarize(stage1 + stage2 + stage4)
for v in ["pc_forecast", "pc_oracle", "fix_base", "dyn_w_q60", "oracle_ws"]:
    print(f"  {v:14s}  A안 {tbl.loc[v, 'A안(2024)']:.4f}   B평균 {tbl.loc[v, 'B안 평균']:.4f}")

=== σ 비교 ===


,변형,σ,편향,밴드폭 ÷ σ
0,pc_forecast,0.1950,-0.0421,0.308
1,pc_oracle,0.0801,-0.0018,0.749
2,fix_base,0.1708,-0.0503,0.351
3,dyn_w_q60,0.1728,0.0539,0.347
4,oracle_ws,0.0786,0.0133,0.763



실제 σ = 0.1728, 오라클 σ = 0.0786
→ 예보 오차 기여 ≈ 0.1539  (전체 σ의 89%)

=== 그룹별 오차 해부: oracle_ws ===


,표본수,평균오차율,편향,잔차σ,≤6% 비중,6~8% 비중,>8% 비중,FICR 재현값
group,,,,,,,,
kpx_group_1,12608,0.0420,0.0051,0.0678,0.7357,0.0949,0.1694,0.8069
kpx_group_2,12553,0.0404,0.0286,0.0711,0.7943,0.0769,0.1288,0.8519
kpx_group_3,11331,0.0592,0.0054,0.0939,0.5856,0.0803,0.3341,0.6458



=== 점수 요약 (A안 / B안 평균) ===
  pc_forecast     A안 0.5825   B평균 0.5737
  pc_oracle       A안 0.8268   B평균 0.8254
  fix_base        A안 0.5915   B평균 0.5888
  dyn_w_q60       A안 0.6252   B평균 0.6124
  oracle_ws       A안 0.8587   B평균 0.8618


## 14-5. 결론 내리기 — 14-0절에 미리 적어둔 규칙대로

`oracle_ws`의 B안 평균을 14-0절 표의 세 구간(0.70 이상 / 0.65~0.70 / 0.65 미만)에 대입해서 다음 방향을 정합니다. **결과를 보고 기준을 바꾸지 않습니다.**

함께 볼 것:

1. **`pc_oracle` vs `pc_forecast`** — 머신러닝을 걷어낸 순수한 "풍속 정보의 가치". 이 차이가 작으면 풍속 개선의 상방 자체가 작다는 뜻입니다
2. **`oracle_ws` vs `pc_oracle`** — 바람을 안 상태에서 **머신러닝이 추가로 벌어주는 몫**. 이게 크면 "바람 외의 정보(기온·풍향·시간대 등)도 쓸모 있다"는 뜻입니다
3. **σ 분해** — 예보 오차가 전체 σ의 몇 %인가
4. **14-1절 바닥과 `oracle_ws`의 σ 비교** — 오라클이 바닥에 거의 닿았으면 모델링은 더 짜낼 게 없고, 아직 멀면 튜닝·앙상블 여지가 남아 있습니다
5. **그룹별 오라클 편차** — group_3의 오라클도 낮으면 "group_3는 원래 어려운 그룹"이 확정됩니다

> ⚠️ 다시 한 번: 여기서 만든 `{group}_ws_oracle`, `{group}_power_curve_oracle`, `build_oracle_frame`, `oracle_ws_fn`은 **전부 진단 전용**입니다. `05_tuning.ipynb`나 `train.ipynb`로 절대 옮기지 마세요.

---

# 15. 확정된 개선 4종 — 리드타임 / 피처 축소 / 그룹별 τ / 시드 앙상블

## 14절 결과가 말해준 것 (먼저 읽기)

| 변형 | 풍속 | 방법 | A안 | B평균 | 잔차 σ |
|---|---|---|---|---|---|
| `pc_forecast` | 예보 | 파워커브만 | 0.5825 | 0.5737 | 0.1950 |
| `dyn_w_q60` | 예보 | LightGBM | 0.6252 | 0.6124 | 0.1728 |
| **`pc_oracle`** | **실측** | **파워커브만 (ML 없음)** | **0.8268** | **0.8254** | **0.0801** |
| **`oracle_ws`** | **실측** | LightGBM | **0.8587** | **0.8618** | **0.0786** |

**예보 오차가 전체 σ의 89%다.** 그리고 머신러닝을 전부 걷어내고 파워커브 하나만 써도 **바람만 정확하면 0.83**이 나온다. 우리는 850개 피처 + LightGBM으로 0.61이다.

게다가 `oracle_ws`의 그룹별 σ(0.068/0.071/0.094)는 14-1절에서 잰 예측 불가 바닥(0.075/0.061/0.082)과 거의 같다 — **바람만 알면 모델링은 이미 바닥에 닿는다.**

> **결론: 앞으로의 진짜 레버는 "풍속 추정 정확도"다.** 현재 `ws_est`는 예보 풍속 **6개 컬럼에 Ridge**를 돌린 것뿐인데, 우리에겐 850개 피처가 있다. SCADA 실측 풍속을 타깃으로 제대로 된 모델을 학습시키는 것이 16절 과제다(누수 아님 — SCADA는 train에 있고, 학습된 모델은 test 예보에 적용하면 된다).

이 절(15절)은 그와 별개로 **확실하고 싼 개선 4종**을 정리한다. 각각은 작지만 합치면 의미가 있고, 무엇보다 **피처 축소는 16절 실험 속도를 몇 배로 올려준다.**

## 순서와 그 근거

| 순서 | 항목 | 왜 이 자리인가 |
|---|---|---|
| 15-1 | 리드타임 피처 | 피처 추가라 가장 먼저. 공짜 |
| **15-2** | **피처 축소** | 850→100개면 **이후 모든 실험이 몇 배 빨라진다.** 뒤로 미룰 이유가 없음 |
| 15-3 | 그룹별 τ | 최적 τ는 모델의 편향에 따라 달라지므로 **피처가 확정된 뒤에** 정해야 함 |
| 15-4 | 시드 앙상블 | 최종 구성에 씌우는 마감재. **먼저 하면 이후 모든 실험 비용이 5배**가 됨 |

## 15-1. 예보 리드타임 피처 — 기대는 낮지만 공짜

**리드타임(forecast lead time)** = "이 예보가 발표된 뒤 몇 시간 지난 시점을 예측하는가". 예보는 멀리 볼수록 틀리므로, 원래는 강력한 피처가 되어야 한다.

**그런데 확인해보니 우리 데이터에서는 새로운 정보가 아니다.**

- 예보 발표 시각이 **항상 13시 한 번뿐**이다 → 리드타임이 **시각(hour)으로 100% 결정된다**(시각별 리드타임 고유값이 전부 1개)
- 즉 `lead = f(hour)`이고, 이미 `hour_sin`/`hour_cos`가 피처에 있다
- 리드타임별 실제 예보 오차도 단조롭지 않다(12~17h 1.38 → 23~29h **1.23** → 29~35h 1.38 m/s). 리드타임 효과와 바람의 일주기가 뒤섞여 있어서다

**그래도 넣는 이유**: 트리 모델에게 "리드타임"이라는 **직선 축**을 하나 주는 것은 `hour_sin/cos`라는 **원형 좌표 2개**로 같은 구간을 표현하는 것보다 분할이 쉽다. 정보량은 같지만 표현이 다르다. 비용이 0이므로 넣고 확인만 한다.

> 정확히는: `hour_sin/cos`로 "리드타임 30시간 이상"을 잘라내려면 2차원 조건이 필요하지만, `forecast_lead_hours`가 있으면 `> 30` 한 번이면 된다.

`hour_raw`(0~23 정수)도 함께 넣는다 — 같은 이유다.

In [53]:
def add_lead_features(df):
    """예보 리드타임(발표 후 경과시간)과 raw hour. 예보 발표시각은 예보와 함께 주어지는
    메타데이터이므로 누수가 아니다."""
    lead = (df["kst_dtm"] - df["data_available_kst_dtm_gfs"]).dt.total_seconds() / 3600.0
    return pd.DataFrame({"forecast_lead_hours": lead.astype(float),
                         "hour_raw": df["kst_dtm"].dt.hour.astype(float)}, index=df.index)


def build_group_feature_frame_v3(df, g, cv_suffix, dynamics=True, lead_feat=True):
    out = build_group_feature_frame_v2(df, g, cv_suffix, dynamics=dynamics)
    if lead_feat:
        out = pd.concat([out, add_lead_features(df)], axis=1)
    return out


def _lgbm_train(X_full, g, train_mask, objective, alpha, weight_mode, seed, n_estimators=2000):
    """학습 구간만 써서 LightGBM 하나 학습하고 모델을 돌려준다(예측은 호출부에서)."""
    fit_idx = train_mask & train[g].notna()
    cutoff_es = train.loc[fit_idx, "kst_dtm"].quantile(0.9)
    tr = fit_idx & (train["kst_dtm"] <= cutoff_es)
    es = fit_idx & (train["kst_dtm"] > cutoff_es)
    Xtr, ytr = X_full.loc[tr], train.loc[tr, g]
    Xes, yes = X_full.loc[es], train.loc[es, g]
    params = dict(objective=objective, random_state=seed, n_estimators=n_estimators,
                  verbosity=-1, importance_type="gain")
    if alpha is not None:
        params["alpha"] = alpha
    w_tr, w_es = make_sample_weight(ytr, g, weight_mode), make_sample_weight(yes, g, weight_mode)
    m = lgb.LGBMRegressor(**params)
    m.fit(Xtr, ytr, sample_weight=w_tr, eval_set=[(Xes, yes)],
          eval_sample_weight=[w_es] if w_es is not None else None,
          eval_metric="l1", callbacks=[lgb.early_stopping(50, verbose=False)])
    return m


def lgbm3_fit_fn(*, objective="quantile", alpha=0.60, weight_mode="actual",
                 lead_feat=True, keep_cols_fn=None, seed=SEED):
    """run_variant에 넘길 fit_fn 생성기. keep_cols_fn(g, cv_suffix, train_mask, X)가 주어지면
    그 함수가 돌려주는 컬럼만 써서 다시 학습한다(피처 축소용)."""
    def fit_fn(g, cv_suffix, train_mask, valid_idx):
        X = build_group_feature_frame_v3(train, g, cv_suffix, dynamics=True, lead_feat=lead_feat)
        if keep_cols_fn is not None:
            X = X[keep_cols_fn(g, cv_suffix, train_mask, X)]
        m = _lgbm_train(X, g, train_mask, objective, alpha, weight_mode, seed)
        p = pd.Series(m.predict(X.loc[valid_idx]), index=valid_idx)
        return p.clip(lower=0, upper=CAPACITY_KWH[g])
    return fit_fn


stage5 = []
print("=== L_lead (dyn_w_q60 + 리드타임 피처) ===")
stage5.append(run_variant("L_lead", lgbm3_fit_fn()))
display(summarize(stage1 + stage2 + stage5))

=== L_lead (dyn_w_q60 + 리드타임 피처) ===
  [L_lead] A안(2024): score=0.6236  (1-NMAE=0.8622, FICR=0.3851)
  [L_lead] B안 fold1: score=0.5946  (1-NMAE=0.8488, FICR=0.3404)
  [L_lead] B안 fold2: score=0.6101  (1-NMAE=0.8538, FICR=0.3665)
  [L_lead] B안 fold3: score=0.6396  (1-NMAE=0.8703, FICR=0.4089)


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
L_lead,0.6236,0.5946,0.6101,0.6396,0.6148,0.0229
dyn_w_q60,0.6252,0.5926,0.6086,0.6361,0.6124,0.0220
dyn_w_q65,0.6214,0.5927,0.6057,0.6382,0.6122,0.0234
dyn_w,0.6171,0.5869,0.6089,0.6337,0.6098,0.0234
dyn_q60,0.6174,0.5789,0.6137,0.6326,0.6084,0.0273
dyn_w_q70,0.6138,0.5931,0.5927,0.6331,0.6063,0.0232
fix_dyn,0.5991,0.5649,0.5981,0.6116,0.5915,0.0240
fix_base,0.5915,0.5665,0.5898,0.6102,0.5888,0.0218


**확인할 것**: `L_lead`가 `dyn_w_q60`(0.6252 / 0.6124)보다 나은가? **A안·B안 3-fold 전부에서 개선돼야 채택**입니다. 위에서 설명한 대로 새 정보가 아니라 재표현이므로, **차이가 없어도 정상**입니다 — 그러면 그냥 빼면 됩니다(피처가 2개 줄어드는 게 이득).

## 15-2. 피처 축소 — 850개 중 실제로 쓰이는 건 극히 일부

10-3b절에서 **상위 15개 피처가 전체 gain의 44~65%** 를 차지하고, 수백 개가 gain 0.000%였습니다. 나머지는 모델이 아예 쓰지 않거나 잡음으로만 쓰이고 있습니다.

**줄이면 얻는 것 두 가지**
1. **속도** — 850개 → 100개면 학습이 몇 배 빨라집니다. 16절(풍속 모델)과 05_tuning의 실험 횟수가 그만큼 늘어납니다
2. **잡음 감소** — 쓸모없는 피처가 많으면 트리가 우연한 분할을 할 기회가 늘어납니다(특히 표본이 적은 fold1에서)

**어떻게 고르는가 (누수 없이)**

피처 선택도 **학습 데이터만 보고** 해야 합니다. 검증 구간을 보고 고르면 그 자체가 누수입니다. 그래서 **2단계**로 합니다.

1. 각 (fold, 그룹)마다 **그 fold의 학습 구간만으로** 전체 피처 모델을 한 번 학습 → gain 중요도를 구한다
2. 상위 N개만 남기고 **같은 학습 구간으로 다시 학습** → 검증 구간을 예측한다

1단계 중요도는 fold마다 다르므로 fold별로 따로 계산합니다(계산은 캐시해서 한 번만).

세 수준을 비교합니다: **gain>0인 것 전부 / 상위 200 / 상위 100 / 상위 50**.

⚠️ 1단계 12번 + 각 수준 12번 = 총 60번 학습. 단 축소 모델은 훨씬 빠릅니다.

In [54]:
IMP_CACHE = {}


def get_fold_importance(g, cv_suffix, train_mask):
    """해당 fold의 학습 구간만으로 전체 피처 모델을 학습해 gain 중요도를 구한다(캐시)."""
    key = (g, cv_suffix)
    if key not in IMP_CACHE:
        X = build_group_feature_frame_v3(train, g, cv_suffix, dynamics=True, lead_feat=True)
        m = _lgbm_train(X, g, train_mask, "quantile", 0.60, "actual", SEED)
        IMP_CACHE[key] = pd.Series(m.feature_importances_, index=m.feature_name_, dtype=float)
    return IMP_CACHE[key]


def make_keep_fn(top_n):
    """top_n=None이면 'gain>0인 것 전부', 정수면 상위 top_n개."""
    def keep_fn(g, cv_suffix, train_mask, X):
        imp = get_fold_importance(g, cv_suffix, train_mask).reindex(X.columns).fillna(0.0)
        s = imp.sort_values(ascending=False)
        return (s[s > 0].index.tolist() if top_n is None else s.head(top_n).index.tolist())
    return keep_fn


# 먼저 실제로 몇 개가 살아있는지 확인
print("=== fold x 그룹별 'gain > 0'인 피처 개수 (전체 848~850개 중) ===")
for fold_name, info in FOLD_INFO.items():
    row = []
    for g in GROUP_COLS:
        imp = get_fold_importance(g, info["cv_suffix"], info["train_mask"])
        row.append(f"{g.replace('kpx_','')}: {int((imp > 0).sum())}")
    print(f"  {fold_name}  |  " + "  ".join(row))

stage6 = []
for label, n in [("F_gain_pos", None), ("F_top200", 200), ("F_top100", 100), ("F_top50", 50)]:
    print(f"\n=== {label} ===")
    stage6.append(run_variant(label, lgbm3_fit_fn(keep_cols_fn=make_keep_fn(n))))

display(summarize(stage1 + stage2 + stage5 + stage6))

=== fold x 그룹별 'gain > 0'인 피처 개수 (전체 848~850개 중) ===
  A안(2024)  |  group_1: 434  group_2: 680  group_3: 516
  B안 fold1  |  group_1: 551  group_2: 778  group_3: 534
  B안 fold2  |  group_1: 434  group_2: 680  group_3: 516
  B안 fold3  |  group_1: 605  group_2: 659  group_3: 581

=== F_gain_pos ===
  [F_gain_pos] A안(2024): score=0.6248  (1-NMAE=0.8630, FICR=0.3866)
  [F_gain_pos] B안 fold1: score=0.5949  (1-NMAE=0.8492, FICR=0.3405)
  [F_gain_pos] B안 fold2: score=0.6122  (1-NMAE=0.8551, FICR=0.3693)
  [F_gain_pos] B안 fold3: score=0.6386  (1-NMAE=0.8709, FICR=0.4063)

=== F_top200 ===
  [F_top200] A안(2024): score=0.6242  (1-NMAE=0.8618, FICR=0.3865)
  [F_top200] B안 fold1: score=0.5900  (1-NMAE=0.8464, FICR=0.3336)
  [F_top200] B안 fold2: score=0.6111  (1-NMAE=0.8534, FICR=0.3688)
  [F_top200] B안 fold3: score=0.6447  (1-NMAE=0.8708, FICR=0.4187)

=== F_top100 ===
  [F_top100] A안(2024): score=0.6249  (1-NMAE=0.8624, FICR=0.3874)
  [F_top100] B안 fold1: score=0.5896  (1-NMAE=0.8455, FICR=0.3338)

fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
F_top200,0.6242,0.5900,0.6111,0.6447,0.6153,0.0276
F_gain_pos,0.6248,0.5949,0.6122,0.6386,0.6152,0.0220
L_lead,0.6236,0.5946,0.6101,0.6396,0.6148,0.0229
F_top100,0.6249,0.5896,0.6132,0.6384,0.6137,0.0244
F_top50,0.6243,0.5923,0.6105,0.6363,0.6131,0.0221
dyn_w_q60,0.6252,0.5926,0.6086,0.6361,0.6124,0.0220
dyn_w_q65,0.6214,0.5927,0.6057,0.6382,0.6122,0.0234
dyn_w,0.6171,0.5869,0.6089,0.6337,0.6098,0.0234
dyn_q60,0.6174,0.5789,0.6137,0.6326,0.6084,0.0273


**확인할 것**

1. **`gain > 0`인 피처가 몇 개인가** — 850개 중 200~400개 정도로 나올 것으로 예상됩니다. 나머지는 모델이 단 한 번도 쓰지 않은 피처입니다
2. **어느 수준까지 줄여도 점수가 유지되는가** — `F_top100`이 `L_lead`와 비슷하면 **피처의 88%를 버려도 손해가 없다**는 뜻입니다
3. **점수가 오르는 수준이 있는가** — 있다면 잡음 제거 효과입니다. 단 A안·B안 모두에서 올라야 합니다
4. **B안 fold1(가장 표본이 적은 fold)에서 특히 개선되는가** — 축소의 이득은 표본이 적을 때 가장 큽니다

**선택 기준**: 점수가 같다면 **더 적은 쪽**을 고릅니다(속도 이득 + 단순한 모델이 실전에서 안전). 점수가 뚜렷이 떨어지기 직전 수준을 고르세요.

## 15-3. 그룹별 τ — 총점이 그룹별 점수의 평균이라는 사실을 이용

### 먼저: 총점은 그룹별 점수의 단순 평균이다

`src/metric.py`를 식으로 풀어보면:

```
1-NMAE = 1 - (1/3)Σ nmae_g
FICR   =     (1/3)Σ ficr_g
총점   = 0.5(1 - (1/3)Σ nmae_g) + 0.5(1/3)Σ ficr_g
       = (1/3) Σ [ 0.5(1 - nmae_g) + 0.5 ficr_g ]
       = 그룹별 점수의 평균
```

**즉 그룹끼리 서로 영향을 주지 않습니다.** 그러니 각 그룹의 τ를 **독립적으로** 최적화해도 되고, 그게 곧 전체 최적입니다. (이 항등식은 아래 셀에서 실제로 검산합니다.)

### 왜 그룹마다 τ가 달라야 하는가

13-6절에서 `dyn_w_q60`의 그룹별 편향이 group_1 **+0.036** / group_2 **+0.076** / group_3 **+0.050** 이었습니다. group_2는 원래 편향이 가장 작았던 그룹(-0.021)이라 같은 τ를 먹이니 과대 쪽으로 과하게 밀렸습니다.

τ ∈ {0.50, 0.55, 0.60, 0.65, 0.70}을 **15-2에서 고른 피처 축소 수준** 위에서 다시 훑고, 그룹별로 최적을 고릅니다.

### 과적합을 막는 규칙

검증 점수를 보고 그룹마다 파라미터를 고르는 것은 **검증 데이터에 대한 과적합** 위험이 있습니다(3개 그룹 × 5개 τ = 15가지 선택지). 그래서 두 가지 안전장치를 둡니다.

1. **B안 3-fold 평균으로 고르고, A안으로 교차 확인**한다. A안에서도 같은 방향이어야 채택
2. **곡선이 완만하면 가운데 값**을 고른다. 최고점이 뾰족하게 하나만 튀면 잡음일 가능성이 높다

⚠️ 60번 학습(5τ × 4fold × 3그룹). 축소된 피처라 13-4절보다 훨씬 빠를 것입니다.

In [55]:
# ── (0) "총점 = 그룹별 점수의 평균" 항등식 검산 ────────────────────
def group_scores(actual_df, pred_df):
    """그룹별 점수 = 0.5(1-nmae_g) + 0.5*ficr_g. 이들의 평균이 공식 총점과 같아야 한다."""
    out = {}
    for g in GROUP_COLS:
        a = actual_df[g].to_numpy(dtype=float)
        p = pred_df[g].to_numpy(dtype=float)
        cap = CAPACITY_KWH[g]
        v = a >= cap * 0.10
        a, p = a[v], p[v]
        er = np.abs(p - a) / cap
        price = np.select([er <= 0.06, er <= 0.08], [4.0, 3.0], default=0.0)
        out[g] = 0.5 * (1 - er.mean()) + 0.5 * (a * price).sum() / (a * 4.0).sum()
    return out


_info = FOLD_INFO["A안(2024)"]
_pred = pd.DataFrame({g: PRED_CACHE[("A안(2024)", "dyn_w_q60", g)] for g in GROUP_COLS},
                     index=_info["valid_idx"])
_gs = group_scores(_info["actual_df"], _pred)
_total = score_predictions(_info["actual_df"], _pred)[0]
print("그룹별 점수:", {k: round(v, 4) for k, v in _gs.items()})
print(f"평균 {np.mean(list(_gs.values())):.6f}  vs  공식 총점 {_total:.6f}  →",
      "✓ 항등식 성립" if abs(np.mean(list(_gs.values())) - _total) < 1e-9 else "⚠️ 불일치")


# ── (1) 15-2에서 가장 좋았던 축소 수준을 자동 선택 ────────────────
_red = summarize(stage6)
BEST_KEEP_LABEL = _red.index[0]
BEST_TOPN = {"F_gain_pos": None, "F_top200": 200, "F_top100": 100, "F_top50": 50}[BEST_KEEP_LABEL]
print(f"\n선택된 피처 축소 수준: {BEST_KEEP_LABEL} (B안 평균 {_red.loc[BEST_KEEP_LABEL, 'B안 평균']:.4f})")
print("※ 점수가 비슷하면 더 적은 쪽이 낫습니다. 위 표를 보고 직접 바꾸려면 BEST_TOPN을 수정하세요.")


# ── (2) τ 스윕 ───────────────────────────────────────────────────
TAUS_G = [0.50, 0.55, 0.60, 0.65, 0.70]
stage7 = []
for tau in TAUS_G:
    name = f"T_q{int(tau*100)}"
    print(f"\n=== {name} ===")
    stage7.append(run_variant(name, lgbm3_fit_fn(alpha=tau, keep_cols_fn=make_keep_fn(BEST_TOPN))))

# ── (3) 그룹별 점수표 ────────────────────────────────────────────
rows = []
for tau in TAUS_G:
    name = f"T_q{int(tau*100)}"
    for fold_name, info in FOLD_INFO.items():
        pred = pd.DataFrame({g: PRED_CACHE[(fold_name, name, g)] for g in GROUP_COLS},
                            index=info["valid_idx"])
        for g, sc in group_scores(info["actual_df"], pred).items():
            rows.append({"tau": tau, "fold": fold_name, "group": g, "score": sc})
gs_df = pd.DataFrame(rows)

print("\n=== 그룹별 τ 점수: B안 3-fold 평균 ===")
b_tbl = gs_df[gs_df["fold"].isin(B_FOLDS)].pivot_table(index="tau", columns="group", values="score")
display(b_tbl.round(4))
print("=== 그룹별 τ 점수: A안 (교차 확인용) ===")
a_tbl = gs_df[gs_df["fold"] == "A안(2024)"].pivot_table(index="tau", columns="group", values="score")
display(a_tbl.round(4))

BEST_TAU = {g: float(b_tbl[g].idxmax()) for g in GROUP_COLS}
print("\nB안 기준 그룹별 최적 τ:", BEST_TAU)
print("A안 기준 그룹별 최적 τ:", {g: float(a_tbl[g].idxmax()) for g in GROUP_COLS})
print("→ 두 기준이 다르면 곡선이 완만하다는 뜻이므로, 위 두 표를 보고 가운데 값으로 직접 정하세요(BEST_TAU 수정).")

그룹별 점수: {'kpx_group_1': np.float64(0.6297), 'kpx_group_2': np.float64(0.6579), 'kpx_group_3': np.float64(0.5879)}
평균 0.625175  vs  공식 총점 0.625175  → ✓ 항등식 성립

선택된 피처 축소 수준: F_top200 (B안 평균 0.6153)
※ 점수가 비슷하면 더 적은 쪽이 낫습니다. 위 표를 보고 직접 바꾸려면 BEST_TOPN을 수정하세요.

=== T_q50 ===
  [T_q50] A안(2024): score=0.6182  (1-NMAE=0.8690, FICR=0.3674)
  [T_q50] B안 fold1: score=0.5836  (1-NMAE=0.8505, FICR=0.3167)
  [T_q50] B안 fold2: score=0.6109  (1-NMAE=0.8634, FICR=0.3584)
  [T_q50] B안 fold3: score=0.6397  (1-NMAE=0.8769, FICR=0.4024)

=== T_q55 ===
  [T_q55] A안(2024): score=0.6204  (1-NMAE=0.8662, FICR=0.3747)
  [T_q55] B안 fold1: score=0.5881  (1-NMAE=0.8498, FICR=0.3265)
  [T_q55] B안 fold2: score=0.6089  (1-NMAE=0.8592, FICR=0.3587)
  [T_q55] B안 fold3: score=0.6390  (1-NMAE=0.8741, FICR=0.4038)

=== T_q60 ===
  [T_q60] A안(2024): score=0.6242  (1-NMAE=0.8618, FICR=0.3865)
  [T_q60] B안 fold1: score=0.5900  (1-NMAE=0.8464, FICR=0.3336)
  [T_q60] B안 fold2: score=0.6111  (1-NMAE=0.8534, FICR=0.3688)
  [T_q

group,kpx_group_1,kpx_group_2,kpx_group_3
tau,,,
0.50,0.6147,0.6495,0.5699
0.55,0.6212,0.6451,0.5697
0.60,0.6272,0.6492,0.5694
0.65,0.6245,0.6426,0.5716
0.70,0.6258,0.6364,0.5612


=== 그룹별 τ 점수: A안 (교차 확인용) ===


group,kpx_group_1,kpx_group_2,kpx_group_3
tau,,,
0.50,0.6145,0.6562,0.5839
0.55,0.6213,0.6579,0.5821
0.60,0.6318,0.6550,0.5858
0.65,0.6267,0.6477,0.5870
0.70,0.6281,0.6420,0.5725



B안 기준 그룹별 최적 τ: {'kpx_group_1': 0.6, 'kpx_group_2': 0.5, 'kpx_group_3': 0.65}
A안 기준 그룹별 최적 τ: {'kpx_group_1': 0.6, 'kpx_group_2': 0.55, 'kpx_group_3': 0.65}
→ 두 기준이 다르면 곡선이 완만하다는 뜻이므로, 위 두 표를 보고 가운데 값으로 직접 정하세요(BEST_TAU 수정).


In [56]:
# ── (4) 그룹별 τ를 적용한 최종 예측을 조립해 채점 ──────────────────
def assemble_group_tau(tau_map, label):
    rows = []
    for fold_name, info in FOLD_INFO.items():
        pred = {}
        for g in GROUP_COLS:
            src = f"T_q{int(tau_map[g]*100)}"
            p = PRED_CACHE[(fold_name, src, g)]
            PRED_CACHE[(fold_name, label, g)] = p       # 이후 앙상블에서 재사용
            pred[g] = p
        s, n, f = score_predictions(info["actual_df"], pd.DataFrame(pred, index=info["valid_idx"]))
        rows.append({"fold": fold_name, "model": label, "score": s, "1-NMAE": n, "FICR": f})
    return pd.DataFrame(rows)


stage8 = [assemble_group_tau(BEST_TAU, "G_grouptau")]
display(summarize(stage1 + stage2 + stage5 + stage6 + stage7 + stage8))
print("\n적용된 그룹별 τ:", BEST_TAU)

fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
G_grouptau,0.6250,0.5926,0.6129,0.6428,0.6161,0.0253
F_top200,0.6242,0.5900,0.6111,0.6447,0.6153,0.0276
T_q60,0.6242,0.5900,0.6111,0.6447,0.6153,0.0276
F_gain_pos,0.6248,0.5949,0.6122,0.6386,0.6152,0.0220
L_lead,0.6236,0.5946,0.6101,0.6396,0.6148,0.0229
F_top100,0.6249,0.5896,0.6132,0.6384,0.6137,0.0244
F_top50,0.6243,0.5923,0.6105,0.6363,0.6131,0.0221
T_q65,0.6205,0.5935,0.6047,0.6406,0.6129,0.0246
dyn_w_q60,0.6252,0.5926,0.6086,0.6361,0.6124,0.0220



적용된 그룹별 τ: {'kpx_group_1': 0.6, 'kpx_group_2': 0.5, 'kpx_group_3': 0.65}


**확인할 것**: `G_grouptau`가 단일 τ 최고(`T_q60` 등)보다 나은가? 그룹별로 τ를 나눈 이득이 실제로 있는지 봅니다. **차이가 0.002 이내면 "단일 τ로 충분"** 으로 보고 단순한 쪽을 고르세요 — 파라미터 3개를 검증 점수로 고르는 것 자체가 낙관 편향을 만들기 때문입니다.

## 15-4. 시드 앙상블 — σ를 줄이는 가장 싸고 확실한 방법

### 왜 이게 σ를 줄이는가

같은 데이터·같은 설정이라도 seed를 바꾸면 모델이 조금씩 달라집니다(트리 분할 시 동점 처리, 피처 샘플링 등). 이 차이에서 오는 오차를 **모델 분산**이라고 하는데, **n개를 평균 내면 분산이 1/n로 줄어듭니다**(표준편차는 1/√n).

> 비유: 같은 문제를 5명이 각자 풀어 평균을 내면, 한 명의 실수가 5분의 1로 희석됩니다. 단 **5명이 공통으로 가진 오해(편향)는 안 줄어듭니다** — 그래서 σ 전체가 아니라 "모델 분산 부분"만 줄어듭니다.

14절에서 예보 오차가 σ의 89%임을 확인했으므로, 시드 앙상블로 줄일 수 있는 건 나머지 11% 안쪽입니다. **큰 이득은 아니지만 확실하고, 리스크가 없습니다**(예측을 평균 내는 것뿐이라 나빠질 이유가 없습니다).

### 어떻게

15-3에서 정한 구성(피처 축소 + 그룹별 τ) 그대로, **seed만 5개**로 학습해 예측을 평균 냅니다. seed 42는 이미 있으므로 4개만 추가합니다.

⚠️ 48번 학습(4 seed × 4 fold × 3 그룹).

In [57]:
EXTRA_SEEDS = [7, 123, 2024, 31]

stage9 = []
for sd in EXTRA_SEEDS:
    label = f"S_seed{sd}"
    print(f"=== {label} ===")
    # 그룹별 τ를 그대로 적용하려면 그룹마다 alpha가 달라야 하므로 fit_fn을 직접 만든다
    def fit_fn(g, cv_suffix, train_mask, valid_idx, _sd=sd):
        X = build_group_feature_frame_v3(train, g, cv_suffix, dynamics=True, lead_feat=True)
        X = X[make_keep_fn(BEST_TOPN)(g, cv_suffix, train_mask, X)]
        m = _lgbm_train(X, g, train_mask, "quantile", BEST_TAU[g], "actual", _sd)
        p = pd.Series(m.predict(X.loc[valid_idx]), index=valid_idx)
        return p.clip(lower=0, upper=CAPACITY_KWH[g])
    stage9.append(run_variant(label, fit_fn))

# 5개 seed(기존 G_grouptau = seed 42 + 추가 4개) 평균
seed_members = ["G_grouptau"] + [f"S_seed{sd}" for sd in EXTRA_SEEDS]
stage10 = [score_blend({m: 1 for m in seed_members}, label="S_seedavg5")]

display(summarize(stage1 + stage2 + stage5 + stage6 + stage7 + stage8 + stage9 + stage10))

# seed별 변동폭 = 모델 분산의 크기
sd_scores = summarize(stage8 + stage9)["B안 평균"]
print(f"\nseed별 B안 평균: {sd_scores.round(4).to_dict()}")
print(f"seed 간 표준편차: {sd_scores.std():.4f}  ← 이 값이 크면 시드 앙상블 이득이 크다")

=== S_seed7 ===
  [S_seed7] A안(2024): score=0.6250  (1-NMAE=0.8628, FICR=0.3872)
  [S_seed7] B안 fold1: score=0.5926  (1-NMAE=0.8487, FICR=0.3365)
  [S_seed7] B안 fold2: score=0.6129  (1-NMAE=0.8550, FICR=0.3708)
  [S_seed7] B안 fold3: score=0.6428  (1-NMAE=0.8707, FICR=0.4149)
=== S_seed123 ===
  [S_seed123] A안(2024): score=0.6250  (1-NMAE=0.8628, FICR=0.3872)
  [S_seed123] B안 fold1: score=0.5926  (1-NMAE=0.8487, FICR=0.3365)
  [S_seed123] B안 fold2: score=0.6129  (1-NMAE=0.8550, FICR=0.3708)
  [S_seed123] B안 fold3: score=0.6428  (1-NMAE=0.8707, FICR=0.4149)
=== S_seed2024 ===
  [S_seed2024] A안(2024): score=0.6250  (1-NMAE=0.8628, FICR=0.3872)
  [S_seed2024] B안 fold1: score=0.5926  (1-NMAE=0.8487, FICR=0.3365)
  [S_seed2024] B안 fold2: score=0.6129  (1-NMAE=0.8550, FICR=0.3708)
  [S_seed2024] B안 fold3: score=0.6428  (1-NMAE=0.8707, FICR=0.4149)
=== S_seed31 ===
  [S_seed31] A안(2024): score=0.6250  (1-NMAE=0.8628, FICR=0.3872)
  [S_seed31] B안 fold1: score=0.5926  (1-NMAE=0.8487, FICR=0.3365

fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
G_grouptau,0.6250,0.5926,0.6129,0.6428,0.6161,0.0253
S_seed123,0.6250,0.5926,0.6129,0.6428,0.6161,0.0253
S_seed2024,0.6250,0.5926,0.6129,0.6428,0.6161,0.0253
S_seedavg5,0.6250,0.5926,0.6129,0.6428,0.6161,0.0253
S_seed7,0.6250,0.5926,0.6129,0.6428,0.6161,0.0253
S_seed31,0.6250,0.5926,0.6129,0.6428,0.6161,0.0253
F_top200,0.6242,0.5900,0.6111,0.6447,0.6153,0.0276
T_q60,0.6242,0.5900,0.6111,0.6447,0.6153,0.0276
F_gain_pos,0.6248,0.5949,0.6122,0.6386,0.6152,0.0220



seed별 B안 평균: {'G_grouptau': 0.6161, 'S_seed123': 0.6161, 'S_seed2024': 0.6161, 'S_seed31': 0.6161, 'S_seed7': 0.6161}
seed 간 표준편차: 0.0000  ← 이 값이 크면 시드 앙상블 이득이 크다


**확인할 것**

1. **`S_seedavg5`가 개별 seed 전부보다 높은가** — 평균이 개별보다 낮으면 뭔가 잘못된 것입니다
2. **seed 간 표준편차** — 이 값이 0.005 이상이면 모델 분산이 꽤 크다는 뜻이라 seed를 더 늘릴 가치가 있습니다. 0.002 이하면 이미 안정적이라 5개면 충분합니다
3. **`B안 최솟값`도 올랐는가** — 시드 앙상블의 진짜 가치는 **가장 나쁜 fold를 끌어올리는 것**(안정성)입니다

## 15-5. 정리 + 다음 (16절 예고)

### 이 절에서 확인할 최종 구성

`LightGBM(quantile, 그룹별 τ) + actual 표본가중 + 예보 lag/lead + 리드타임 + 피처 축소 + 시드 5개 평균`

여기에 13-5절의 **MLP 블렌드**를 다시 얹을지는, MLP도 같은 조건(축소·τ·가중)으로 맞춘 뒤 판단합니다(현재 블렌드 이득은 편향 상쇄에서 왔을 수 있음 — 13절 주의사항).

### 그러나 진짜 승부는 16절이다

14절이 알려준 것을 다시 강조합니다.

- **예보 오차가 σ의 89%**
- **머신러닝 없이 파워커브만 써도 바람만 맞으면 0.83** (우리는 850피처 + LightGBM으로 0.61)
- **바람만 알면 모델링은 이미 바닥에 닿는다**(오라클 σ ≈ 예측 불가 바닥)

그런데 현재 풍속 추정 `ws_est`는 **예보 풍속 6개 컬럼에 Ridge 선형회귀**를 돌린 것뿐입니다(`03_features.ipynb` 3절). 우리에겐 850개 피처가 있는데 풍속 추정에는 6개만 쓰고 있습니다.

**16절 과제: SCADA 실측 풍속을 타깃으로 제대로 된 풍속 모델을 학습한다.**

- 입력: 850개 피처 전부(모든 격자·높이·시간 파생 포함)
- 타깃: `scada_ws_{group}` (train에만 있음)
- 모델: LightGBM / MLP / (나중에) 격자 CNN
- **누수 아님**: SCADA는 학습 데이터에 존재하고, 학습된 모델은 test의 **예보**에 적용된다. 지금 Ridge로 하고 있는 일과 구조가 완전히 같고 모델만 바뀐다
- **fold-safe 필수**: `03_features.ipynb` 3-5절의 `CV_CUTOFFS` 패턴을 그대로 따라 fold별로 따로 학습해야 한다
- 평가는 두 단계로: ① 추정 풍속 자체가 SCADA에 얼마나 가까워졌나(상관·RMSE) ② 그 풍속으로 발전량 모델을 돌렸을 때 점수가 올랐나

현재 `ws_est`의 SCADA 상관은 0.854 / 0.856 / 0.870입니다. 이걸 0.90 이상으로 올릴 수 있다면 σ가 직접 줄어들고, 14절이 보여준 0.86이라는 천장에 한 걸음 다가갑니다.

## 15-6. 시드 앙상블 재시도 — 무작위성을 먼저 켜야 했다

15-4에서 **seed 5개가 소수점 넷째 자리까지 완전히 같은 점수**(표준편차 0.0000)로 나왔습니다. 버그가 아니라 **설계 실수**입니다.

LightGBM 기본값은 `subsample=1.0`(행 전부 사용), `colsample_bytree=1.0`(피처 전부 사용), `subsample_freq=0`(행 샘플링 끔)이라 **무작위 요소가 하나도 없습니다.** 알고리즘이 완전히 결정적이므로 seed를 바꿔도 똑같은 트리가 나옵니다.

> 비유: 주사위를 안 굴리는 게임에서 "주사위 색깔"만 바꾼 셈입니다. 결과가 같을 수밖에 없습니다.

**고치려면 무작위성을 먼저 넣어야 합니다.**

| 파라미터 | 값 | 뜻 |
|---|---|---|
| `subsample` | 0.8 | 트리마다 학습 행의 80%만 무작위로 사용 |
| `subsample_freq` | 1 | 매 트리마다 행을 다시 뽑음 (0이면 샘플링 자체가 꺼짐 — **이걸 빠뜨리면 subsample이 무시된다**) |
| `colsample_bytree` | 0.8 | 트리마다 피처의 80%만 무작위로 사용 |

이 자체가 **정규화(regularization)** 효과도 있어서 점수가 바뀔 수 있습니다. 그래서 **먼저 무작위성만 켠 단일 모델(`R_rand_seed42`)** 을 기준선으로 두고, 그다음 seed 5개 평균과 비교합니다. 그래야 "무작위성 도입 효과"와 "앙상블 효과"를 분리할 수 있습니다.

15-2에서 확인한 대로 **피처를 top50까지 줄여도 점수가 같으므로**, 여기서는 속도를 위해 `BEST_TOPN`을 그대로 쓰되 τ는 단일 0.60을 씁니다(15-3에서 그룹별 τ 이득이 노이즈 수준으로 판명).

⚠️ 60번 학습(5 seed × 4 fold × 3 그룹). 축소된 피처라 빠릅니다.

In [62]:
RAND_PARAMS = dict(subsample=0.8, subsample_freq=1, colsample_bytree=0.8)


def _lgbm_train_rand(X_full, g, train_mask, objective, alpha, weight_mode, seed, n_estimators=2000):
    """_lgbm_train과 같되 행/피처 샘플링을 켜서 seed가 실제로 영향을 주게 한다."""
    fit_idx = train_mask & train[g].notna()
    cutoff_es = train.loc[fit_idx, "kst_dtm"].quantile(0.9)
    tr = fit_idx & (train["kst_dtm"] <= cutoff_es)
    es = fit_idx & (train["kst_dtm"] > cutoff_es)
    Xtr, ytr = X_full.loc[tr], train.loc[tr, g]
    Xes, yes = X_full.loc[es], train.loc[es, g]
    params = dict(objective=objective, random_state=seed, n_estimators=n_estimators,
                  verbosity=-1, importance_type="gain", **RAND_PARAMS)
    if alpha is not None:
        params["alpha"] = alpha
    w_tr, w_es = make_sample_weight(ytr, g, weight_mode), make_sample_weight(yes, g, weight_mode)
    m = lgb.LGBMRegressor(**params)
    m.fit(Xtr, ytr, sample_weight=w_tr, eval_set=[(Xes, yes)],
          eval_sample_weight=[w_es] if w_es is not None else None,
          eval_metric="l1", callbacks=[lgb.early_stopping(50, verbose=False)])
    return m


def rand_fit_fn(seed):
    def fit_fn(g, cv_suffix, train_mask, valid_idx):
        X = build_group_feature_frame_v3(train, g, cv_suffix, dynamics=True, lead_feat=True)
        X = X[make_keep_fn(BEST_TOPN)(g, cv_suffix, train_mask, X)]
        m = _lgbm_train_rand(X, g, train_mask, "quantile", 0.60, "actual", seed)
        p = pd.Series(m.predict(X.loc[valid_idx]), index=valid_idx)
        return p.clip(lower=0, upper=CAPACITY_KWH[g])
    return fit_fn


ALL_SEEDS = [SEED, 7, 123, 2024, 31]
stage11 = []
for sd in ALL_SEEDS:
    label = f"R_rand_seed{sd}"
    print(f"=== {label} ===")
    stage11.append(run_variant(label, rand_fit_fn(sd)))

stage12 = [score_blend({f"R_rand_seed{sd}": 1 for sd in ALL_SEEDS}, label="R_randavg5")]

rand_tbl = summarize(stage11 + stage12)
display(rand_tbl)
sd_only = summarize(stage11)["B안 평균"]
print(f"\nseed별 B안 평균: {sd_only.round(4).to_dict()}")
print(f"seed 간 표준편차: {sd_only.std():.4f}   ← 0이 아니어야 정상 (15-4에서는 0이었음)")
print(f"기준(무작위성 없음, T_q60): B안 평균 0.6153")

=== R_rand_seed42 ===
  [R_rand_seed42] A안(2024): score=0.6236  (1-NMAE=0.8624, FICR=0.3848)
  [R_rand_seed42] B안 fold1: score=0.5896  (1-NMAE=0.8473, FICR=0.3319)
  [R_rand_seed42] B안 fold2: score=0.6097  (1-NMAE=0.8550, FICR=0.3644)
  [R_rand_seed42] B안 fold3: score=0.6400  (1-NMAE=0.8716, FICR=0.4084)
=== R_rand_seed7 ===
  [R_rand_seed7] A안(2024): score=0.6246  (1-NMAE=0.8623, FICR=0.3870)
  [R_rand_seed7] B안 fold1: score=0.5942  (1-NMAE=0.8495, FICR=0.3390)
  [R_rand_seed7] B안 fold2: score=0.6138  (1-NMAE=0.8541, FICR=0.3735)
  [R_rand_seed7] B안 fold3: score=0.6406  (1-NMAE=0.8720, FICR=0.4092)
=== R_rand_seed123 ===
  [R_rand_seed123] A안(2024): score=0.6251  (1-NMAE=0.8620, FICR=0.3882)
  [R_rand_seed123] B안 fold1: score=0.5950  (1-NMAE=0.8476, FICR=0.3424)
  [R_rand_seed123] B안 fold2: score=0.6128  (1-NMAE=0.8540, FICR=0.3716)
  [R_rand_seed123] B안 fold3: score=0.6429  (1-NMAE=0.8722, FICR=0.4136)
=== R_rand_seed2024 ===
  [R_rand_seed2024] A안(2024): score=0.6245  (1-NMAE=0.8620

fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
R_randavg5,0.6259,0.5955,0.6131,0.6423,0.6170,0.0237
R_rand_seed123,0.6251,0.5950,0.6128,0.6429,0.6169,0.0242
R_rand_seed7,0.6246,0.5942,0.6138,0.6406,0.6162,0.0233
R_rand_seed2024,0.6245,0.5970,0.6124,0.6383,0.6159,0.0208
R_rand_seed31,0.6238,0.5950,0.6086,0.6399,0.6145,0.0230
R_rand_seed42,0.6236,0.5896,0.6097,0.6400,0.6131,0.0254



seed별 B안 평균: {'R_rand_seed123': 0.6169, 'R_rand_seed7': 0.6162, 'R_rand_seed2024': 0.6159, 'R_rand_seed31': 0.6145, 'R_rand_seed42': 0.6131}
seed 간 표준편차: 0.0015   ← 0이 아니어야 정상 (15-4에서는 0이었음)
기준(무작위성 없음, T_q60): B안 평균 0.6153


**확인할 것 (두 가지를 분리해서)**

1. **무작위성 도입 효과** — `R_rand_seed42` vs `T_q60`(0.6153). 정규화 효과로 오를 수도, 정보 손실로 내릴 수도 있습니다
2. **앙상블 효과** — `R_randavg5` vs 개별 seed 최고. **이번엔 seed 간 표준편차가 0이 아니어야** 정상입니다. 표준편차가 클수록 앙상블 이득도 큽니다

`R_randavg5`가 `T_q60`을 이기면 채택하고, 아니면 무작위성 없는 결정적 모델을 그대로 씁니다(그 경우 시드 앙상블은 이 문제에서 쓸 수 없다는 결론).

---

# 16. 풍속 추정 모델 업그레이드 — 진짜 승부처

## 왜 이게 핵심인가

14절이 보여준 것을 다시 놓고 봅니다.

| | 점수(B평균) | 잔차 σ |
|---|---|---|
| 우리 현재 최고 | 0.6161 | 0.173 |
| **바람만 정확하면, ML 없이 파워커브만** | **0.8254** | 0.080 |
| **바람만 정확하면, LightGBM까지** | **0.8618** | 0.079 |

**예보 오차가 전체 σ의 89%.** 그리고 오라클의 σ는 이미 "예측 불가 바닥"에 닿아 있어서, 발전량 모델 쪽에서 짜낼 여지는 사실상 없습니다(15절이 +0.0037에 그친 것이 그 증거입니다).

**그런데 우리 풍속 추정은 이렇게 만들어져 있습니다.**

```
ws_est = Ridge(절편 + LDAPS최근접 + LDAPS16격자평균 + GFS10m + GFS80m + GFS100m + GFS850hPa)
         ← 입력이 6개뿐인 선형회귀
```

850개 피처가 있는데 **풍속 추정에는 6개만, 그것도 선형으로** 쓰고 있습니다. 발전량 예측에는 LightGBM을 쓰면서, 정작 성능의 89%를 좌우하는 풍속 추정은 선형회귀에 맡겨둔 셈입니다.

## 무엇을 하는가

**SCADA 실측 풍속을 타깃으로, 850개 예보 피처 전부를 써서 LightGBM을 학습합니다.**

```
[지금]  예보 6개 ──Ridge──→ ws_est ──파워커브──→ 발전량 모델 ──→ 예측
[16절]  예보 850개 ──LightGBM──→ ws_gbdt ──파워커브──→ 발전량 모델 ──→ 예측
                    ↑ 여기만 바뀜
```

## 이게 누수가 아닌 이유 (`leakage-guard` 점검)

"SCADA를 타깃으로 쓴다"고 하면 반사적으로 누수를 의심하게 되는데, **지금 `ws_est`가 하고 있는 일과 구조가 완전히 같습니다.**

- SCADA는 **train 기간에만** 존재합니다. 우리는 그걸로 **"예보 → 실제 풍속" 변환 규칙**을 배우는 것이지, 예측할 때 SCADA를 보는 게 아닙니다
- 학습된 모델은 test(2025)의 **예보에** 적용됩니다. test에 SCADA가 없어도 아무 문제 없습니다
- 14절 오라클과의 차이가 여기 있습니다. **오라클은 예측 시점에 SCADA 값 자체를 봤습니다(불가능)**. 16절은 SCADA로 학습만 하고 예측은 예보로 합니다(가능)
- **fold-safe 필수**: 각 fold의 학습 구간에서만 풍속 모델을 적합합니다. `03_features.ipynb` 3-5절의 `CV_CUTOFFS` 패턴과 동일합니다. 이걸 안 지키면 검증 구간의 SCADA를 미리 본 셈이 됩니다
- **입력에서 라벨 의존 피처를 전부 제거**합니다. `ws_est`·`power_curve_est`와 그 파생(regime/high_wind/icing)은 이미 SCADA·라벨로 만들어진 것이라 입력에 넣으면 순환이 됩니다

## 평가는 두 단계로

| 단계 | 무엇을 보나 | 기준 |
|---|---|---|
| **1단계** | 추정 풍속이 SCADA에 얼마나 가까워졌나 (상관·RMSE·MAE) | 현재 `ws_est` 상관 0.854 / 0.856 / 0.870 |
| **2단계** | 그 풍속으로 발전량 모델을 돌렸을 때 점수가 올랐나 | 현재 최고 B평균 0.6153 (`T_q60`) |

**1단계가 좋아졌는데 2단계가 안 좋아지는 경우**도 있을 수 있습니다(풍속 상관이 올라도 파워커브가 민감한 구간에서 나빠졌다면). 그래서 둘 다 봅니다.

## 16-1. 풍속 모델 학습 — 입력에서 라벨 의존 피처를 전부 걷어낸다

**입력 구성**
- `COMMON_RAW_COLS`(804개) — GFS/LDAPS 격자 예보 전부 + 시간 sin/cos
- 그룹 전용 피처 중 **예보에서만 나온 것**: `{g}_ws10_nearest`, `{g}_wd_sin/cos`, `{g}_air_density`, `{g}_gfs_ldaps_diff/absdiff`, `{g}_gust_proxy` 등
- 예보 풍속 3종(`{g}_ws10_nearest`, `ldaps_ws10_avg16`, `gfs_g5_ws_100m`)의 **lag/lead 파생 39개** — 13절에서 만든 `add_forecast_dynamics` 재사용
- 리드타임 피처 2개

**입력에서 반드시 뺄 것** (`WIND_EXCLUDE_MARKERS`)
`ws_est` / `power_curve_est` / `regime_` / `high_wind_caution` / `icing_risk`.
앞의 둘은 SCADA·라벨로 만든 것이고, 뒤의 셋은 그 `ws_est`에서 파생된 것이라 넣으면 **자기 자신을 입력으로 쓰는 순환**이 됩니다.

**학습 데이터가 오히려 늘어납니다**: 발전량 모델은 라벨(`kpx_group_*`)이 있어야 학습할 수 있지만, 풍속 모델은 **SCADA만 있으면** 됩니다. curtailment로 라벨을 NaN 처리한 구간도 SCADA는 살아 있으므로 풍속 학습에는 쓸 수 있습니다.

**손실함수**: 기본은 `l2`(RMSE)입니다. Ridge가 최소화하던 것과 같은 기준이라 **직접 비교가 공정**합니다. `l1`(MAE) 버전도 함께 돌려 비교합니다 — 풍속 분포에 이상치가 있다면 MAE가 나을 수 있습니다.

⚠️ 24번 학습(2 손실 × 4 fold × 3 그룹). 피처가 많아 발전량 모델보다 느릴 수 있습니다.

In [58]:
WIND_EXCLUDE_MARKERS = ("ws_est", "power_curve_est", "regime_", "high_wind_caution", "icing_risk")
WIND_PRED_CACHE = {}   # (g, cv_suffix, objective) -> 전체 기간 추정 풍속 Series


def build_wind_input_frame(df, g):
    """풍속 모델의 입력. 라벨/SCADA에서 파생된 컬럼은 전부 제외한다."""
    base = [c for c in GROUP_SPECIFIC_COLS[g] if not any(m in c for m in WIND_EXCLUDE_MARKERS)]
    dyn_src = [c for c in [f"{g}_ws10_nearest", "ldaps_ws10_avg16", "gfs_g5_ws_100m"] if c in df.columns]
    out = pd.concat([df[COMMON_RAW_COLS + base],
                     add_forecast_dynamics(df, dyn_src),
                     add_lead_features(df)], axis=1)
    bad = [c for c in out.columns if any(m in c for m in ("ws_est", "power_curve_est"))]
    assert not bad, f"풍속 모델 입력에 라벨 의존 컬럼 잔존: {bad[:5]}"
    assert out.isna().sum().sum() == 0, "풍속 모델 입력에 결측"
    return out


def get_wind_estimate(g, cv_suffix, train_mask, objective="l2", n_estimators=3000):
    """그 fold의 학습 구간에서만 SCADA 실측 풍속을 타깃으로 학습하고,
    전체 기간에 대한 추정 풍속을 돌려준다(fold-safe)."""
    key = (g, cv_suffix, objective)
    if key in WIND_PRED_CACHE:
        return WIND_PRED_CACHE[key]

    X = build_wind_input_frame(train, g)
    y = train[f"scada_ws_{g}"]
    fit_idx = train_mask & y.notna()          # 라벨(발전량)이 없어도 SCADA만 있으면 학습에 쓸 수 있다
    cutoff = train.loc[fit_idx, "kst_dtm"].quantile(0.9)
    tr = fit_idx & (train["kst_dtm"] <= cutoff)
    es = fit_idx & (train["kst_dtm"] > cutoff)

    m = lgb.LGBMRegressor(objective=objective, random_state=SEED, n_estimators=n_estimators,
                          verbosity=-1, importance_type="gain")
    m.fit(X.loc[tr], y[tr], eval_set=[(X.loc[es], y[es])], eval_metric="rmse",
          callbacks=[lgb.early_stopping(100, verbose=False)])

    pred = pd.Series(m.predict(X), index=train.index).clip(lower=0.0)
    WIND_PRED_CACHE[key] = pred
    return pred


# 학습 실행 (fold x 그룹 x 손실)
for objective in ["l2", "l1"]:
    for fold_name, info in FOLD_INFO.items():
        for g in GROUP_COLS:
            get_wind_estimate(g, info["cv_suffix"], info["train_mask"], objective=objective)
        print(f"  [{objective}] {fold_name} 완료")

print("\n풍속 모델 캐시:", len(WIND_PRED_CACHE), "개")
print("입력 피처 수:", {g: build_wind_input_frame(train, g).shape[1] for g in GROUP_COLS})

  [l2] A안(2024) 완료
  [l2] B안 fold1 완료
  [l2] B안 fold2 완료
  [l2] B안 fold3 완료
  [l1] A안(2024) 완료
  [l1] B안 fold1 완료
  [l1] B안 fold2 완료
  [l1] B안 fold3 완료

풍속 모델 캐시: 18 개
입력 피처 수: {'kpx_group_1': 853, 'kpx_group_2': 853, 'kpx_group_3': 853}


## 16-2. 1단계 평가 — 추정 풍속이 실제로 정확해졌는가

각 fold의 **검증 구간에서만** (학습에 안 쓴 기간) SCADA 실측 풍속과 비교합니다.

- **상관**: 현재 `ws_est`는 0.854 / 0.856 / 0.870. **0.90 넘으면 큰 성공**
- **RMSE / MAE**: 낮을수록 좋음. `02_eda`에서 확인한 LDAPS-SCADA RMSE 2~3 m/s가 원래 출발점이었습니다

In [59]:
rows = []
for fold_name, info in FOLD_INFO.items():
    vi, cs = info["valid_idx"], info["cv_suffix"]
    for g in GROUP_COLS:
        truth = train.loc[vi, f"scada_ws_{g}"]
        ok = truth.notna()
        t = truth[ok]
        cands = {"ws_est(Ridge, 현재)": train.loc[vi, f"{g}_ws_est_cv_{cs}"][ok],
                 "ws_gbdt_l2": get_wind_estimate(g, cs, info["train_mask"], "l2").loc[vi][ok],
                 "ws_gbdt_l1": get_wind_estimate(g, cs, info["train_mask"], "l1").loc[vi][ok]}
        for name, p in cands.items():
            rows.append({"fold": fold_name, "group": g, "추정법": name, "표본수": len(t),
                         "상관": t.corr(p), "RMSE": float(np.sqrt(((p - t) ** 2).mean())),
                         "MAE": float((p - t).abs().mean()), "편향": float((p - t).mean())})
ws_eval = pd.DataFrame(rows)

print("=== 그룹별 (4 fold 평균) ===")
display(ws_eval.groupby(["group", "추정법"])[["상관", "RMSE", "MAE", "편향"]].mean().round(4))
print("\n=== fold별 상관 ===")
display(ws_eval.pivot_table(index=["fold"], columns="추정법", values="상관").round(4))
print("\n=== fold x 그룹 상관 (자세히) ===")
display(ws_eval.pivot_table(index=["fold", "group"], columns="추정법", values="상관").round(4))

=== 그룹별 (4 fold 평균) ===


상관    RMSE     MAE      편향
group       추정법                                              
kpx_group_1 ws_est(Ridge, 현재)  0.8734  1.6581  1.2956 -0.1669
            ws_gbdt_l1         0.9109  1.4104  1.0678 -0.1749
            ws_gbdt_l2         0.9096  1.4161  1.0734 -0.1633
kpx_group_2 ws_est(Ridge, 현재)  0.8788  1.8279  1.4128 -0.0618
            ws_gbdt_l1         0.9172  1.5303  1.1530 -0.1043
            ws_gbdt_l2         0.9159  1.5404  1.1647 -0.0884
kpx_group_3 ws_est(Ridge, 현재)  0.8871  1.6749  1.2809 -0.0794
            ws_gbdt_l1         0.9092  1.5207  1.1228 -0.0369
            ws_gbdt_l2         0.9073  1.5299  1.1329  0.0643


=== fold별 상관 ===


추정법,"ws_est(Ridge, 현재)",ws_gbdt_l1,ws_gbdt_l2
fold,,,
A안(2024),0.8830,0.9153,0.9136
B안 fold1,0.8743,0.9019,0.9016
B안 fold2,0.8626,0.9089,0.9054
B안 fold3,0.8992,0.9234,0.9232



=== fold x 그룹 상관 (자세히) ===


추정법                   ws_est(Ridge, 현재)  ws_gbdt_l1  ws_gbdt_l2
fold     group                                                 
A안(2024) kpx_group_1             0.8772      0.9122      0.9114
         kpx_group_2             0.8826      0.9201      0.9170
         kpx_group_3             0.8891      0.9136      0.9124
B안 fold1 kpx_group_1             0.8659      0.9024      0.9020
         kpx_group_2             0.8714      0.9078      0.9105
         kpx_group_3             0.8856      0.8956      0.8922
B안 fold2 kpx_group_1             0.8602      0.9089      0.9051
         kpx_group_2             0.8636      0.9144      0.9093
         kpx_group_3             0.8640      0.9035      0.9017
B안 fold3 kpx_group_1             0.8905      0.9199      0.9198
         kpx_group_2             0.8975      0.9262      0.9269
         kpx_group_3             0.9096      0.9241      0.9230

**확인할 것**

1. **`ws_gbdt_*`의 상관이 `ws_est`(Ridge)를 모든 fold·모든 그룹에서 이기는가** — 하나라도 지면 원인을 봐야 합니다(특히 B안 fold1은 학습 구간이 짧아 GBDT가 불리할 수 있습니다)
2. **상관이 0.90을 넘는가** — 넘으면 14절이 보여준 0.86이라는 천장에 실제로 다가갈 수 있습니다
3. **`l2` vs `l1`** — RMSE는 `l2`가, MAE는 `l1`이 유리한 게 정상입니다. **어느 쪽이 2단계(발전량 점수)에서 이기는지가 진짜 기준**입니다
4. **편향** — 0에서 멀면 추정 풍속이 체계적으로 높거나 낮다는 뜻입니다. 파워커브를 통과하면 이 편향이 증폭됩니다

## 16-3. 새 풍속으로 파워커브와 파생 피처를 다시 만든다

풍속을 바꿨으면 **그 풍속에서 나온 모든 피처도 다시 만들어야** 합니다. 이건 HANDOFF 주의사항 11번("파생의 파생까지 챙겨라")을 그대로 적용하는 것입니다.

새로 만들 것:
- `{g}_power_curve_new` — 새 풍속을 **학습 구간에서 적합한 파워커브**에 통과시킨 값 (fold-safe)
- 밀도보정·제곱·세제곱·구간더미·고풍속·결빙 8~9종 (`add_fold_safe_ws_features` 재사용)
- lag/lead 26종 (`add_forecast_dynamics` 재사용)

14절 `build_oracle_frame`과 구조가 같아서, **풍속만 갈아끼우는 범용 함수**로 다시 씁니다. 이렇게 하면 오라클·Ridge·GBDT 어느 풍속이든 같은 코드로 프레임을 만들 수 있어 비교가 공정합니다.

In [60]:
def build_frame_with_wind(df, g, cv_suffix, train_mask, ws_series, tag,
                          dynamics=True, lead_feat=True):
    """주어진 풍속 추정치로 풍속 계열 피처를 전부 새로 만든 프레임.
    (14절 build_oracle_frame의 일반화 — 풍속 출처만 인자로 받는다)"""
    all_cv = set(all_cv_cols_for_group(g))
    exclude = set(leaky_cols_for_group(g)) | all_cv        # 기존 풍속·파워커브 계열 전부 제거
    base_cols = [c for c in GROUP_SPECIFIC_COLS[g] if c not in exclude]

    ws_col, pc_col = f"{g}_ws_{tag}", f"{g}_power_curve_{tag}"
    tmp = pd.DataFrame({"kst_dtm": df["kst_dtm"], f"{g}_air_density": df[f"{g}_air_density"]},
                       index=df.index)
    tmp[ws_col] = ws_series.astype(float)
    if g in ICING_RISK_GROUPS:
        tcol = f"ldaps_g{GROUP_NEAREST_LDAPS[g]}_heightAboveGround_2_t"
        tmp[tcol] = df[tcol]

    # 파워커브는 학습 구간에서만 적합 (fold-safe)
    fit_idx = train_mask & df[g].notna()
    edges, vals = fit_power_curve_oracle(tmp.loc[fit_idx, ws_col].to_numpy(dtype=float),
                                         df.loc[fit_idx, g].to_numpy(dtype=float))
    tmp[pc_col] = apply_power_curve_oracle(tmp[ws_col].to_numpy(dtype=float), edges, vals)

    parts = [df[COMMON_RAW_COLS + base_cols], tmp[[ws_col, pc_col]],
             add_fold_safe_ws_features(tmp, g, ws_col)]
    if dynamics:
        parts.append(add_forecast_dynamics(tmp, [ws_col, pc_col]))
    if lead_feat:
        parts.append(add_lead_features(df))
    out = pd.concat(parts, axis=1)
    assert out.isna().sum().sum() == 0, "프레임에 결측"
    return out


# 검산: 기존(Ridge) 풍속으로 이 범용 함수를 쓰면 15절 프레임과 피처 수가 같아야 한다
_info = FOLD_INFO["A안(2024)"]
for g in GROUP_COLS:
    n_v3 = build_group_feature_frame_v3(train, g, _info["cv_suffix"]).shape[1]
    n_gen = build_frame_with_wind(train, g, _info["cv_suffix"], _info["train_mask"],
                                  train[f"{g}_ws_est_cv_{_info['cv_suffix']}"], "ridge").shape[1]
    print(f"{g}: 15절 방식 {n_v3}개 / 범용 함수(Ridge 풍속) {n_gen}개",
          "✓ 동일" if n_v3 == n_gen else "⚠️ 다름")

kpx_group_1: 15절 방식 850개 / 범용 함수(Ridge 풍속) 850개 ✓ 동일
kpx_group_2: 15절 방식 850개 / 범용 함수(Ridge 풍속) 850개 ✓ 동일
kpx_group_3: 15절 방식 849개 / 범용 함수(Ridge 풍속) 849개 ✓ 동일


## 16-4. 2단계 평가 — 새 풍속으로 발전량 점수가 오르는가

**풍속 출처만 바꾸고 나머지는 전부 동일하게** 둡니다: 피처 축소(`BEST_TOPN`), 표본가중(actual), τ=0.60, lag/lead, 리드타임.

| 변형 | 풍속 출처 |
|---|---|
| `W_ridge` | `ws_est_cv_*` (현재, Ridge 6개 입력) — 범용 함수로 다시 돌린 대조군 |
| `W_gbdt_l2` | 16-1의 LightGBM(l2) 추정 풍속 |
| `W_gbdt_l1` | 16-1의 LightGBM(l1) 추정 풍속 |
| `W_blend` | Ridge와 GBDT(더 좋은 쪽) 풍속의 평균 — 두 추정의 오차가 다르면 이득 |

`W_ridge`를 다시 돌리는 이유는, 범용 함수가 기존 경로와 미세하게 다를 수 있어서 **같은 코드 경로에서 비교**하기 위해서입니다(`T_q60` 0.6153과 거의 같아야 정상).

⚠️ 48번 학습(4변형 × 4fold × 3그룹). 피처 축소가 적용돼 빠릅니다.

In [63]:
def wind_source(kind):
    """kind에 따라 (g, cv_suffix, train_mask) -> 풍속 Series 를 돌려준다."""
    def f(g, cv_suffix, train_mask):
        if kind == "ridge":
            return train[f"{g}_ws_est_cv_{cv_suffix}"]
        if kind in ("l2", "l1"):
            return get_wind_estimate(g, cv_suffix, train_mask, kind)
        if kind == "blend":
            return 0.5 * train[f"{g}_ws_est_cv_{cv_suffix}"] + \
                   0.5 * get_wind_estimate(g, cv_suffix, train_mask, BEST_WIND_OBJ)
        raise ValueError(kind)
    return f


def wind_fit_fn(kind, tag, top_n=None, tau=0.60):
    src = wind_source(kind)
    def fit_fn(g, cv_suffix, train_mask, valid_idx):
        X = build_frame_with_wind(train, g, cv_suffix, train_mask, src(g, cv_suffix, train_mask), tag)
        # 15-2와 동일한 2단계 피처 축소 (풍속 출처가 바뀌면 중요도도 달라지므로 tag별로 따로 계산)
        imp_key = (g, cv_suffix, tag)
        if imp_key not in IMP_CACHE:
            m0 = _lgbm_train(X, g, train_mask, "quantile", tau, "actual", SEED)
            IMP_CACHE[imp_key] = pd.Series(m0.feature_importances_, index=m0.feature_name_, dtype=float)
        imp = IMP_CACHE[imp_key].reindex(X.columns).fillna(0.0).sort_values(ascending=False)
        keep = imp[imp > 0].index.tolist() if top_n is None else imp.head(top_n).index.tolist()
        X = X[keep]
        m = _lgbm_train(X, g, train_mask, "quantile", tau, "actual", SEED)
        p = pd.Series(m.predict(X.loc[valid_idx]), index=valid_idx)
        return p.clip(lower=0, upper=CAPACITY_KWH[g])
    return fit_fn


# 16-2 결과를 보고 더 좋았던 손실을 고르세요 (기본 l2)
BEST_WIND_OBJ = "l1"

stage13 = []
for label, kind, tag in [("W_ridge", "ridge", "ridge"),
                         ("W_gbdt_l2", "l2", "gbdtl2"),
                         ("W_gbdt_l1", "l1", "gbdtl1"),
                         ("W_blend", "blend", "wblend")]:
    print(f"=== {label} ===")
    stage13.append(run_variant(label, wind_fit_fn(kind, tag, top_n=BEST_TOPN)))

display(summarize(stage13 + stage7 + stage8))
print("\n참고 — 14절 오라클 상한: A안 0.8587 / B평균 0.8618")

=== W_ridge ===
  [W_ridge] A안(2024): score=0.6267  (1-NMAE=0.8631, FICR=0.3903)
  [W_ridge] B안 fold1: score=0.5906  (1-NMAE=0.8461, FICR=0.3350)
  [W_ridge] B안 fold2: score=0.6143  (1-NMAE=0.8553, FICR=0.3732)
  [W_ridge] B안 fold3: score=0.6353  (1-NMAE=0.8703, FICR=0.4004)
=== W_gbdt_l2 ===
  [W_gbdt_l2] A안(2024): score=0.6391  (1-NMAE=0.8736, FICR=0.4045)
  [W_gbdt_l2] B안 fold1: score=0.6061  (1-NMAE=0.8564, FICR=0.3558)
  [W_gbdt_l2] B안 fold2: score=0.6316  (1-NMAE=0.8690, FICR=0.3943)
  [W_gbdt_l2] B안 fold3: score=0.6517  (1-NMAE=0.8774, FICR=0.4261)
=== W_gbdt_l1 ===
  [W_gbdt_l1] A안(2024): score=0.6352  (1-NMAE=0.8705, FICR=0.3999)
  [W_gbdt_l1] B안 fold1: score=0.6052  (1-NMAE=0.8556, FICR=0.3548)
  [W_gbdt_l1] B안 fold2: score=0.6311  (1-NMAE=0.8668, FICR=0.3955)
  [W_gbdt_l1] B안 fold3: score=0.6396  (1-NMAE=0.8741, FICR=0.4050)
=== W_blend ===
  [W_blend] A안(2024): score=0.6335  (1-NMAE=0.8696, FICR=0.3975)
  [W_blend] B안 fold1: score=0.5991  (1-NMAE=0.8528, FICR=0.3454)
  [W_b

fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
W_gbdt_l2,0.6391,0.6061,0.6316,0.6517,0.6298,0.0229
W_gbdt_l1,0.6352,0.6052,0.6311,0.6396,0.6253,0.0179
W_blend,0.6335,0.5991,0.6277,0.6441,0.6236,0.0228
G_grouptau,0.6250,0.5926,0.6129,0.6428,0.6161,0.0253
T_q60,0.6242,0.5900,0.6111,0.6447,0.6153,0.0276
W_ridge,0.6267,0.5906,0.6143,0.6353,0.6134,0.0224
T_q65,0.6205,0.5935,0.6047,0.6406,0.6129,0.0246
T_q55,0.6204,0.5881,0.6089,0.6390,0.6120,0.0256
T_q50,0.6182,0.5836,0.6109,0.6397,0.6114,0.0281



참고 — 14절 오라클 상한: A안 0.8587 / B평균 0.8618


**확인할 것**

1. **`W_ridge`가 `T_q60`(0.6153)과 비슷한가** — 크게 다르면 범용 함수 경로에 문제가 있는 것이니 멈추고 알려주세요
2. **`W_gbdt_*`가 `W_ridge`를 A안·B안 3-fold 전부에서 이기는가**
3. **1단계(풍속 상관)와 2단계(발전량 점수)의 방향이 일치하는가** — 풍속은 좋아졌는데 점수가 안 오르면, 파워커브가 민감한 ramp 구간에서 오차가 커졌을 가능성이 있습니다. 그럴 땐 16-5의 구간별 진단을 보세요
4. **`W_blend`** — Ridge와 GBDT의 오차 방향이 다르면 평균이 둘 다를 이깁니다. 13절 MLP 블렌드와 같은 원리입니다

## 16-5. 진단 — σ가 실제로 줄었는가

이번 작업의 성패는 점수보다 **σ**로 판단하는 게 정확합니다. 14절에서 확인했듯 우리 문제의 병목은 오차의 폭이기 때문입니다.

- `T_q60`(Ridge 풍속): σ 0.173
- 오라클(완벽한 풍속): σ 0.079
- **`W_gbdt_*`가 이 사이 어디에 있는가**가 이번 개선의 크기입니다

함께 볼 것: 풍속 구간별 잔차(파워커브가 가파른 ramp 구간에서 오히려 나빠지지 않았는지)와 그룹별 편향입니다.

In [64]:
print("=== σ 비교 ===")
rows = []
for v in ["W_ridge", "W_gbdt_l2", "W_gbdt_l1", "W_blend", "oracle_ws"]:
    if (list(FOLD_INFO)[0], v, GROUP_COLS[0]) not in PRED_CACHE:
        continue
    e = eval_frame(v)
    rows.append({"변형": v, "σ": round(e["signed"].std(), 4), "편향": round(e["signed"].mean(), 4),
                 "밴드폭 ÷ σ": round(0.06 / e["signed"].std(), 3)})
display(pd.DataFrame(rows))

BEST_W = summarize(stage13).index[0]
print(f"\n이번 최고 변형: {BEST_W}")

print(f"\n=== 그룹별 오차 해부: {BEST_W} ===")
ef_w = eval_frame(BEST_W)
display(error_anatomy(ef_w, "group"))

print(f"\n=== 풍속 구간별 잔차: {BEST_W} ===")
ef_w["regime"] = pd.cut(ef_w["ws"], [-np.inf, 3.0, 12.0, 15.0, np.inf],
                        labels=["calm(<3)", "ramp(3~12)", "rated(12~15)", "high(>=15)"], right=False)
rows = []
for g in GROUP_COLS:
    for r in ["calm(<3)", "ramp(3~12)", "rated(12~15)", "high(>=15)"]:
        d = ef_w[(ef_w["group"] == g) & (ef_w["regime"] == r)]
        if len(d) == 0:
            continue
        rows.append({"group": g, "regime": r, "표본수": len(d),
                     "평균오차율": round(d["er"].mean(), 4), "편향": round(d["signed"].mean(), 4),
                     "≤6% 통과율": round(wmean(d["er"] <= 0.06, d["actual"].to_numpy()), 4)})
display(pd.DataFrame(rows))

=== σ 비교 ===


,변형,σ,편향,밴드폭 ÷ σ
0,W_ridge,0.1727,0.0526,0.347
1,W_gbdt_l2,0.1684,0.0283,0.356
2,W_gbdt_l1,0.1700,0.0329,0.353
3,W_blend,0.1684,0.0421,0.356
4,oracle_ws,0.0786,0.0133,0.763



이번 최고 변형: W_gbdt_l2

=== 그룹별 오차 해부: W_gbdt_l2 ===


,표본수,평균오차율,편향,잔차σ,≤6% 비중,6~8% 비중,>8% 비중,FICR 재현값
group,,,,,,,,
kpx_group_1,12608,0.1234,0.0154,0.1604,0.3225,0.1132,0.5644,0.4074
kpx_group_2,12553,0.1247,0.0435,0.1637,0.3904,0.1011,0.5085,0.4662
kpx_group_3,11331,0.1424,0.0259,0.1805,0.2528,0.0847,0.6625,0.3163



=== 풍속 구간별 잔차: W_gbdt_l2 ===


,group,regime,표본수,평균오차율,편향,≤6% 통과율
0,kpx_group_1,calm(<3),9,0.1055,-0.1055,0.3155
1,kpx_group_1,ramp(3~12),11344,0.1261,0.0187,0.3053
2,kpx_group_1,rated(12~15),1206,0.0957,-0.0181,0.4169
3,kpx_group_1,high(>=15),49,0.1790,0.0853,0.2591
4,kpx_group_2,calm(<3),54,0.0898,-0.0602,0.3476
5,kpx_group_2,ramp(3~12),10203,0.1305,0.0393,0.3297
6,kpx_group_2,rated(12~15),1897,0.0989,0.0633,0.5091
7,kpx_group_2,high(>=15),399,0.1033,0.0697,0.7128
8,kpx_group_3,calm(<3),125,0.1172,-0.1102,0.1223
9,kpx_group_3,ramp(3~12),10246,0.1418,0.0335,0.2618


**확인할 것**

1. **σ가 0.173에서 얼마나 내려갔는가** — 0.16이면 작은 개선, 0.14 이하면 큰 개선입니다. `밴드폭 ÷ σ`가 0.347에서 얼마나 올랐는지가 FICR 상방을 직접 결정합니다
2. **ramp(3~12) 구간이 나빠지지 않았는가** — 파워커브가 가장 가파른 구간이라 풍속 오차가 발전량 오차로 증폭됩니다. 여기가 나빠졌다면 풍속 상관이 올라도 점수는 안 오릅니다
3. **group_3의 rated 구간 과소예측(-0.051)이 완화됐는가**

### 이 절 다음에

- **효과가 크면**: 같은 접근을 MLP·격자 CNN으로 확장합니다(풍속 추정에 딥러닝을 쓰는 것 — 갈래 A2/A3의 진짜 자리). 그리고 `03_features.ipynb`로 정식 이관합니다
- **효과가 작으면**: 예보에서 짜낼 수 있는 풍속 정보가 이미 한계라는 뜻이므로, **분포 예측 + 기대 FICR 최대화**(갈래 B1)로 방향을 틉니다
- 어느 쪽이든 **E1(test 경로 검증)** 은 곧 해야 합니다. 13·15·16절의 파생을 test에서 한 번도 안 돌려봤습니다

---

# 17. 풍속 모델 심화 + 제출 경로 리허설

## 16절 결과 — 진단이 옳았다

| 변형 | 풍속 출처 | A안 | B평균 | 잔차 σ | 편향 |
|---|---|---|---|---|---|
| `W_ridge` | Ridge(예보 6개) | 0.6267 | 0.6134 | 0.1727 | +0.0526 |
| **`W_gbdt_l2`** | **LightGBM(예보 850개)** | **0.6391** | **0.6298** | **0.1684** | **+0.0283** |
| `W_gbdt_l1` | LightGBM(l1) | 0.6352 | 0.6253 | 0.1700 | +0.0329 |
| `W_blend` | Ridge+GBDT 평균 | 0.6335 | 0.6236 | 0.1684 | +0.0421 |

**풍속 모델 하나 바꿔서 B평균 +0.0164, A안 +0.0124. 4개 fold 전부 개선.** 15절 전체(+0.0037)의 네 배가 넘습니다. 14절 오라클 진단이 가리킨 방향이 맞았습니다.

### 눈여겨볼 세 가지

**1. 1단계와 2단계의 승자가 다르다.** 풍속 정확도는 `l1`이 이겼는데(상관 0.912 vs 0.910, RMSE·MAE도 전부), **발전량 점수는 `l2`가 이겼습니다**(0.6298 vs 0.6253). 16-0절에서 미리 경고했던 경우입니다.
> 왜 그럴까: `l2`는 **큰 풍속 오차에 더 큰 벌점**을 줍니다. 파워커브가 가파른 ramp 구간(3~12 m/s)에서는 풍속을 1 m/s 틀리면 발전량이 크게 틀리므로, "가끔 크게 틀리는 것"을 막는 `l2`가 발전량 관점에서 유리합니다. 평균적으로는 `l1`이 정확해도 말입니다. **최종 판단 기준은 항상 2단계(발전량 점수)입니다.**

**2. Ridge와 섞는 것은 손해다.** `W_blend`(0.6236)가 순수 GBDT(0.6298)보다 낮습니다. Ridge 풍속이 모든 면에서 열등하고 오차 방향도 GBDT와 비슷해서, 섞으면 나쁜 쪽으로 끌려갑니다. **13절 MLP 블렌드가 통했던 것과 대조적** — 그때는 두 모델의 오차 방향이 달랐습니다.

**3. σ는 조금 줄었는데 점수는 많이 올랐다.** σ 0.1727→0.1684(-2.5%)인데 점수는 +0.016입니다. 이유는 **편향**입니다: +0.0526 → **+0.0283**. 풍속이 정확해지니 발전량 예측이 저절로 중앙에 가까워졌고, 그만큼 FICR 밴드 안에 더 많이 들어왔습니다(≤6% 비중이 그룹별로 0.30→0.32, 0.36→0.39, 0.25→0.25).

## 이 절에서 할 것

| 절 | 내용 | 학습 |
|---|---|---|
| 17-1 | **풍속 앙상블** — l1+l2 평균(무료) + 시드 앙상블 | 36 |
| 17-2 | 2단계 평가 (발전량 점수) | 24 |
| 17-3 | **τ 재조정** — 편향이 +0.053 → +0.028로 바뀌었으니 최적 τ도 달라졌다 | 36 |
| 17-4 | **⭐ 제출 경로 전체 리허설 (E1)** — 한 번도 안 해봤다 | 6 |

## 17-1. 풍속 앙상블 — 풍속 자체를 여러 개 만들어 평균낸다

발전량 모델을 앙상블하는 것보다 **풍속 모델을 앙상블하는 것**이 더 효과적일 수 있습니다. 우리 오차의 89%가 풍속에서 오기 때문입니다. 파이프라인의 상류에서 오차를 줄이면 하류 전체가 좋아집니다.

세 가지 후보를 만듭니다.

| 후보 | 만드는 법 | 비용 |
|---|---|---|
| `avg12` | 이미 있는 `l1`·`l2` 예측의 평균 | **무료** |
| `seedens` | `l2` + 행/피처 샘플링을 켜고 seed 3개 → 평균 | 36번 학습 |
| `avgall` | 위 전부 평균 | 무료 |

15-6절에서 배운 대로, 시드 앙상블을 하려면 **`subsample`/`colsample_bytree`/`subsample_freq`로 무작위성을 먼저 켜야** 합니다.

1단계 평가(SCADA와의 상관·RMSE)를 먼저 봐서, **풍속 자체가 좋아졌는지**부터 확인합니다.

⚠️ 36번 학습(풍속 모델 3 seed × 4 fold × 3 그룹). 풍속 모델은 피처가 853개라 다소 느립니다.

In [77]:
def get_wind_estimate_rand(g, cv_suffix, train_mask, objective, seed, n_estimators=3000):
    """15-6에서 배운 대로 무작위성을 켠 풍속 모델(seed가 실제로 영향을 준다)."""
    key = (g, cv_suffix, f"{objective}_rand{seed}")
    if key in WIND_PRED_CACHE:
        return WIND_PRED_CACHE[key]
    X = build_wind_input_frame(train, g)
    y = train[f"scada_ws_{g}"]
    fit_idx = train_mask & y.notna()
    cutoff = train.loc[fit_idx, "kst_dtm"].quantile(0.9)
    tr = fit_idx & (train["kst_dtm"] <= cutoff)
    es = fit_idx & (train["kst_dtm"] > cutoff)
    m = lgb.LGBMRegressor(objective=objective, random_state=seed, n_estimators=n_estimators,
                          verbosity=-1, **RAND_PARAMS)
    m.fit(X.loc[tr], y[tr], eval_set=[(X.loc[es], y[es])], eval_metric="rmse",
          callbacks=[lgb.early_stopping(100, verbose=False)])
    pred = pd.Series(m.predict(X), index=train.index).clip(lower=0.0)
    WIND_PRED_CACHE[key] = pred
    return pred


WIND_SEEDS = [SEED, 7, 123]

for fold_name, info in FOLD_INFO.items():
    for g in GROUP_COLS:
        for sd in WIND_SEEDS:
            get_wind_estimate_rand(g, info["cv_suffix"], info["train_mask"], "l2", sd)
    print(f"  {fold_name} 완료")


def wind_candidate(g, cv_suffix, train_mask, kind):
    """kind별 추정 풍속 Series를 돌려준다."""
    l2 = get_wind_estimate(g, cv_suffix, train_mask, "l2")
    l1 = get_wind_estimate(g, cv_suffix, train_mask, "l1")
    seeds = [get_wind_estimate_rand(g, cv_suffix, train_mask, "l2", sd) for sd in WIND_SEEDS]
    if kind == "l2":      return l2
    if kind == "l1":      return l1
    if kind == "avg12":   return (l1 + l2) / 2
    if kind == "seedens": return sum(seeds) / len(seeds)
    if kind == "avgall":  return (l1 + l2 + sum(seeds)) / (2 + len(seeds))
    if kind == "ridge":   return train[f"{g}_ws_est_cv_{cv_suffix}"]
    raise ValueError(kind)


WIND_KINDS = ["ridge", "l2", "l1", "avg12", "seedens", "avgall"]
rows = []
for fold_name, info in FOLD_INFO.items():
    vi, cs, tm = info["valid_idx"], info["cv_suffix"], info["train_mask"]
    for g in GROUP_COLS:
        truth = train.loc[vi, f"scada_ws_{g}"]
        ok = truth.notna()
        t = truth[ok]
        for kind in WIND_KINDS:
            p = wind_candidate(g, cs, tm, kind).loc[vi][ok]
            rows.append({"fold": fold_name, "group": g, "풍속": kind, "상관": t.corr(p),
                         "RMSE": float(np.sqrt(((p - t) ** 2).mean())),
                         "MAE": float((p - t).abs().mean()), "편향": float((p - t).mean())})
ws_eval2 = pd.DataFrame(rows)

print("\n=== 1단계: 풍속 정확도 (12개 fold x 그룹 평균) ===")
display(ws_eval2.groupby("풍속")[["상관", "RMSE", "MAE", "편향"]].mean().round(4)
        .reindex(WIND_KINDS))
print("\n=== 그룹별 상관 ===")
display(ws_eval2.pivot_table(index="group", columns="풍속", values="상관")[WIND_KINDS].round(4))

  A안(2024) 완료
  B안 fold1 완료
  B안 fold2 완료
  B안 fold3 완료

=== 1단계: 풍속 정확도 (12개 fold x 그룹 평균) ===


,상관,RMSE,MAE,편향
풍속,,,,
ridge,0.8798,1.7203,1.3298,-0.1027
l2,0.9109,1.4955,1.1237,-0.0625
l1,0.9124,1.4871,1.1145,-0.1053
avg12,0.9142,1.4704,1.1032,-0.0839
seedens,0.9135,1.4768,1.1109,-0.0556
avgall,0.9146,1.4674,1.1027,-0.0669



=== 그룹별 상관 ===


풍속,ridge,l2,l1,avg12,seedens,avgall
group,,,,,,
kpx_group_1,0.8734,0.9096,0.9109,0.9126,0.9120,0.9130
kpx_group_2,0.8788,0.9159,0.9172,0.9191,0.9182,0.9194
kpx_group_3,0.8871,0.9073,0.9092,0.9110,0.9101,0.9113


**확인할 것**: `avg12` / `seedens` / `avgall`이 단일 `l2`(상관 0.9110)를 넘는가? 평균 내는 것만으로 상관이 오르면, **하류(발전량)까지 이득이 전달되는지**를 17-2에서 확인합니다.

단, 16절에서 배운 대로 **1단계 승자가 2단계 승자가 아닐 수 있습니다.** 최종 판단은 17-2에서 합니다.

## 17-2. 2단계 평가 — 앙상블 풍속으로 발전량 점수가 오르는가

1단계에서 좋았던 후보 2개(`avg12`, `avgall`)와 `seedens`를 발전량 모델에 넣습니다. 나머지 설정은 16-4와 완전히 동일하게 둡니다(피처 축소 `BEST_TOPN`, 표본가중 actual, τ=0.60).

⚠️ 36번 학습(3후보 × 4fold × 3그룹).

In [78]:
def wind_fit_fn2(kind, tag, top_n=None, tau=0.60, seed=SEED):
    def fit_fn(g, cv_suffix, train_mask, valid_idx):
        ws = wind_candidate(g, cv_suffix, train_mask, kind)
        X = build_frame_with_wind(train, g, cv_suffix, train_mask, ws, tag)
        imp_key = (g, cv_suffix, tag, tau)
        if imp_key not in IMP_CACHE:
            m0 = _lgbm_train(X, g, train_mask, "quantile", tau, "actual", SEED)
            IMP_CACHE[imp_key] = pd.Series(m0.feature_importances_, index=m0.feature_name_, dtype=float)
        imp = IMP_CACHE[imp_key].reindex(X.columns).fillna(0.0).sort_values(ascending=False)
        keep = imp[imp > 0].index.tolist() if top_n is None else imp.head(top_n).index.tolist()
        m = _lgbm_train(X[keep], g, train_mask, "quantile", tau, "actual", seed)
        p = pd.Series(m.predict(X.loc[valid_idx, keep]), index=valid_idx)
        return p.clip(lower=0, upper=CAPACITY_KWH[g])
    return fit_fn


stage14 = []
for label, kind in [("W_avg12", "avg12"), ("W_seedens", "seedens"), ("W_avgall", "avgall")]:
    print(f"=== {label} ===")
    stage14.append(run_variant(label, wind_fit_fn2(kind, kind, top_n=BEST_TOPN)))

display(summarize(stage13 + stage14))

print("\n=== σ 비교 ===")
rows = []
for v in ["W_ridge", "W_gbdt_l2", "W_avg12", "W_seedens", "W_avgall", "oracle_ws"]:
    if (list(FOLD_INFO)[0], v, GROUP_COLS[0]) not in PRED_CACHE:
        continue
    e = eval_frame(v)
    rows.append({"변형": v, "σ": round(e["signed"].std(), 4), "편향": round(e["signed"].mean(), 4),
                 "밴드폭 ÷ σ": round(0.06 / e["signed"].std(), 3)})
display(pd.DataFrame(rows))

=== W_avg12 ===
  [W_avg12] A안(2024): score=0.6386  (1-NMAE=0.8728, FICR=0.4044)
  [W_avg12] B안 fold1: score=0.6059  (1-NMAE=0.8576, FICR=0.3542)
  [W_avg12] B안 fold2: score=0.6339  (1-NMAE=0.8686, FICR=0.3991)
  [W_avg12] B안 fold3: score=0.6462  (1-NMAE=0.8767, FICR=0.4157)
=== W_seedens ===
  [W_seedens] A안(2024): score=0.6355  (1-NMAE=0.8716, FICR=0.3995)
  [W_seedens] B안 fold1: score=0.6050  (1-NMAE=0.8573, FICR=0.3528)
  [W_seedens] B안 fold2: score=0.6302  (1-NMAE=0.8662, FICR=0.3941)
  [W_seedens] B안 fold3: score=0.6481  (1-NMAE=0.8771, FICR=0.4191)
=== W_avgall ===
  [W_avgall] A안(2024): score=0.6361  (1-NMAE=0.8733, FICR=0.3989)
  [W_avgall] B안 fold1: score=0.6046  (1-NMAE=0.8572, FICR=0.3521)
  [W_avgall] B안 fold2: score=0.6294  (1-NMAE=0.8684, FICR=0.3903)
  [W_avgall] B안 fold3: score=0.6488  (1-NMAE=0.8779, FICR=0.4196)


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
W_gbdt_l2,0.6391,0.6061,0.6316,0.6517,0.6298,0.0229
W_avg12,0.6386,0.6059,0.6339,0.6462,0.6286,0.0206
W_seedens,0.6355,0.6050,0.6302,0.6481,0.6278,0.0216
W_avgall,0.6361,0.6046,0.6294,0.6488,0.6276,0.0221
W_gbdt_l1,0.6352,0.6052,0.6311,0.6396,0.6253,0.0179
W_blend,0.6335,0.5991,0.6277,0.6441,0.6236,0.0228
W_ridge,0.6267,0.5906,0.6143,0.6353,0.6134,0.0224



=== σ 비교 ===


,변형,σ,편향,밴드폭 ÷ σ
0,W_ridge,0.1727,0.0526,0.347
1,W_gbdt_l2,0.1684,0.0283,0.356
2,W_avg12,0.1678,0.0312,0.357
3,W_seedens,0.1682,0.0348,0.357
4,W_avgall,0.1672,0.0323,0.359
5,oracle_ws,0.0786,0.0133,0.763


**확인할 것**: 가장 좋은 풍속 후보를 고르세요. **A안·B안 3-fold 전부에서** `W_gbdt_l2`(0.6391 / 0.6298)를 이겨야 채택합니다. 아래 셀의 `BEST_WIND_KIND`를 그 값으로 바꿔주세요(기본값은 자동 선택).

## 17-3. τ 재조정 — 편향이 바뀌었으니 최적점도 바뀐다

τ=0.60은 **Ridge 풍속 시절의 편향(-0.05 과소예측)** 을 기준으로 고른 값입니다. 그런데 풍속 모델을 바꾸자 편향이 **+0.0526 → +0.0283** 으로 변했습니다(과대 쪽이지만 크기는 절반).

τ의 역할은 "예측을 위아래로 미는 것"인데, **밀어야 할 거리가 달라졌으므로 τ도 다시 정해야** 합니다. 13절에서 τ를 정할 때와 지금은 모델이 다릅니다.

방향 예측: 지금 편향이 **양수(+0.028, 과대)** 이므로 τ를 **낮추는** 쪽이 유리할 수 있습니다. 하지만 FICR은 actual 가중이라 큰 발전량 구간이 중요하고, 그쪽에서는 여전히 과소일 수 있으므로 **양쪽을 다 봅니다**: τ ∈ {0.50, 0.55, 0.60, 0.65}.

⚠️ 36번 학습(τ=0.60은 17-2에서 이미 있으므로 3개만 추가).

In [79]:
# 17-2에서 가장 좋았던 풍속 후보 (자동 선택 — 표를 보고 직접 바꿔도 됨)
_wt = summarize(stage13 + stage14)
BEST_WIND_LABEL = _wt.index[0]
BEST_WIND_KIND = {"W_gbdt_l2": "l2", "W_gbdt_l1": "l1", "W_blend": "blend",
                  "W_ridge": "ridge", "W_avg12": "avg12", "W_seedens": "seedens",
                  "W_avgall": "avgall"}[BEST_WIND_LABEL]
print(f"선택된 풍속: {BEST_WIND_LABEL} ({BEST_WIND_KIND}), B안 평균 {_wt.loc[BEST_WIND_LABEL, 'B안 평균']:.4f}")
print("※ 직접 바꾸려면 BEST_WIND_KIND를 수정하세요.")

stage15 = []
for tau in [0.50, 0.55, 0.65]:
    label = f"X_tau{int(tau*100)}"
    print(f"\n=== {label} ===")
    stage15.append(run_variant(label, wind_fit_fn2(BEST_WIND_KIND, f"{BEST_WIND_KIND}_t{int(tau*100)}",
                                                  top_n=BEST_TOPN, tau=tau)))

display(summarize(stage13 + stage14 + stage15))

# 그룹별로도 확인 (15-3의 항등식 활용)
rows = []
for label in [f"X_tau{int(t*100)}" for t in [0.50, 0.55, 0.65]] + [BEST_WIND_LABEL]:
    for fold_name, info in FOLD_INFO.items():
        pred = pd.DataFrame({g: PRED_CACHE[(fold_name, label, g)] for g in GROUP_COLS},
                            index=info["valid_idx"])
        for g, sc in group_scores(info["actual_df"], pred).items():
            rows.append({"변형": label, "fold": fold_name, "group": g, "score": sc})
gt = pd.DataFrame(rows)
print("\n=== 그룹별 점수 (B안 3-fold 평균) ===")
display(gt[gt["fold"].isin(B_FOLDS)].pivot_table(index="변형", columns="group", values="score").round(4))

선택된 풍속: W_gbdt_l2 (l2), B안 평균 0.6298
※ 직접 바꾸려면 BEST_WIND_KIND를 수정하세요.

=== X_tau50 ===
  [X_tau50] A안(2024): score=0.6288  (1-NMAE=0.8736, FICR=0.3841)
  [X_tau50] B안 fold1: score=0.5939  (1-NMAE=0.8553, FICR=0.3326)
  [X_tau50] B안 fold2: score=0.6251  (1-NMAE=0.8703, FICR=0.3799)
  [X_tau50] B안 fold3: score=0.6419  (1-NMAE=0.8789, FICR=0.4048)

=== X_tau55 ===
  [X_tau55] A안(2024): score=0.6345  (1-NMAE=0.8740, FICR=0.3950)
  [X_tau55] B안 fold1: score=0.6004  (1-NMAE=0.8561, FICR=0.3448)
  [X_tau55] B안 fold2: score=0.6313  (1-NMAE=0.8707, FICR=0.3919)
  [X_tau55] B안 fold3: score=0.6461  (1-NMAE=0.8790, FICR=0.4133)

=== X_tau65 ===
  [X_tau65] A안(2024): score=0.6360  (1-NMAE=0.8692, FICR=0.4027)
  [X_tau65] B안 fold1: score=0.6070  (1-NMAE=0.8549, FICR=0.3592)
  [X_tau65] B안 fold2: score=0.6278  (1-NMAE=0.8639, FICR=0.3916)
  [X_tau65] B안 fold3: score=0.6484  (1-NMAE=0.8741, FICR=0.4228)


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
W_gbdt_l2,0.6391,0.6061,0.6316,0.6517,0.6298,0.0229
W_avg12,0.6386,0.6059,0.6339,0.6462,0.6286,0.0206
X_tau65,0.6360,0.6070,0.6278,0.6484,0.6278,0.0207
W_seedens,0.6355,0.6050,0.6302,0.6481,0.6278,0.0216
W_avgall,0.6361,0.6046,0.6294,0.6488,0.6276,0.0221
X_tau55,0.6345,0.6004,0.6313,0.6461,0.6260,0.0233
W_gbdt_l1,0.6352,0.6052,0.6311,0.6396,0.6253,0.0179
W_blend,0.6335,0.5991,0.6277,0.6441,0.6236,0.0228
X_tau50,0.6288,0.5939,0.6251,0.6419,0.6203,0.0243



=== 그룹별 점수 (B안 3-fold 평균) ===


group,kpx_group_1,kpx_group_2,kpx_group_3
변형,,,
W_gbdt_l2,0.6386,0.6684,0.5825
X_tau50,0.6212,0.6632,0.5764
X_tau55,0.6318,0.6650,0.5810
X_tau65,0.6377,0.6616,0.5839


**확인할 것**: τ를 낮추는 게 이득인가? 그리고 **그룹별로 방향이 다른가**? 15-3에서는 그룹별 τ 이득이 노이즈 수준이었지만, 풍속 모델이 바뀌었으니 **다시 확인할 가치**가 있습니다.

## 17-4. ⭐ 제출 경로 전체 리허설 — 한 번도 안 해본 일

지금까지 모든 실험은 **train 안에서만** 돌았습니다. **test(2025년)에 실제로 적용해본 적이 한 번도 없습니다.** 마감이 2026-08-14인데, 제출 직전에 여기서 터지면 손 쓸 시간이 없습니다.

### 미리 확인한 구조적 문제

`test_features_v1.parquet`(8760행, 860컬럼)에는 **train에만 있는 컬럼 28개가 없습니다.**

| 없는 컬럼 | 왜 없나 | 우리 코드가 터지는 지점 |
|---|---|---|
| `kpx_group_1/2/3` | 정답이니까 (예측 대상) | `build_frame_with_wind`이 파워커브를 적합할 때 `df[g]`를 씀 |
| `{g}_ws_est_cv_*`, `{g}_power_curve_est_cv_*` | fold 검증용이라 test엔 안 만듦 | `build_group_feature_frame_v3(test, ...)`는 **KeyError로 죽는다** |
| `scada_*` | 실측이니까 | 풍속 모델의 **타깃**이라 예측엔 불필요 (문제 없음) |

그래서 **검증용 코드를 그대로 test에 쓸 수 없습니다.** 최종 경로는 구조가 다릅니다.

```
[검증]  fold 학습구간 → 풍속모델 → 파워커브 → 발전량모델 → 같은 표의 검증구간 예측
[제출]  train 전체    → 풍속모델 → 파워커브 → 발전량모델 → 다른 표(test) 예측
                                    ↑ 파워커브도 train에서 적합해 test에 "적용"만 한다
```

### 이 절에서 하는 것

1. test 시간축 연속성 확인 (lag/lead 파생의 전제)
2. **전체 train으로** 풍속 모델 학습 → test 예보에 적용
3. **전체 train으로** 파워커브 적합 → test 풍속에 적용
4. **전체 train으로** 발전량 모델 학습 → test 예측
5. train/test 피처 프레임의 **컬럼이 정확히 일치**하는지 확인
6. `src/submission.py`의 `validate_submission()` 통과 확인
7. 예측값 분포가 상식적인지(음수 없음, 용량 초과 없음, 이용률 분포가 train과 비슷한지) 확인

⚠️ 6번 학습(풍속 3 + 발전량 3, 전체 train 사용). **여기서 만든 제출 파일은 실제로 내도 됩니다.**

In [80]:
import src.submission as subm
from src.submission import build_submission, validate_submission, save_submission

# ⚠️ CWD 주의: 노트북은 notebooks/ 안에서 실행되므로 src/submission.py의 기본 경로
#    Path("data/sample_submission.csv")가 notebooks/data/... 로 잘못 해석된다.
#    2절의 PROCESSED_DIR와 같은 방식으로 저장소 루트를 잡아 모듈 전역을 바로잡는다.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE_PATH = REPO_ROOT / "data" / "sample_submission.csv"
subm.SAMPLE_SUBMISSION_PATH = SAMPLE_PATH          # 기본 인자용
subm.SUBMISSIONS_DIR = REPO_ROOT / "submissions"   # 저장 위치도 루트 기준으로
assert SAMPLE_PATH.exists(), f"sample_submission.csv를 못 찾음: {SAMPLE_PATH}"
print("저장소 루트:", REPO_ROOT)
print("sample_submission:", SAMPLE_PATH)

test = pd.read_parquet(PROCESSED_DIR / "test_features_v1.parquet")
print("test:", test.shape, "|", test["kst_dtm"].min(), "~", test["kst_dtm"].max())

# ── (1) 시간축 연속성 (lag/lead 파생의 전제) ──────────────────────
gaps = test["kst_dtm"].diff().dropna().unique()
assert test["kst_dtm"].is_monotonic_increasing and len(gaps) == 1 and gaps[0] == pd.Timedelta("1h"), \
    f"test 시간축이 1시간 연속이 아님: {gaps[:5]}"
print("✓ test 시간축 1시간 연속")

# ── (2) 전체 train으로 풍속 모델 학습 → test에 적용 ────────────────
FULL_MASK = pd.Series(True, index=train.index)      # fold 컷오프 없음 = 전체 train


def fit_final_wind(g, objective, seed, n_estimators=3000, rand=False):
    """⚠️ rand=False가 기본이다.
    17-2에서 확인: 무작위성을 켠 풍속(W_seedens 0.6278)·풍속 평균(W_avgall 0.6276)이
    결정적 단일 l2 풍속(W_gbdt_l2 0.6298)보다 오히려 낮았다.
    발전량이 풍속에 볼록(~v^3)하게 반응하므로 풍속 추정을 평균/평활하면 극값이 눌려
    강풍 구간 발전량을 과소평가하기 때문이다(Jensen 부등식).
    → 풍속 모델은 결정적으로, 앙상블은 파이프라인 마지막 단(발전량)에서만."""
    X = build_wind_input_frame(train, g)
    y = train[f"scada_ws_{g}"]
    fit_idx = y.notna()
    cutoff = train.loc[fit_idx, "kst_dtm"].quantile(0.9)   # 뒤 10%는 early stopping용
    tr, es = fit_idx & (train["kst_dtm"] <= cutoff), fit_idx & (train["kst_dtm"] > cutoff)
    extra = RAND_PARAMS if rand else {}
    m = lgb.LGBMRegressor(objective=objective, random_state=seed, n_estimators=n_estimators,
                          verbosity=-1, **extra)
    m.fit(X.loc[tr], y[tr], eval_set=[(X.loc[es], y[es])], eval_metric="rmse",
          callbacks=[lgb.early_stopping(100, verbose=False)])
    return m


final_wind_models, ws_train_final, ws_test_final = {}, {}, {}
for g in GROUP_COLS:
    m = fit_final_wind(g, "l2", SEED)
    final_wind_models[g] = m
    Xtr_w, Xte_w = build_wind_input_frame(train, g), build_wind_input_frame(test, g)
    assert list(Xtr_w.columns) == list(Xte_w.columns), f"{g}: 풍속 모델 입력 컬럼이 train/test 불일치"
    ws_train_final[g] = pd.Series(m.predict(Xtr_w), index=train.index).clip(lower=0.0)
    ws_test_final[g] = pd.Series(m.predict(Xte_w), index=test.index).clip(lower=0.0)
    r = ws_train_final[g].corr(train[f"scada_ws_{g}"])
    print(f"✓ {g}: 풍속모델 학습 완료 (train 내 상관 {r:.4f}, "
          f"test 추정풍속 평균 {ws_test_final[g].mean():.2f} m/s / train {ws_train_final[g].mean():.2f} m/s)")

저장소 루트: d:\공모전\wind_forecast_new
sample_submission: d:\공모전\wind_forecast_new\data\sample_submission.csv
test: (8760, 860) | 2025-01-01 01:00:00 ~ 2026-01-01 00:00:00
✓ test 시간축 1시간 연속
✓ kpx_group_1: 풍속모델 학습 완료 (train 내 상관 0.9284, test 추정풍속 평균 7.17 m/s / train 6.78 m/s)
✓ kpx_group_2: 풍속모델 학습 완료 (train 내 상관 0.9542, test 추정풍속 평균 7.67 m/s / train 7.19 m/s)
✓ kpx_group_3: 풍속모델 학습 완료 (train 내 상관 0.9586, test 추정풍속 평균 6.31 m/s / train 5.89 m/s)


In [81]:
# ── (3)(4) 파워커브 + 발전량 모델: 라벨 없는 test에도 쓸 수 있는 프레임 빌더 ──
def build_frame_given_pc(df, g, ws_series, edges, vals, tag, dynamics=True, lead_feat=True):
    """파워커브를 '적합'하지 않고 '적용'만 한다 → 라벨이 없는 test에도 쓸 수 있다.
    (build_frame_with_wind과 결과가 같아야 하며, 아래에서 검산한다)"""
    all_cv = set(all_cv_cols_for_group(g))
    exclude = set(leaky_cols_for_group(g)) | all_cv
    base_cols = [c for c in GROUP_SPECIFIC_COLS[g] if c not in exclude]
    missing = [c for c in COMMON_RAW_COLS + base_cols if c not in df.columns]
    assert not missing, f"{g}: df에 없는 컬럼 {missing[:5]}"

    ws_col, pc_col = f"{g}_ws_{tag}", f"{g}_power_curve_{tag}"
    tmp = pd.DataFrame({"kst_dtm": df["kst_dtm"], f"{g}_air_density": df[f"{g}_air_density"]},
                       index=df.index)
    tmp[ws_col] = ws_series.astype(float)
    if g in ICING_RISK_GROUPS:
        tcol = f"ldaps_g{GROUP_NEAREST_LDAPS[g]}_heightAboveGround_2_t"
        tmp[tcol] = df[tcol]
    tmp[pc_col] = apply_power_curve_oracle(tmp[ws_col].to_numpy(dtype=float), edges, vals)

    parts = [df[COMMON_RAW_COLS + base_cols], tmp[[ws_col, pc_col]],
             add_fold_safe_ws_features(tmp, g, ws_col)]
    if dynamics:
        parts.append(add_forecast_dynamics(tmp, [ws_col, pc_col]))
    if lead_feat:
        parts.append(add_lead_features(df))
    out = pd.concat(parts, axis=1)
    assert out.isna().sum().sum() == 0, f"{g}: 프레임에 결측"
    return out


FINAL_TAU = 0.60          # 17-3 결과를 보고 수정하세요
FINAL_SEEDS = [SEED, 7, 123, 2024, 31]

sub_pred = {}
for g in GROUP_COLS:
    # 파워커브: 전체 train의 (최종 추정풍속, 실제 발전량)으로 적합
    ok = train[g].notna()
    edges, vals = fit_power_curve_oracle(ws_train_final[g][ok].to_numpy(dtype=float),
                                         train.loc[ok, g].to_numpy(dtype=float))

    Xtr = build_frame_given_pc(train, g, ws_train_final[g], edges, vals, "final")
    Xte = build_frame_given_pc(test, g, ws_test_final[g], edges, vals, "final")
    assert list(Xtr.columns) == list(Xte.columns), f"{g}: 발전량 모델 입력 컬럼이 train/test 불일치"

    # 피처 축소 (전체 train 기준 중요도)
    m0 = _lgbm_train(Xtr, g, FULL_MASK, "quantile", FINAL_TAU, "actual", SEED)
    imp = pd.Series(m0.feature_importances_, index=m0.feature_name_, dtype=float).sort_values(ascending=False)
    keep = imp[imp > 0].index.tolist() if BEST_TOPN is None else imp.head(BEST_TOPN).index.tolist()

    # 시드 앙상블
    preds = []
    for sd in FINAL_SEEDS:
        m = _lgbm_train_rand(Xtr[keep], g, FULL_MASK, "quantile", FINAL_TAU, "actual", sd)
        preds.append(m.predict(Xte[keep]))
    sub_pred[g] = np.clip(np.mean(preds, axis=0), 0, CAPACITY_KWH[g])
    print(f"✓ {g}: 피처 {len(keep)}개 / seed {len(FINAL_SEEDS)}개 / "
          f"예측 평균 {sub_pred[g].mean():,.0f} kWh (이용률 {sub_pred[g].mean()/CAPACITY_KWH[g]*100:.1f}%)")

✓ kpx_group_1: 피처 200개 / seed 5개 / 예측 평균 8,769 kWh (이용률 40.6%)
✓ kpx_group_2: 피처 200개 / seed 5개 / 예측 평균 9,316 kWh (이용률 43.1%)
✓ kpx_group_3: 피처 200개 / seed 5개 / 예측 평균 7,628 kWh (이용률 36.3%)


In [82]:
# ── (5)(6)(7) 제출 파일 생성 + 공식 검증 + 상식 점검 ──────────────
pred_df = pd.DataFrame(sub_pred, index=test.index)
pred_df["forecast_kst_dtm"] = test["kst_dtm"].dt.strftime("%Y-%m-%d %H:%M:%S")

submission = build_submission(pred_df, sample_path=SAMPLE_PATH)
validate_submission(submission, sample_path=SAMPLE_PATH)
print("✓ validate_submission() 통과 —", submission.shape)
display(submission.head(3))

print("\n=== 상식 점검: test 예측 vs train 실제 이용률 분포 ===")
rows = []
for g in GROUP_COLS:
    cap = CAPACITY_KWH[g]
    p, a = submission[g] / cap, train[g].dropna() / cap
    rows.append({"group": g,
                 "예측 평균": round(p.mean(), 4), "실제(train) 평균": round(a.mean(), 4),
                 "예측 표준편차": round(p.std(), 4), "실제 표준편차": round(a.std(), 4),
                 "예측 ≥10% 비율": round((p >= 0.10).mean(), 4), "실제 ≥10% 비율": round((a >= 0.10).mean(), 4),
                 "예측 최대": round(p.max(), 4)})
display(pd.DataFrame(rows))

print("\n=== 월별 예측 이용률 (계절성이 상식적인지) ===")
mm = submission.copy()
mm["month"] = pd.to_datetime(mm["forecast_kst_dtm"]).dt.month
display((mm.groupby("month")[GROUP_COLS].mean() / pd.Series(CAPACITY_KWH)).round(3))

✓ validate_submission() 통과 — (8760, 5)


,forecast_id,forecast_kst_dtm,kpx_group_1,kpx_group_2,kpx_group_3
0,forecast_0001,2025-01-01 01:00:00,19673.073603,19335.629276,18771.807243
1,forecast_0002,2025-01-01 02:00:00,18875.858504,19181.584488,18548.372327
2,forecast_0003,2025-01-01 03:00:00,19300.067212,18825.074632,18233.804654



=== 상식 점검: test 예측 vs train 실제 이용률 분포 ===


,group,예측 평균,실제(train) 평균,예측 표준편차,실제 표준편차,예측 ≥10% 비율,실제 ≥10% 비율,예측 최대
0,kpx_group_1,0.4060,0.3112,0.2876,0.3047,0.7864,0.6167,0.9719
1,kpx_group_2,0.4313,0.3326,0.3038,0.3242,0.7739,0.6149,0.9640
2,kpx_group_3,0.3632,0.2698,0.3006,0.3006,0.6969,0.5465,0.9545



=== 월별 예측 이용률 (계절성이 상식적인지) ===


,kpx_group_1,kpx_group_2,kpx_group_3
month,,,
1,0.542,0.568,0.488
2,0.606,0.649,0.577
3,0.421,0.455,0.385
4,0.435,0.462,0.402
5,0.372,0.389,0.334
6,0.361,0.383,0.318
7,0.307,0.346,0.275
8,0.305,0.330,0.281
9,0.226,0.250,0.185


**확인할 것**

1. **`validate_submission() 통과`가 떴는가** — 안 뜨면 어떤 assert에서 걸렸는지 알려주세요
2. **예측 평균 이용률이 train 실제와 비슷한가** — 크게 다르면(예: 예측 25% vs 실제 18%) 뭔가 잘못된 것입니다
3. **예측 표준편차가 실제보다 많이 작지 않은가** — 회귀 모델은 평균으로 수축하는 성질이 있어 예측이 실제보다 덜 흔들립니다. 너무 작으면 FICR에 불리합니다
4. **월별 계절성이 상식적인가** — 태백 가덕산은 겨울(12~2월)에 바람이 강하고 여름(7~8월)에 약합니다. 반대로 나오면 문제입니다
5. **`예측 ≥10% 비율`** — 채점 대상이 될 시간대의 비율입니다. train 실제와 비슷해야 정상입니다

통과했다면 아래 셀로 **실제 제출 파일을 저장**할 수 있습니다.

In [83]:
# 저장은 확인 후 직접 실행하세요 (파일명 규칙: YYYYMMDD_vN_설명.csv)
path = save_submission(submission, "20260801_v1_wsgbdt_q60_seed5.csv", sample_path=SAMPLE_PATH)
print("저장:", path)

저장: d:\공모전\wind_forecast_new\submissions\20260801_v1_wsgbdt_q60_seed5.csv


## 17-5. 여기까지의 전체 진행

| 단계 | A안 | B평균 | 비고 |
|---|---|---|---|
| `fix_base` (누수 없는 출발점) | 0.5915 | 0.5888 | 13절 |
| `dyn_w_q60` (표본가중 + τ) | 0.6252 | 0.6124 | 13절 |
| `G_grouptau` (+리드타임·피처축소·그룹별τ) | 0.6250 | 0.6161 | 15절 |
| `R_randavg5` (+시드 앙상블) | 0.6259 | 0.6170 | 15-6절 |
| **`W_gbdt_l2` (풍속 모델 교체)** | **0.6391** | **0.6298** | **16절 — 단일 최대 기여** |
| (17절 결과) | | | |
| *오라클 상한* | *0.8587* | *0.8618* | *14절* |

**아직 오라클까지 0.23이 남아 있습니다.** 풍속 상관을 0.91에서 더 올리는 것이 계속 최대 레버입니다. 남은 후보:

- **격자 CNN 풍속 모델** — 격자를 이미지로 보는 접근. 아직 전혀 시도 안 함(갈래 A2)
- **MLP 풍속 모델** — GBDT와 오차 방향이 달라 앙상블 가치가 큼
- **풍속 모델 하이퍼파라미터 튜닝** — 지금은 기본값 + `n_estimators=3000`뿐
- **분위수 풍속 예측** — 풍속의 불확실성을 발전량 모델에 넘기기
- **분포 예측 + 기대 FICR 최대화**(갈래 B1) — 산식을 직접 노리는 마지막 카드

---

# 18. 풍속 모델 심화 — 튜닝 + MLP + 마지막 단 앙상블

## 왜 계속 풍속인가

| | 값 |
|---|---|
| 현재 리더보드 | **0.6271** (1-NMAE 0.8592 / FICR **0.3950**) |
| 오라클 상한(완벽한 바람) | 0.8618 (FICR 0.76) |
| 예보 오차가 차지하는 σ 비중 | **89%** |
| 현재 풍속 모델 | LightGBM **기본값** + `n_estimators=3000` 뿐 |

FICR이 0.395로 1에서 가장 멀고, 오라클이 0.76을 보여줬으므로 **상방은 전부 풍속에 있습니다.** 그런데 정작 풍속 모델은 **하이퍼파라미터를 한 번도 안 건드렸습니다.**

## 이 절에서 바꾸는 것 하나 — 스크리닝 지표

17절에서 **"풍속 상관 1등이 발전량 점수 꼴찌"** 를 겪었습니다. 그러니 풍속 후보를 상관·RMSE로 고르면 또 틀립니다. 그렇다고 후보마다 발전량 모델을 다 학습시키면 너무 비쌉니다.

**해결: 발전량 모델을 학습하지 않고도 "이 풍속 오차가 발전량에 얼마나 손해인지"를 재는 지표를 만듭니다.**

```
추정 풍속 ──파워커브──→ 발전량 A
실측 풍속 ──파워커브──→ 발전량 B     (같은 파워커브)
            → |A - B| / 설비용량 = "발전량 환산 오차율"
```

두 풍속을 **같은 파워커브**에 통과시켜 발전량 단위로 바꾼 뒤 오차를 재는 것입니다. 이러면:

- **볼록성(v³)이 지표 안에 자동으로 반영**됩니다. 강풍 구간의 1 m/s 오차가 약풍 구간보다 훨씬 크게 계산됩니다 — 실제 손해와 같은 방식으로요
- 산식과 같은 형태로 **`1-NMAE 상당`, `FICR 상당`** 을 바로 뽑을 수 있습니다
- 학습이 **0번**입니다 (파워커브 적용은 표 조회일 뿐)

> 이건 "풍속만이 유일한 오차원이라면 몇 점이 나올까"를 계산하는 것입니다. 14절 `pc_oracle`(0.8254)의 축소판이라고 보면 됩니다.

## 순서

| 절 | 내용 | 학습 |
|---|---|---|
| 18-1 | 발전량 환산 오차 지표 만들기 + 현재 모델 진단 | 0 |
| 18-2 | 풍속 하이퍼파라미터 후보 5종 **스크리닝**(A안 fold만) | 15 |
| 18-3 | 상위 2종을 전체 fold + 발전량 점수로 확인 | 24+24 |
| 18-4 | **MLP 풍속 모델** (GBDT와 오차 방향이 다른 자원) | 12 |
| 18-5 | **마지막 단 앙상블** (17절 교훈: 풍속 말고 발전량 예측을 평균) | 0 |

## 18-1. 발전량 환산 오차 지표

각 fold의 **검증 구간에서만**, SCADA 실측이 있는 행에 대해 계산합니다. 파워커브는 **그 fold의 학습 구간에서** (실측풍속, 실제발전량)으로 적합합니다(fold-safe).

산식과 같은 방식으로 두 숫자를 뽑습니다.
- **`1-NMAE 상당`** = 1 − 평균(발전량 환산 오차율)
- **`FICR 상당`** = 실제발전량 가중으로 (오차율 ≤6%)×1.0 + (6~8%)×0.75
- **`Score 상당`** = 0.5 × 둘의 합

숫자 자체는 실제 점수보다 **높게** 나옵니다(발전량 모델의 오차가 빠져 있으니까요). **중요한 건 절대값이 아니라 후보들 사이의 순위**입니다.

In [90]:
PC_CACHE = {}   # (g, cv_suffix) -> 물리 파워커브(실측풍속 기준, 학습구간 적합)


def get_physical_pc(g, cv_suffix, train_mask):
    key = (g, cv_suffix)
    if key not in PC_CACHE:
        fit_idx = train_mask & train[g].notna() & train[f"scada_ws_{g}"].notna()
        PC_CACHE[key] = fit_power_curve_oracle(
            train.loc[fit_idx, f"scada_ws_{g}"].to_numpy(dtype=float),
            train.loc[fit_idx, g].to_numpy(dtype=float))
    return PC_CACHE[key]


def wind_power_error(ws_pred_full, g, cv_suffix, train_mask, valid_idx):
    """추정 풍속과 실측 풍속을 같은 파워커브에 통과시켜 '발전량 단위' 오차를 잰다.
    발전량 모델을 학습하지 않고도 풍속 후보의 우열을 산식과 같은 형태로 비교할 수 있다."""
    cap = CAPACITY_KWH[g]
    truth_ws = train.loc[valid_idx, f"scada_ws_{g}"]
    idx = valid_idx[truth_ws.notna().to_numpy()]
    edges, vals = get_physical_pc(g, cv_suffix, train_mask)

    p_true = apply_power_curve_oracle(train.loc[idx, f"scada_ws_{g}"].to_numpy(dtype=float), edges, vals)
    p_pred = apply_power_curve_oracle(ws_pred_full.loc[idx].to_numpy(dtype=float), edges, vals)

    scored = p_true >= cap * 0.10          # 채점 대상 근사
    a, er = p_true[scored], np.abs(p_pred - p_true)[scored] / cap
    one_minus_nmae = 1 - er.mean()
    ficr = float(np.average((er <= 0.06).astype(float), weights=a)
                 + 0.75 * np.average(((er > 0.06) & (er <= 0.08)).astype(float), weights=a))
    return {"1-NMAE 상당": one_minus_nmae, "FICR 상당": ficr,
            "Score 상당": 0.5 * one_minus_nmae + 0.5 * ficr,
            "풍속MAE": float((ws_pred_full.loc[idx] - train.loc[idx, f"scada_ws_{g}"]).abs()[scored].mean())}


def screen_wind(ws_fn, label, folds=None):
    """ws_fn(g, cv_suffix, train_mask) -> 풍속 Series 를 받아 발전량 환산 지표로 채점."""
    rows = []
    for fold_name, info in FOLD_INFO.items():
        if folds is not None and fold_name not in folds:
            continue
        for g in GROUP_COLS:
            ws = ws_fn(g, info["cv_suffix"], info["train_mask"])
            r = wind_power_error(ws, g, info["cv_suffix"], info["train_mask"], info["valid_idx"])
            rows.append({"풍속": label, "fold": fold_name, "group": g, **r})
    return pd.DataFrame(rows)


# 현재 쓰는 풍속들을 이 지표로 다시 줄세워 본다 (학습 0번 — 전부 캐시에 있음)
base_screen = pd.concat([
    screen_wind(lambda g, c, t: train[f"{g}_ws_est_cv_{c}"], "ridge"),
    screen_wind(lambda g, c, t: get_wind_estimate(g, c, t, "l2"), "gbdt_l2"),
    screen_wind(lambda g, c, t: get_wind_estimate(g, c, t, "l1"), "gbdt_l1"),
    screen_wind(lambda g, c, t: (get_wind_estimate(g, c, t, "l1") + get_wind_estimate(g, c, t, "l2")) / 2, "avg12"),
], ignore_index=True)

print("=== 발전량 환산 지표 (12 fold x 그룹 평균) ===")
display(base_screen.groupby("풍속")[["Score 상당", "1-NMAE 상당", "FICR 상당", "풍속MAE"]].mean().round(4)
        .reindex(["ridge", "gbdt_l2", "gbdt_l1", "avg12"]))
print("\n=== 검산: 실제 발전량 모델 점수(B안 평균)와 순위가 같은가? ===")
print("  ridge 0.6134 / gbdt_l2 0.6298 / gbdt_l1 0.6253 / avg12 0.6286   ← 16·17절 실측")

=== 발전량 환산 지표 (12 fold x 그룹 평균) ===


,Score 상당,1-NMAE 상당,FICR 상당,풍속MAE
풍속,,,,
ridge,0.6551,0.8686,0.4416,1.5202
gbdt_l2,0.6985,0.8887,0.5084,1.3112
gbdt_l1,0.6962,0.8884,0.5041,1.3086
avg12,0.6999,0.8900,0.5097,1.2896



=== 검산: 실제 발전량 모델 점수(B안 평균)와 순위가 같은가? ===
  ridge 0.6134 / gbdt_l2 0.6298 / gbdt_l1 0.6253 / avg12 0.6286   ← 16·17절 실측


**확인할 것 (이 지표를 믿어도 되는지 검증하는 단계)**

새 지표의 순위가 **실제 발전량 점수의 순위와 같아야** 합니다.

- 실측 순위: `gbdt_l2`(0.6298) > `avg12`(0.6286) > `gbdt_l1`(0.6253) > `ridge`(0.6134)
- 새 지표도 같은 순서로 나오면 → **이 지표로 후보를 스크리닝해도 안전**합니다
- 순서가 다르면 → 이 지표를 믿으면 안 되고, 18-2의 후보들은 전부 발전량 모델까지 돌려서 비교해야 합니다

특히 `avg12`(풍속 평균)가 `gbdt_l2`보다 낮게 나오는지 보세요. 상관·RMSE로는 `avg12`가 이겼지만 실제 점수는 졌습니다. **새 지표가 이걸 잡아내면 볼록성이 제대로 반영된 것입니다.**

## 18-2. 풍속 하이퍼파라미터 후보 스크리닝

현재 풍속 모델은 **LightGBM 기본값**입니다: `num_leaves=31`, `learning_rate=0.1`, `min_child_samples=20`, 정규화 없음. 피처가 853개이고 학습 표본이 1.7~2.4만 행인 문제에 기본값이 맞을 이유가 없습니다.

다섯 가지 방향을 시험합니다. 각각 **왜 이 방향인지** 근거가 있습니다.

| 후보 | 설정 | 가설 |
|---|---|---|
| `base` | 현재 (leaves 31, lr 0.1) | 기준선 |
| `slow` | leaves 31, **lr 0.03**, n 8000 | 학습률을 낮추고 트리를 늘리면 더 정밀하게 수렴한다(GBDT의 정석) |
| `wide` | **leaves 127**, lr 0.05, n 5000 | 피처가 853개라 트리 하나가 담을 수 있는 상호작용이 부족하다 |
| `reg` | leaves 63, lr 0.05, **colsample 0.7 / subsample 0.8 / min_child 40 / L2 1.0** | 피처 대부분이 잡음이므로 정규화로 과적합을 막는다 |
| `wide_reg` | **leaves 255**, lr 0.03, n 8000, colsample 0.6, min_child 40 | 위 둘의 조합 — 크게 만들되 강하게 규제 |

**A안 fold에서만 먼저 돌려 스크리닝**합니다(15번 학습). 전체 4 fold를 다 돌리면 60번이라 낭비입니다.

⚠️ 15번 학습. 풍속 모델은 피처가 많아 후보당 몇 분씩 걸릴 수 있습니다.

In [91]:
WIND_CONFIGS = {
    "base":     dict(num_leaves=31,  learning_rate=0.10, n_estimators=3000),
    "slow":     dict(num_leaves=31,  learning_rate=0.03, n_estimators=8000),
    "wide":     dict(num_leaves=127, learning_rate=0.05, n_estimators=5000),
    "reg":      dict(num_leaves=63,  learning_rate=0.05, n_estimators=5000,
                     colsample_bytree=0.7, subsample=0.8, subsample_freq=1,
                     min_child_samples=40, reg_lambda=1.0),
    "wide_reg": dict(num_leaves=255, learning_rate=0.03, n_estimators=8000,
                     colsample_bytree=0.6, subsample=0.8, subsample_freq=1,
                     min_child_samples=40, reg_lambda=1.0),
}


def get_wind_cfg(g, cv_suffix, train_mask, cfg_name, objective="l2", seed=None):
    """설정 이름으로 풍속 모델을 학습(캐시)."""
    seed = SEED if seed is None else seed
    key = (g, cv_suffix, f"cfg_{cfg_name}_{objective}_s{seed}")
    if key in WIND_PRED_CACHE:
        return WIND_PRED_CACHE[key]
    X = build_wind_input_frame(train, g)
    y = train[f"scada_ws_{g}"]
    fit_idx = train_mask & y.notna()
    cutoff = train.loc[fit_idx, "kst_dtm"].quantile(0.9)
    tr, es = fit_idx & (train["kst_dtm"] <= cutoff), fit_idx & (train["kst_dtm"] > cutoff)
    m = lgb.LGBMRegressor(objective=objective, random_state=seed, verbosity=-1,
                          **WIND_CONFIGS[cfg_name])
    m.fit(X.loc[tr], y[tr], eval_set=[(X.loc[es], y[es])], eval_metric="rmse",
          callbacks=[lgb.early_stopping(150, verbose=False)])
    pred = pd.Series(m.predict(X), index=train.index).clip(lower=0.0)
    WIND_PRED_CACHE[key] = pred
    print(f"    {cfg_name}/{g}: 트리 {m.best_iteration_}개 사용")
    return pred


screen_rows = []
for name in WIND_CONFIGS:
    print(f"=== {name} ===")
    screen_rows.append(screen_wind(lambda g, c, t, n=name: get_wind_cfg(g, c, t, n),
                                   name, folds=["A안(2024)"]))
cfg_screen = pd.concat(screen_rows, ignore_index=True)

print("\n=== A안 fold 스크리닝 (3그룹 평균) ===")
display(cfg_screen.groupby("풍속")[["Score 상당", "1-NMAE 상당", "FICR 상당", "풍속MAE"]].mean()
        .round(4).sort_values("Score 상당", ascending=False))
print("\n=== 그룹별 Score 상당 ===")
display(cfg_screen.pivot_table(index="풍속", columns="group", values="Score 상당").round(4))

=== base ===
    base/kpx_group_1: 트리 99개 사용
    base/kpx_group_2: 트리 282개 사용
    base/kpx_group_3: 트리 52개 사용
=== slow ===
    slow/kpx_group_1: 트리 327개 사용
    slow/kpx_group_2: 트리 752개 사용
    slow/kpx_group_3: 트리 158개 사용
=== wide ===
    wide/kpx_group_1: 트리 242개 사용
    wide/kpx_group_2: 트리 301개 사용
    wide/kpx_group_3: 트리 93개 사용
=== reg ===
    reg/kpx_group_1: 트리 159개 사용
    reg/kpx_group_2: 트리 471개 사용
    reg/kpx_group_3: 트리 121개 사용
=== wide_reg ===
    wide_reg/kpx_group_1: 트리 188개 사용
    wide_reg/kpx_group_2: 트리 209개 사용
    wide_reg/kpx_group_3: 트리 186개 사용

=== A안 fold 스크리닝 (3그룹 평균) ===


,Score 상당,1-NMAE 상당,FICR 상당,풍속MAE
풍속,,,,
reg,0.7040,0.8918,0.5162,1.2713
wide_reg,0.7036,0.8915,0.5157,1.2722
slow,0.7020,0.8918,0.5122,1.2730
base,0.7016,0.8911,0.5121,1.2849
wide,0.6998,0.8911,0.5085,1.2850



=== 그룹별 Score 상당 ===


group,kpx_group_1,kpx_group_2,kpx_group_3
풍속,,,
base,0.7019,0.7169,0.6859
reg,0.7026,0.7201,0.6893
slow,0.7011,0.7197,0.6850
wide,0.6985,0.7172,0.6836
wide_reg,0.7010,0.7197,0.6901


**확인할 것**: `base`(현재 설정)를 이기는 후보가 있는가? **세 그룹 모두에서** 이겨야 신뢰할 만합니다. 상위 2개를 골라 18-3에서 전체 fold + 실제 발전량 점수로 확인합니다.

`트리 N개 사용`도 눈여겨보세요. 설정한 `n_estimators`를 다 쓰고 멈췄다면(early stopping이 안 걸렸다면) **더 늘릴 여지**가 있다는 뜻입니다.

## 18-3. 상위 후보를 전체 fold + 실제 발전량 점수로 확인

18-2는 A안 fold 하나만 본 스크리닝이라, **B안 3-fold까지 보고 실제 발전량 모델로 확인**해야 채택할 수 있습니다. 리더보드가 B안 평균과 0.003 이내로 맞았으므로 **B안 평균이 주 지표**입니다.

⚠️ 후보 2개 × (풍속 24번 + 발전량 24번) = 48번. 아래 `TOP_CFGS`를 18-2 결과에 맞춰 바꾸세요.

In [92]:
def wind_fit_fn3(cfg_name, tag, top_n=None, tau=0.60, objective="l2", seed=SEED):
    """설정 이름으로 만든 풍속을 써서 발전량 모델까지 학습하는 fit_fn."""
    def fit_fn(g, cv_suffix, train_mask, valid_idx):
        ws = get_wind_cfg(g, cv_suffix, train_mask, cfg_name, objective)
        X = build_frame_with_wind(train, g, cv_suffix, train_mask, ws, tag)
        imp_key = (g, cv_suffix, tag, tau)
        if imp_key not in IMP_CACHE:
            m0 = _lgbm_train(X, g, train_mask, "quantile", tau, "actual", SEED)
            IMP_CACHE[imp_key] = pd.Series(m0.feature_importances_, index=m0.feature_name_, dtype=float)
        imp = IMP_CACHE[imp_key].reindex(X.columns).fillna(0.0).sort_values(ascending=False)
        keep = imp[imp > 0].index.tolist() if top_n is None else imp.head(top_n).index.tolist()
        m = _lgbm_train(X[keep], g, train_mask, "quantile", tau, "actual", seed)
        p = pd.Series(m.predict(X.loc[valid_idx, keep]), index=valid_idx)
        return p.clip(lower=0, upper=CAPACITY_KWH[g])
    return fit_fn


print("준비 완료")

준비 완료


In [93]:
# 18-2 스크리닝 상위 2개 (자동 선택 — 표를 보고 직접 바꿔도 됩니다)
TOP_CFGS = cfg_screen.groupby("풍속")["Score 상당"].mean().sort_values(ascending=False).head(2).index.tolist()
TOP_CFGS = [c for c in TOP_CFGS if c != "base"] or [TOP_CFGS[0]]
print("전체 fold로 확인할 후보:", TOP_CFGS)

# (1) 전체 fold에서 발전량 환산 지표
full_screen = pd.concat([screen_wind(lambda g, c, t, n=name: get_wind_cfg(g, c, t, n), name)
                         for name in TOP_CFGS] + [base_screen[base_screen["풍속"] == "gbdt_l2"]],
                        ignore_index=True)
print("\n=== 전체 fold 발전량 환산 지표 ===")
display(full_screen.groupby("풍속")[["Score 상당", "1-NMAE 상당", "FICR 상당"]].mean().round(4)
        .sort_values("Score 상당", ascending=False))

# (2) 실제 발전량 모델까지 (최종 판단)
stage16 = []
for name in TOP_CFGS:
    label = f"C_{name}"
    print(f"\n=== {label} ===")
    stage16.append(run_variant(label, wind_fit_fn3(name, f"cfg{name}", top_n=BEST_TOPN)))

display(summarize(stage13 + stage14 + stage16))

전체 fold로 확인할 후보: ['reg', 'wide_reg']
    reg/kpx_group_1: 트리 68개 사용
    reg/kpx_group_2: 트리 77개 사용
    reg/kpx_group_3: 트리 70개 사용
    reg/kpx_group_1: 트리 76개 사용
    reg/kpx_group_2: 트리 140개 사용
    reg/kpx_group_3: 트리 71개 사용
    wide_reg/kpx_group_1: 트리 94개 사용
    wide_reg/kpx_group_2: 트리 278개 사용
    wide_reg/kpx_group_3: 트리 91개 사용
    wide_reg/kpx_group_1: 트리 117개 사용
    wide_reg/kpx_group_2: 트리 123개 사용
    wide_reg/kpx_group_3: 트리 135개 사용

=== 전체 fold 발전량 환산 지표 ===


,Score 상당,1-NMAE 상당,FICR 상당
풍속,,,
reg,0.7002,0.8897,0.5107
wide_reg,0.6988,0.8895,0.5081
gbdt_l2,0.6985,0.8887,0.5084



=== C_reg ===
  [C_reg] A안(2024): score=0.6352  (1-NMAE=0.8753, FICR=0.3951)
  [C_reg] B안 fold1: score=0.6018  (1-NMAE=0.8567, FICR=0.3469)
  [C_reg] B안 fold2: score=0.6336  (1-NMAE=0.8726, FICR=0.3947)
  [C_reg] B안 fold3: score=0.6433  (1-NMAE=0.8777, FICR=0.4089)

=== C_wide_reg ===
  [C_wide_reg] A안(2024): score=0.6369  (1-NMAE=0.8764, FICR=0.3975)
  [C_wide_reg] B안 fold1: score=0.5965  (1-NMAE=0.8546, FICR=0.3385)
  [C_wide_reg] B안 fold2: score=0.6311  (1-NMAE=0.8731, FICR=0.3890)
  [C_wide_reg] B안 fold3: score=0.6411  (1-NMAE=0.8784, FICR=0.4037)


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
W_gbdt_l2,0.6391,0.6061,0.6316,0.6517,0.6298,0.0229
W_avg12,0.6386,0.6059,0.6339,0.6462,0.6286,0.0206
W_seedens,0.6355,0.6050,0.6302,0.6481,0.6278,0.0216
W_avgall,0.6361,0.6046,0.6294,0.6488,0.6276,0.0221
C_reg,0.6352,0.6018,0.6336,0.6433,0.6262,0.0217
W_gbdt_l1,0.6352,0.6052,0.6311,0.6396,0.6253,0.0179
W_blend,0.6335,0.5991,0.6277,0.6441,0.6236,0.0228
C_wide_reg,0.6369,0.5965,0.6311,0.6411,0.6229,0.0234
W_ridge,0.6267,0.5906,0.6143,0.6353,0.6134,0.0224


**확인할 것**

1. **발전량 환산 지표의 순위와 실제 점수의 순위가 같은가** — 같으면 앞으로 이 싼 지표로 빠르게 스크리닝할 수 있습니다
2. **`gbdt_l2`(B평균 0.6298)를 A안·B안 3-fold 전부에서 이기는가**

## 18-4. MLP 풍속 모델 — 오차 방향이 다른 자원

GBDT 하나를 아무리 튜닝해도 한계가 있습니다. **완전히 다른 종류의 모델**이 필요한데, 그 이유는 앙상블입니다.

13절에서 확인했듯 GBDT끼리는 잔차 상관이 0.93~0.95로 섞어도 이득이 거의 없었지만, MLP는 0.84~0.89로 낮아 섞을 가치가 있었습니다. 같은 논리를 **풍속 단계**에 적용합니다.

**단, 17절 교훈을 반드시 지킵니다.** 풍속을 평균 내면 극값이 눌려 손해입니다(Jensen). 그래서 **각 풍속으로 발전량까지 따로 예측한 뒤, 마지막에 발전량 예측을 평균**합니다(18-5절).

### 13절 MLP와 달라진 점

13절 MLP는 **full-batch로 36~104 에폭 만에 멈추는 사실상 미학습 상태**였습니다. 이번엔 제대로 학습시킵니다.

| | 13절 | 18절 |
|---|---|---|
| 배치 | 전체 (full-batch) | **미니배치 512** |
| 학습률 | 고정 1e-3 | **1e-3 + 코사인 감쇠** |
| 정규화 | weight decay만 | **BatchNorm + Dropout 0.15** |
| 층 | 128-64 | **512-256-128** |
| 에폭 | 최대 200 (실제 ~50) | 최대 60, 미니배치라 실제 갱신 횟수는 수백 배 |

타깃은 **풍속을 표준화한 값**입니다(발전량이 아니라). 입력도 학습 구간 통계로 표준화합니다 — **평균·표준편차를 학습 구간에서만 계산**하는 게 중요합니다(전체로 계산하면 누수).

⚠️ 12번 학습(4 fold × 3 그룹). CPU에서 한 번에 1~3분 정도 예상.

In [94]:
class WindMLP(nn.Module):
    def __init__(self, n_features, hidden=(512, 256, 128), p_drop=0.15):
        super().__init__()
        layers, prev = [], n_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(p_drop)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def get_wind_mlp(g, cv_suffix, train_mask, seed=None, max_epochs=60, batch_size=512, patience=8):
    seed = SEED if seed is None else seed
    key = (g, cv_suffix, f"mlp_s{seed}")
    if key in WIND_PRED_CACHE:
        return WIND_PRED_CACHE[key]

    X = build_wind_input_frame(train, g)
    y = train[f"scada_ws_{g}"]
    fit_idx = train_mask & y.notna()
    cutoff = train.loc[fit_idx, "kst_dtm"].quantile(0.9)
    tr, es = fit_idx & (train["kst_dtm"] <= cutoff), fit_idx & (train["kst_dtm"] > cutoff)

    # 표준화 통계는 반드시 '학습 구간'에서만 (전체로 구하면 누수)
    mu, sd = X.loc[tr].mean(), X.loc[tr].std().replace(0, 1)
    ymu, ysd = float(y[tr].mean()), float(y[tr].std())

    def tx(d):
        return torch.tensor(((d - mu) / sd).to_numpy(), dtype=torch.float32)

    Xtr, Xes, Xall = tx(X.loc[tr]), tx(X.loc[es]), tx(X)
    ytr = torch.tensor(((y[tr] - ymu) / ysd).to_numpy(), dtype=torch.float32)
    yes = torch.tensor(((y[es] - ymu) / ysd).to_numpy(), dtype=torch.float32)

    torch.manual_seed(seed)
    model = WindMLP(Xtr.shape[1])
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    n_batches = max(1, int(np.ceil(len(Xtr) / batch_size)))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs * n_batches)
    loss_fn = nn.MSELoss()

    best, best_state, left = float("inf"), None, patience
    for ep in range(max_epochs):
        model.train()
        perm = torch.randperm(len(Xtr))
        for b in range(n_batches):
            sel = perm[b * batch_size:(b + 1) * batch_size]
            if len(sel) < 2:
                continue
            opt.zero_grad()
            loss_fn(model(Xtr[sel]), ytr[sel]).backward()
            opt.step()
            sched.step()
        model.eval()
        with torch.no_grad():
            v = loss_fn(model(Xes), yes).item()
        if v < best - 1e-5:
            best, best_state, left = v, {k: t.clone() for k, t in model.state_dict().items()}, patience
        else:
            left -= 1
            if left <= 0:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        out = np.concatenate([model(Xall[i:i + 4096]).numpy() for i in range(0, len(Xall), 4096)])
    pred = pd.Series(out * ysd + ymu, index=train.index).clip(lower=0.0)
    WIND_PRED_CACHE[key] = pred
    print(f"    mlp/{g}: {ep + 1}에폭 (검증MSE {best:.4f})")
    return pred


print("=== MLP 풍속 모델 학습 ===")
for fold_name, info in FOLD_INFO.items():
    print(f"  {fold_name}")
    for g in GROUP_COLS:
        get_wind_mlp(g, info["cv_suffix"], info["train_mask"])

mlp_screen = screen_wind(lambda g, c, t: get_wind_mlp(g, c, t), "mlp")
print("\n=== 발전량 환산 지표 비교 ===")
cmp = pd.concat([base_screen[base_screen["풍속"].isin(["ridge", "gbdt_l2"])], mlp_screen], ignore_index=True)
display(cmp.groupby("풍속")[["Score 상당", "1-NMAE 상당", "FICR 상당", "풍속MAE"]].mean().round(4))

# 잔차 상관 — GBDT와 오차 방향이 다른가 (앙상블 가치의 근거)
rows = []
for fold_name, info in FOLD_INFO.items():
    for g in GROUP_COLS:
        vi, cs, tm = info["valid_idx"], info["cv_suffix"], info["train_mask"]
        t = train.loc[vi, f"scada_ws_{g}"]
        ok = t.notna()
        rows.append(pd.DataFrame({
            "gbdt": (get_wind_cfg(g, cs, tm, "base").loc[vi][ok] - t[ok]).to_numpy(),
            "mlp": (get_wind_mlp(g, cs, tm).loc[vi][ok] - t[ok]).to_numpy()}))
resid_w = pd.concat(rows, ignore_index=True)
print("\n=== 풍속 잔차 상관 (낮을수록 앙상블 이득) ===")
print(f"  gbdt vs mlp: {resid_w['gbdt'].corr(resid_w['mlp']):.3f}   (참고: 13절 발전량단계 GBDT끼리 0.93~0.95, GBDT-MLP 0.84~0.89)")

=== MLP 풍속 모델 학습 ===
  A안(2024)
    mlp/kpx_group_1: 13에폭 (검증MSE 0.2073)
    mlp/kpx_group_2: 13에폭 (검증MSE 0.1756)
    mlp/kpx_group_3: 17에폭 (검증MSE 0.2247)
  B안 fold1
    mlp/kpx_group_1: 17에폭 (검증MSE 0.1765)
    mlp/kpx_group_2: 22에폭 (검증MSE 0.1386)
    mlp/kpx_group_3: 17에폭 (검증MSE 0.2354)
  B안 fold2
  B안 fold3
    mlp/kpx_group_1: 12에폭 (검증MSE 0.1567)
    mlp/kpx_group_2: 18에폭 (검증MSE 0.1373)
    mlp/kpx_group_3: 15에폭 (검증MSE 0.1675)

=== 발전량 환산 지표 비교 ===


,Score 상당,1-NMAE 상당,FICR 상당,풍속MAE
풍속,,,,
gbdt_l2,0.6985,0.8887,0.5084,1.3112
mlp,0.6959,0.8853,0.5066,1.3420
ridge,0.6551,0.8686,0.4416,1.5202


    base/kpx_group_1: 트리 55개 사용
    base/kpx_group_2: 트리 105개 사용
    base/kpx_group_3: 트리 33개 사용
    base/kpx_group_1: 트리 37개 사용
    base/kpx_group_2: 트리 64개 사용
    base/kpx_group_3: 트리 61개 사용

=== 풍속 잔차 상관 (낮을수록 앙상블 이득) ===
  gbdt vs mlp: 0.851   (참고: 13절 발전량단계 GBDT끼리 0.93~0.95, GBDT-MLP 0.84~0.89)


**확인할 것**

1. **MLP 풍속이 GBDT를 이기는가** — 못 이겨도 괜찮습니다. 목적은 **다양성**입니다
2. **잔차 상관이 0.9 미만인가** — 낮을수록 18-5의 앙상블 이득이 큽니다. 0.95 이상이면 섞어도 소용없습니다
3. **에폭 수** — 60을 다 채우고 멈췄다면 더 학습시킬 여지가 있습니다. 5~10에폭 만에 멈췄다면 과적합이 빨리 온 것이니 Dropout을 올려야 합니다

## 18-5. 마지막 단 앙상블 — 풍속이 아니라 발전량 예측을 평균

**17절 교훈을 그대로 적용합니다.**

```
❌ 잘못된 방법:  풍속A, 풍속B → 평균 → 파워커브 → 발전량모델 → 예측
                  (극값이 눌려서 손해. Jensen 부등식)

✅ 올바른 방법:  풍속A → 파워커브 → 발전량모델 → 예측A ┐
                 풍속B → 파워커브 → 발전량모델 → 예측B ┴→ 평균
```

발전량 예측을 평균 내는 건 안전합니다. 그 뒤에 볼록한 변환이 더 없기 때문입니다.

MLP 풍속으로 발전량 모델을 12번 학습한 뒤, 캐시된 예측들을 가중평균해 최적 비율을 찾습니다(가중치 탐색은 **재학습 없이 몇 초**).

⚠️ 12번 학습.

In [95]:
def mlp_wind_fit_fn(top_n=None, tau=0.60, seed=SEED):
    def fit_fn(g, cv_suffix, train_mask, valid_idx):
        ws = get_wind_mlp(g, cv_suffix, train_mask)
        X = build_frame_with_wind(train, g, cv_suffix, train_mask, ws, "mlpws")
        imp_key = (g, cv_suffix, "mlpws", tau)
        if imp_key not in IMP_CACHE:
            m0 = _lgbm_train(X, g, train_mask, "quantile", tau, "actual", SEED)
            IMP_CACHE[imp_key] = pd.Series(m0.feature_importances_, index=m0.feature_name_, dtype=float)
        imp = IMP_CACHE[imp_key].reindex(X.columns).fillna(0.0).sort_values(ascending=False)
        keep = imp[imp > 0].index.tolist() if top_n is None else imp.head(top_n).index.tolist()
        m = _lgbm_train(X[keep], g, train_mask, "quantile", tau, "actual", seed)
        p = pd.Series(m.predict(X.loc[valid_idx, keep]), index=valid_idx)
        return p.clip(lower=0, upper=CAPACITY_KWH[g])
    return fit_fn


print("=== M_mlpws (MLP 풍속 → 발전량) ===")
stage17 = [run_variant("M_mlpws", mlp_wind_fit_fn(top_n=BEST_TOPN))]

# 가장 좋은 GBDT-풍속 기반 변형 자동 선택
_best_tbl = summarize(stage13 + stage14 + stage16)
BEST_GBDT_LABEL = _best_tbl.index[0]
print(f"\n최고 GBDT-풍속 변형: {BEST_GBDT_LABEL} (B안 평균 {_best_tbl.loc[BEST_GBDT_LABEL, 'B안 평균']:.4f})")

# 발전량 예측 단계에서 가중 평균 (재학습 없음)
rows = []
for w in np.arange(0.0, 1.01, 0.1):
    df = score_blend({BEST_GBDT_LABEL: w, "M_mlpws": 1 - w}, label="blend")
    piv = df.set_index("fold")["score"]
    rows.append({f"{BEST_GBDT_LABEL} 비중": round(w, 1), "A안": piv["A안(2024)"],
                 "B안 평균": piv[B_FOLDS].mean(), "B안 최솟값": piv[B_FOLDS].min()})
print(f"\n=== {BEST_GBDT_LABEL} + M_mlpws 블렌드 (발전량 예측 단계) ===")
display(pd.DataFrame(rows).set_index(f"{BEST_GBDT_LABEL} 비중").round(4))

display(summarize(stage13 + stage14 + stage16 + stage17))

=== M_mlpws (MLP 풍속 → 발전량) ===
  [M_mlpws] A안(2024): score=0.6287  (1-NMAE=0.8624, FICR=0.3950)
  [M_mlpws] B안 fold1: score=0.6028  (1-NMAE=0.8504, FICR=0.3553)
  [M_mlpws] B안 fold2: score=0.6217  (1-NMAE=0.8543, FICR=0.3890)
  [M_mlpws] B안 fold3: score=0.6388  (1-NMAE=0.8714, FICR=0.4061)

최고 GBDT-풍속 변형: W_gbdt_l2 (B안 평균 0.6298)

=== W_gbdt_l2 + M_mlpws 블렌드 (발전량 예측 단계) ===


,A안,B안 평균,B안 최솟값
W_gbdt_l2 비중,,,
0.0,0.6287,0.6211,0.6028
0.1,0.6321,0.6236,0.6041
0.2,0.6347,0.6263,0.6062
0.3,0.6373,0.6280,0.6082
0.4,0.6390,0.6297,0.6094
0.5,0.6402,0.6306,0.6099
0.6,0.6418,0.6316,0.6100
0.7,0.6412,0.6321,0.6107
0.8,0.6410,0.6312,0.6086


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
model,,,,,,
W_gbdt_l2,0.6391,0.6061,0.6316,0.6517,0.6298,0.0229
W_avg12,0.6386,0.6059,0.6339,0.6462,0.6286,0.0206
W_seedens,0.6355,0.6050,0.6302,0.6481,0.6278,0.0216
W_avgall,0.6361,0.6046,0.6294,0.6488,0.6276,0.0221
C_reg,0.6352,0.6018,0.6336,0.6433,0.6262,0.0217
W_gbdt_l1,0.6352,0.6052,0.6311,0.6396,0.6253,0.0179
W_blend,0.6335,0.5991,0.6277,0.6441,0.6236,0.0228
C_wide_reg,0.6369,0.5965,0.6311,0.6411,0.6229,0.0234
M_mlpws,0.6287,0.6028,0.6217,0.6388,0.6211,0.0180


**확인할 것**

1. **블렌드 곡선이 완만한 봉우리를 그리는가** — 봉우리 근처의 **둥근 값**(0.6, 0.7)을 고르세요. 최고점이 0.63 같은 값이면 그건 검증 과적합입니다
2. **`B안 최솟값`(가장 나쁜 fold)도 같이 오르는가** — 평균만 오르고 최악 fold가 나빠지면 실전에서 위험합니다
3. **`M_mlpws` 단독 점수** — GBDT 풍속보다 낮아도 괜찮습니다. 블렌드에서 이득이 나면 채택입니다

### 채택되면 다음
- 17-4의 제출 경로에 **① 튜닝된 풍속 설정 ② MLP 풍속 ③ 블렌드 비중**을 반영해 제출 파일 재생성
- 그다음은 `train.ipynb` / `inference.ipynb` 분리 (2차 평가 필수 요건, 마감 8/14)

---

# 19. 제출 파일 v2 — MLP 풍속 블렌드 반영

17-4에서 만든 제출 경로에 18절 결과를 반영합니다.

| | v1 (17-4) | **v2 (여기)** |
|---|---|---|
| 풍속 | GBDT(l2) 하나 | **GBDT(l2) + MLP 두 개** |
| 발전량 모델 | 풍속 1개 × seed 5개 | **풍속 2개 × seed 5개** |
| 합치는 방법 | – | **발전량 예측 단계에서 0.7 : 0.3** |
| 검증 점수(B평균) | 0.6298 | **0.6321** |

**17절 교훈 준수**: 풍속을 평균 내지 않습니다. 각 풍속으로 발전량까지 따로 예측한 뒤 **마지막에 발전량 예측을 섞습니다.**

⚠️ 학습: 풍속 6번(GBDT 3 + MLP 3) + 발전량 30번(2 풍속 × 3 그룹 × 5 seed).

In [96]:
def fit_final_mlp_wind(g, seed=None, max_epochs=60, batch_size=512, patience=8):
    """전체 train으로 MLP 풍속 모델 학습 → (train 추정풍속, test 추정풍속) 반환.
    18-4의 get_wind_mlp과 같은 구조이나 fold 컷오프 없이 전체 train을 쓴다."""
    seed = SEED if seed is None else seed
    Xtr_df, Xte_df = build_wind_input_frame(train, g), build_wind_input_frame(test, g)
    assert list(Xtr_df.columns) == list(Xte_df.columns), f"{g}: 풍속 입력 컬럼 train/test 불일치"

    y = train[f"scada_ws_{g}"]
    fit_idx = y.notna()
    cutoff = train.loc[fit_idx, "kst_dtm"].quantile(0.9)
    tr, es = fit_idx & (train["kst_dtm"] <= cutoff), fit_idx & (train["kst_dtm"] > cutoff)

    mu, sd = Xtr_df.loc[tr].mean(), Xtr_df.loc[tr].std().replace(0, 1)
    ymu, ysd = float(y[tr].mean()), float(y[tr].std())

    def tx(d):
        return torch.tensor(((d - mu) / sd).to_numpy(), dtype=torch.float32)

    Xtr, Xes = tx(Xtr_df.loc[tr]), tx(Xtr_df.loc[es])
    ytr = torch.tensor(((y[tr] - ymu) / ysd).to_numpy(), dtype=torch.float32)
    yes = torch.tensor(((y[es] - ymu) / ysd).to_numpy(), dtype=torch.float32)

    torch.manual_seed(seed)
    model = WindMLP(Xtr.shape[1])
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    nb_ = max(1, int(np.ceil(len(Xtr) / batch_size)))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs * nb_)
    loss_fn = nn.MSELoss()

    best, best_state, left = float("inf"), None, patience
    for ep in range(max_epochs):
        model.train()
        perm = torch.randperm(len(Xtr))
        for b in range(nb_):
            sel = perm[b * batch_size:(b + 1) * batch_size]
            if len(sel) < 2:
                continue
            opt.zero_grad()
            loss_fn(model(Xtr[sel]), ytr[sel]).backward()
            opt.step()
            sched.step()
        model.eval()
        with torch.no_grad():
            v = loss_fn(model(Xes), yes).item()
        if v < best - 1e-5:
            best, best_state, left = v, {k: t.clone() for k, t in model.state_dict().items()}, patience
        else:
            left -= 1
            if left <= 0:
                break
    model.load_state_dict(best_state)
    model.eval()

    def infer(df_):
        X_ = tx(df_)
        with torch.no_grad():
            o = np.concatenate([model(X_[i:i + 4096]).numpy() for i in range(0, len(X_), 4096)])
        return o * ysd + ymu

    print(f"    mlp/{g}: {ep + 1}에폭 (검증MSE {best:.4f})")
    return (pd.Series(infer(Xtr_df), index=train.index).clip(lower=0.0),
            pd.Series(infer(Xte_df), index=test.index).clip(lower=0.0))


# ── 최종 풍속 2종 (전체 train 학습) ────────────────────────────────
FINAL_WS = {}
print("=== GBDT 풍속 (결정적) ===")
for g in GROUP_COLS:
    m = fit_final_wind(g, "l2", SEED)           # rand=False 기본값
    Xtr_w, Xte_w = build_wind_input_frame(train, g), build_wind_input_frame(test, g)
    FINAL_WS[("gbdt", g)] = (pd.Series(m.predict(Xtr_w), index=train.index).clip(lower=0.0),
                             pd.Series(m.predict(Xte_w), index=test.index).clip(lower=0.0))
    print(f"  {g}: train 내 상관 {FINAL_WS[('gbdt', g)][0].corr(train[f'scada_ws_{g}']):.4f}")

print("\n=== MLP 풍속 ===")
for g in GROUP_COLS:
    FINAL_WS[("mlp", g)] = fit_final_mlp_wind(g)
    print(f"  {g}: train 내 상관 {FINAL_WS[('mlp', g)][0].corr(train[f'scada_ws_{g}']):.4f}")

print("\n=== 두 풍속의 test 평균 (train과 비교) ===")
for g in GROUP_COLS:
    print(f"  {g}: gbdt {FINAL_WS[('gbdt', g)][1].mean():.2f} / mlp {FINAL_WS[('mlp', g)][1].mean():.2f} m/s"
          f"   (train 실측 {train[f'scada_ws_{g}'].mean():.2f})")

=== GBDT 풍속 (결정적) ===
  kpx_group_1: train 내 상관 0.9301
  kpx_group_2: train 내 상관 0.9458
  kpx_group_3: train 내 상관 0.9531

=== MLP 풍속 ===
    mlp/kpx_group_1: 10에폭 (검증MSE 0.1494)
  kpx_group_1: train 내 상관 0.9105
    mlp/kpx_group_2: 10에폭 (검증MSE 0.1462)
  kpx_group_2: train 내 상관 0.9207
    mlp/kpx_group_3: 14에폭 (검증MSE 0.1737)
  kpx_group_3: train 내 상관 0.9324

=== 두 풍속의 test 평균 (train과 비교) ===
  kpx_group_1: gbdt 7.17 / mlp 7.19 m/s   (train 실측 6.78)
  kpx_group_2: gbdt 7.67 / mlp 7.76 m/s   (train 실측 7.20)
  kpx_group_3: gbdt 6.30 / mlp 6.21 m/s   (train 실측 5.85)


In [97]:
# ── 풍속별 발전량 모델 → test 예측 → 블렌드 ────────────────────────
FINAL_TAU = 0.60
FINAL_SEEDS = [SEED, 7, 123, 2024, 31]
BLEND_W = {"gbdt": 0.7, "mlp": 0.3}     # 18-5 결과 (곡선이 완만해 둥근 값 선택)

test_pred_by_wind = {}
for src in ["gbdt", "mlp"]:
    for g in GROUP_COLS:
        ws_tr, ws_te = FINAL_WS[(src, g)]

        # 파워커브: 전체 train의 (추정풍속, 실제 발전량)으로 적합 → test엔 '적용'만
        ok = train[g].notna()
        edges, vals = fit_power_curve_oracle(ws_tr[ok].to_numpy(dtype=float),
                                             train.loc[ok, g].to_numpy(dtype=float))

        Xtr = build_frame_given_pc(train, g, ws_tr, edges, vals, f"fin_{src}")
        Xte = build_frame_given_pc(test, g, ws_te, edges, vals, f"fin_{src}")
        assert list(Xtr.columns) == list(Xte.columns), f"{src}/{g}: 발전량 입력 컬럼 불일치"

        m0 = _lgbm_train(Xtr, g, FULL_MASK, "quantile", FINAL_TAU, "actual", SEED)
        imp = pd.Series(m0.feature_importances_, index=m0.feature_name_, dtype=float).sort_values(ascending=False)
        keep = imp[imp > 0].index.tolist() if BEST_TOPN is None else imp.head(BEST_TOPN).index.tolist()

        preds = [_lgbm_train_rand(Xtr[keep], g, FULL_MASK, "quantile", FINAL_TAU, "actual", sd).predict(Xte[keep])
                 for sd in FINAL_SEEDS]
        test_pred_by_wind[(src, g)] = np.clip(np.mean(preds, axis=0), 0, CAPACITY_KWH[g])
        print(f"✓ {src}/{g}: 피처 {len(keep)}개, 예측 이용률 "
              f"{test_pred_by_wind[(src, g)].mean() / CAPACITY_KWH[g] * 100:.1f}%")

sub_pred_v2 = {g: sum(BLEND_W[s] * test_pred_by_wind[(s, g)] for s in BLEND_W) for g in GROUP_COLS}
print("\n=== 블렌드 후 (gbdt 0.7 : mlp 0.3) ===")
for g in GROUP_COLS:
    print(f"  {g}: 이용률 {sub_pred_v2[g].mean() / CAPACITY_KWH[g] * 100:.1f}%")

✓ gbdt/kpx_group_1: 피처 200개, 예측 이용률 40.7%
✓ gbdt/kpx_group_2: 피처 200개, 예측 이용률 43.2%
✓ gbdt/kpx_group_3: 피처 200개, 예측 이용률 36.8%
✓ mlp/kpx_group_1: 피처 200개, 예측 이용률 41.0%
✓ mlp/kpx_group_2: 피처 200개, 예측 이용률 44.2%
✓ mlp/kpx_group_3: 피처 200개, 예측 이용률 37.1%

=== 블렌드 후 (gbdt 0.7 : mlp 0.3) ===
  kpx_group_1: 이용률 40.8%
  kpx_group_2: 이용률 43.5%
  kpx_group_3: 이용률 36.9%


In [98]:
# ── 제출 파일 생성 + 검증 + 상식 점검 ─────────────────────────────
pred_df_v2 = pd.DataFrame(sub_pred_v2, index=test.index)
pred_df_v2["forecast_kst_dtm"] = test["kst_dtm"].dt.strftime("%Y-%m-%d %H:%M:%S")

submission_v2 = build_submission(pred_df_v2, sample_path=SAMPLE_PATH)
validate_submission(submission_v2, sample_path=SAMPLE_PATH)
print("✓ validate_submission() 통과 —", submission_v2.shape)

print("\n=== v1 대비 변화 ===")
rows = []
for g in GROUP_COLS:
    cap = CAPACITY_KWH[g]
    rows.append({"group": g,
                 "v1 이용률": round(submission[g].mean() / cap, 4),
                 "v2 이용률": round(submission_v2[g].mean() / cap, 4),
                 "v1 표준편차": round((submission[g] / cap).std(), 4),
                 "v2 표준편차": round((submission_v2[g] / cap).std(), 4),
                 "평균 절대변화(kWh)": round((submission_v2[g] - submission[g]).abs().mean(), 1),
                 "상관(v1,v2)": round(submission[g].corr(submission_v2[g]), 4)})
display(pd.DataFrame(rows))

print("\n=== 월별 예측 이용률 (계절성 점검) ===")
mm = submission_v2.copy()
mm["month"] = pd.to_datetime(mm["forecast_kst_dtm"]).dt.month
display((mm.groupby("month")[GROUP_COLS].mean() / pd.Series(CAPACITY_KWH)).round(3))

✓ validate_submission() 통과 — (8760, 5)

=== v1 대비 변화 ===


,group,v1 이용률,v2 이용률,v1 표준편차,v2 표준편차,평균 절대변화(kWh),"상관(v1,v2)"
0,kpx_group_1,0.4071,0.4080,0.2877,0.2865,231.6,0.9987
1,kpx_group_2,0.4325,0.4353,0.3064,0.3058,250.5,0.9986
2,kpx_group_3,0.3682,0.3689,0.2952,0.2964,256.8,0.9983



=== 월별 예측 이용률 (계절성 점검) ===


,kpx_group_1,kpx_group_2,kpx_group_3
month,,,
1,0.545,0.575,0.492
2,0.613,0.657,0.587
3,0.420,0.463,0.391
4,0.432,0.469,0.409
5,0.374,0.394,0.343
6,0.365,0.381,0.321
7,0.311,0.340,0.286
8,0.310,0.330,0.281
9,0.228,0.255,0.189


**확인할 것**

1. **`validate_submission() 통과`**
2. **v1과의 상관이 0.98 이상인가** — 두 버전이 크게 다르면 뭔가 잘못된 것입니다. 블렌드는 예측을 조금만 바꿔야 정상입니다
3. **v2 표준편차가 v1보다 약간 작은가** — 평균을 내면 산포가 줄어드는 게 정상입니다. **많이 줄면 문제**입니다(FICR에 불리)
4. **월별 계절성이 v1과 같은 모양인가**

In [99]:
# 확인 후 저장 (파일명 규칙: YYYYMMDD_vN_설명.csv)
path = save_submission(submission_v2, "20260801_v2_wsgbdt_mlpblend_q60.csv", sample_path=SAMPLE_PATH)
print("저장:", path)

저장: d:\공모전\wind_forecast_new\submissions\20260801_v2_wsgbdt_mlpblend_q60.csv
